# PoliMillionaire - All Competitions Notebook
## Competitions covered
| ID | Name |
|----|------|
| 0 | Entertainment |
| 1 | Ancient History & Politics |
| 2 | Science & Nature |
| 3 | Maths |
| 4 | Philosophy & Psychology |
| 5 | News & Current Events |

This notebook merges independently developed pipelines that share the same
`MillionaireClient` API and, where possible, the same loaded base model
(`Qwen/Qwen2.5-7B-Instruct`):

- **Entertainment / Science / Psychology** - zero-shot, few-shot, CoT, Wikipedia RAG,
  DuckDuckGo hybrid RAG, multi-model ensemble (from `PoliMillionaire_Starter_Clean`)
- **History** - advanced BM25 + sentence-embedding RAG with PyTerrier, cross-encoder
  reranker, direct logit scoring, and agentic tool router (from `History_v2`)
- **News** - live Serper news search, Bing RSS fallback + FAISS semantic ranking
  (from `nlpproject_news`)
- **Maths** - shared 7B planner, Qwen Math 1.5B solver, and SymPy tools

## 0. Setup - Mount Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')


Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [2]:
import os, sys, importlib.util, subprocess

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Adjust these paths to match your Drive layout
PACKAGE_PARENT_DIR = '/content/gdrive/MyDrive/Colab Notebooks/NLP_Test'
NLP_ASSIGNMENT_DIR = '/content/gdrive/MyDrive/NLP_assignment'

for _dir in (PACKAGE_PARENT_DIR, NLP_ASSIGNMENT_DIR):
    if _dir not in sys.path:
        sys.path.append(_dir)

print('Paths added:', PACKAGE_PARENT_DIR, NLP_ASSIGNMENT_DIR)


Paths added: /content/gdrive/MyDrive/Colab Notebooks/NLP_Test /content/gdrive/MyDrive/NLP_assignment


In [3]:
# Install all dependencies needed across all notebook pipelines
!pip install -q transformers accelerate bitsandbytes sentencepiece sympy wikipedia-api
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118
!pip install -q python-terrier sentence-transformers scikit-learn
!pip install -q protobuf latex2sympy2 faiss-cpu trafilatura requests beautifulsoup4 openai-whisper

In [4]:
from millionaire_client import MillionaireClient, AuthenticationError
import time, json, re, random, gc
from datetime import datetime, timezone, timedelta
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import quote, urlencode
from urllib.request import Request, urlopen
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
print('Imports complete')


Imports complete


## 1. Login & Explore the Game

In [5]:
# Credentials(store in Colab Secrets as 'poli-millionaire')
#from google.colab import userdata

API_URL = "http://131.175.15.22:51111/"
USERNAME = "gary"
PASSWORD = "13790229"

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f"Welcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed, it has: {e}")

Welcome, gary! (Role: student)


In [6]:
# List all competitions
print("=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  [{comp.id}] {comp.name} | {comp.max_levels} questions | {comp.description}")

=== Available Competitions ===
  [0] Entertainment | 15 questions | Music, Movies, Celebrities and more
  [1] Ancient History and Politics | 15 questions | The Roman Empire, The Greeks, and more
  [2] Science and Nature | 15 questions | Chemistry, Biology, Physics and similar subjects
  [3] Maths | 15 questions | Mathematics and Statistics from High School and College
  [4] Philosophy and Psychology | 15 questions | Great thinkers and the human psyche
  [5] News | 15 questions | Staying current with global breaking news


## 2. Shared Notebook Models

The shared base model (`Qwen/Qwen2.5-7B-Instruct`) is loaded once as
`model` / `tokenizer` and reused by the non-math pipelines and by the Maths
planner. The Maths solver model (`Qwen/Qwen2.5-Math-1.5B-Instruct`) is also
loaded once here as `math_model` / `math_tokenizer`.


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MATH_MODEL_ID = "Qwen/Qwen2.5-Math-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
).eval()

answer_tokenizer = tokenizer
answer_model = model
planner_tokenizer = tokenizer
planner_model = model
_ANSWER_MODEL_CACHE = {MODEL_ID: (tokenizer, model)}


def load_answer_model(model_name: str = MODEL_ID):
    """Return the already-loaded shared answer/planner model used by every non-math pipeline."""
    requested_model = model_name or MODEL_ID
    shared_model_id = globals().get("MODEL_ID", MODEL_ID)

    if requested_model != shared_model_id:
        raise ValueError(
            f"Requested {requested_model}, but the shared loaded model is {shared_model_id}. "
            "Change MODEL_ID in the shared loader cell and rerun the notebook instead of loading a second planner model."
        )

    _ANSWER_MODEL_CACHE[shared_model_id] = (tokenizer, model)
    return tokenizer, model


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Planner/base model loaded once: {MODEL_ID}")
if torch.cuda.is_available():
    print(f"GPU memory allocated after planner/base model: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Loading math model once: {MATH_MODEL_ID}")
math_tokenizer = AutoTokenizer.from_pretrained(MATH_MODEL_ID, trust_remote_code=True)
if math_tokenizer.pad_token is None:
    math_tokenizer.pad_token = math_tokenizer.eos_token

math_model_kwargs = {"trust_remote_code": True, "low_cpu_mem_usage": True}
if torch.cuda.is_available():
    math_model_kwargs.update({"device_map": "auto", "torch_dtype": torch.float16})
else:
    math_model_kwargs.update({"torch_dtype": torch.float32})

math_model = AutoModelForCausalLM.from_pretrained(
    MATH_MODEL_ID,
    **math_model_kwargs,
).eval()
if not torch.cuda.is_available():
    math_model.to("cpu")

_MATH_MODEL_CACHE = {MATH_MODEL_ID: (math_tokenizer, math_model)}


def load_math_model(model_name: str = MATH_MODEL_ID):
    """Return the already-loaded Qwen Math model used only by the Maths pipeline."""
    requested_model = model_name or MATH_MODEL_ID
    if requested_model != MATH_MODEL_ID:
        raise ValueError(
            f"Requested {requested_model}, but the loaded math model is {MATH_MODEL_ID}. "
            "Change MATH_MODEL_ID in this shared loader cell and rerun from setup."
        )
    _MATH_MODEL_CACHE[MATH_MODEL_ID] = (math_tokenizer, math_model)
    return math_tokenizer, math_model

print(f"Math model loaded once: {MATH_MODEL_ID}")
if torch.cuda.is_available():
    print(f"GPU memory allocated after math model: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Math model loaded once: Qwen/Qwen2.5-Math-1.5B-Instruct
GPU memory allocated after math model: 8.63 GB


---
## 3. Speech to Text

Whisper `turbo` is loaded after Qwen 7B and Qwen Math, then reused by all speech-mode game loops.


In [8]:
# ---- Shared Speech Mode / Whisper Loader ----
import whisper
import traceback
from pathlib import Path

# Load order in this cell is intentional: Qwen 7B first, Qwen Math second, Whisper turbo last.
# Pass mode="text" or mode="speech" directly in each game run cell.
WHISPER_MODEL_SIZE = globals().get("WHISPER_MODEL_SIZE", "turbo")
# The all-competitions notebook already keeps the 7B planner and math model in memory,
# so the fallback list avoids full large-v3 and steps down to smaller models if needed.
WHISPER_FALLBACK_MODEL_SIZES = globals().get("WHISPER_FALLBACK_MODEL_SIZES", ["medium", "small", "base"])
WHISPER_DEVICE = globals().get("WHISPER_DEVICE", "cuda")
WHISPER_LANGUAGE = globals().get("WHISPER_LANGUAGE", "en")
WHISPER_FP16 = globals().get("WHISPER_FP16", True)
WHISPER_RETRY_EMPTY_OPTIONS = globals().get("WHISPER_RETRY_EMPTY_OPTIONS", True)
WHISPER_MIN_TRANSCRIPT_CHARS = globals().get("WHISPER_MIN_TRANSCRIPT_CHARS", 2)
SAVE_SPEECH_AUDIO = globals().get("SAVE_SPEECH_AUDIO", True)
DISPLAY_SPEECH_AUDIO = globals().get("DISPLAY_SPEECH_AUDIO", False)
SPEECH_AUDIO_DIR = globals().get("SPEECH_AUDIO_DIR", "/content/gdrive/MyDrive/NLP_assignment/speech_game_audio")
LOAD_WHISPER_IN_SHARED_MODEL_CELL = globals().get("LOAD_WHISPER_IN_SHARED_MODEL_CELL", True)

_WHISPER_MODEL_CACHE = globals().setdefault("_WHISPER_MODEL_CACHE", {})
ACTIVE_WHISPER_MODEL_SIZE = globals().get("ACTIVE_WHISPER_MODEL_SIZE", None)
ACTIVE_WHISPER_DEVICE = globals().get("ACTIVE_WHISPER_DEVICE", None)


def print_cuda_memory(label):
    try:
        if not torch.cuda.is_available():
            print(f"{label}: CUDA not available")
            return
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        gb = 1024 ** 3
        print(
            f"{label}: "
            f"free={free_bytes / gb:.2f}GB | "
            f"total={total_bytes / gb:.2f}GB | "
            f"allocated={torch.cuda.memory_allocated() / gb:.2f}GB | "
            f"reserved={torch.cuda.memory_reserved() / gb:.2f}GB"
        )
    except Exception as exc:
        print(f"{label}: CUDA memory check unavailable ({exc})")


def load_whisper_model_for_speech():
    """Load Whisper once for speech mode and return the cached model afterwards."""

    requested_device = WHISPER_DEVICE
    device_name = requested_device if requested_device == "cpu" or torch.cuda.is_available() else "cpu"
    candidate_sizes = [WHISPER_MODEL_SIZE]
    if device_name == "cuda":
        for fallback_size in WHISPER_FALLBACK_MODEL_SIZES:
            if fallback_size not in candidate_sizes:
                candidate_sizes.append(fallback_size)

    last_oom = None
    for model_size in candidate_sizes:
        cache_key = (model_size, device_name)
        if cache_key in _WHISPER_MODEL_CACHE:
            globals()["ACTIVE_WHISPER_MODEL_SIZE"] = model_size
            globals()["ACTIVE_WHISPER_DEVICE"] = device_name
            return _WHISPER_MODEL_CACHE[cache_key], device_name

        try:
            if device_name == "cuda":
                torch.cuda.empty_cache()
            print_cuda_memory(f"Before Whisper {model_size} load")
            print(f"Loading Whisper {model_size!r} on {device_name}...")
            started_at = time.time()
            _WHISPER_MODEL_CACHE[cache_key] = whisper.load_model(model_size, device=device_name)
            globals()["ACTIVE_WHISPER_MODEL_SIZE"] = model_size
            globals()["ACTIVE_WHISPER_DEVICE"] = device_name
            print(f"Whisper ready in {time.time() - started_at:.1f}s")
            print_cuda_memory(f"After Whisper {model_size} load")
            return _WHISPER_MODEL_CACHE[cache_key], device_name
        except RuntimeError as exc:
            message = str(exc).lower()
            if device_name == "cuda" and ("out of memory" in message or "cuda" in message):
                last_oom = RuntimeError(str(exc))
                traceback.clear_frames(exc.__traceback__)
                print(f"Whisper {model_size!r} did not fit on CUDA; trying fallback.")
                _WHISPER_MODEL_CACHE.pop(cache_key, None)
                del exc
                gc.collect()
                torch.cuda.empty_cache()
                try:
                    torch.cuda.ipc_collect()
                except Exception:
                    pass
                print_cuda_memory(f"After Whisper {model_size} OOM cleanup")
                continue
            raise

    if device_name == "cuda":
        print("Whisper did not fit on CUDA; retrying requested model on CPU.")
        device_name = "cpu"
        cache_key = (WHISPER_MODEL_SIZE, device_name)
        if cache_key not in _WHISPER_MODEL_CACHE:
            _WHISPER_MODEL_CACHE[cache_key] = whisper.load_model(WHISPER_MODEL_SIZE, device=device_name)
        globals()["ACTIVE_WHISPER_MODEL_SIZE"] = WHISPER_MODEL_SIZE
        globals()["ACTIVE_WHISPER_DEVICE"] = device_name
        return _WHISPER_MODEL_CACHE[cache_key], device_name

    raise RuntimeError("Could not load Whisper model") from last_oom


def clean_whisper_text(text):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    return text.strip(' \"')


def strip_speech_noise(text):
    text = clean_whisper_text(text)
    if not text:
        return ""

    hallucination_phrases = [
        r"\bthanks? for watching[.!?]*",
        r"\bthank you for watching[.!?]*",
        r"\bbut you all too much for me to download[.!?]*",
        r"\byou all too much for me to download[.!?]*",
    ]
    for pattern in hallucination_phrases:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    laughter_or_filler = (
        r"(?:\b(?:a?ha(?:ha)+|ha|he(?:he)+h?|ehe(?:he)+h?|ah+|eh+|uh+|um+|ahem|pfft+)\b"
        r"[\s,.;:!?-]*)+"
    )
    previous = None
    while previous != text:
        previous = text
        text = re.sub(laughter_or_filler, " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"(?:^|\s)[,.;:!?-]+(?=\s|$)", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" ,.;:!?-")
    return text


def clean_question_transcript(text):
    text = strip_speech_noise(text)
    text = re.sub(r"^(?:oh|uh|um|ahem)[,!.?\s]+", "", text, flags=re.IGNORECASE).strip()
    return clean_whisper_text(text)


def clean_option_transcript(text, letter):
    text = strip_speech_noise(text)
    patterns = [
        rf"^Option\s*{letter}\s*[\.:,\)]?\s*",
        r"^Option\s*[A-D]\s*[\.:,\)]?\s*",
        rf"^{letter}\s*[\.:\)]\s*",
    ]
    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()
    return strip_speech_noise(text)


def transcript_has_content(text):
    return len(re.sub(r"[^A-Za-z0-9]", "", text or "")) >= WHISPER_MIN_TRANSCRIPT_CHARS


def maybe_display_audio(audio_bytes):
    if not DISPLAY_SPEECH_AUDIO:
        return
    try:
        from IPython.display import Audio, display
        display(Audio(audio_bytes))
    except Exception as exc:
        print(f"Could not display audio inline: {exc}")


def save_speech_audio(audio_bytes, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as handle:
        handle.write(audio_bytes)
    return path


def transcribe_audio_file(audio_path, initial_prompt=None, is_option=False, letter=None):
    whisper_model, whisper_device = load_whisper_model_for_speech()
    fp16 = bool(WHISPER_FP16 and whisper_device == "cuda")
    attempts = [
        {
            "initial_prompt": initial_prompt,
            "temperature": 0.0,
            "no_speech_threshold": 0.95,
            "logprob_threshold": -1.5,
            "compression_ratio_threshold": 2.8,
        },
        {
            "initial_prompt": None,
            "temperature": 0.0,
            "no_speech_threshold": 1.0,
            "logprob_threshold": -2.0,
            "compression_ratio_threshold": 3.5,
            "suppress_blank": False,
        },
        {
            "initial_prompt": None,
            "temperature": 0.2,
            "no_speech_threshold": 1.0,
            "logprob_threshold": -2.0,
            "compression_ratio_threshold": 3.5,
            "suppress_blank": False,
        },
    ]
    if not (is_option and WHISPER_RETRY_EMPTY_OPTIONS):
        attempts = attempts[:1]

    best_text = ""
    best_raw_text = ""
    for attempt_index, attempt in enumerate(attempts, start=1):
        kwargs = {
            "language": WHISPER_LANGUAGE,
            "task": "transcribe",
            "fp16": fp16,
            "condition_on_previous_text": False,
            **attempt,
        }
        if float(kwargs.get("temperature", 0.0)) == 0.0:
            kwargs["beam_size"] = 5
        else:
            kwargs["best_of"] = 5
        prompt = kwargs.pop("initial_prompt", None)
        if prompt:
            kwargs["initial_prompt"] = prompt

        result = whisper_model.transcribe(str(audio_path), **kwargs)
        raw_text = clean_whisper_text(result.get("text", ""))
        cleaned_text = clean_option_transcript(raw_text, letter or "") if is_option else clean_question_transcript(raw_text)

        if raw_text and (not best_raw_text or len(raw_text) > len(best_raw_text)):
            best_raw_text = raw_text
        if cleaned_text and (not best_text or len(cleaned_text) > len(best_text)):
            best_text = cleaned_text

        if transcript_has_content(cleaned_text):
            if raw_text != cleaned_text:
                print(f"Cleaned transcript noise: {raw_text!r} -> {cleaned_text!r}")
            return cleaned_text

        if is_option and attempt_index < len(attempts):
            print(f"Empty/low-content option transcript from {Path(audio_path).name}; retrying Whisper pass {attempt_index + 1}...")

    if best_raw_text and best_raw_text != best_text:
        print(f"Cleaned transcript noise: {best_raw_text!r} -> {best_text!r}")
    return best_text


def transcribe_speech_question(game):
    question = game.current_question
    if question is None:
        return None, {"error": "No active question returned by server."}

    audio_dir = Path(SPEECH_AUDIO_DIR)
    level = game.current_level
    session_id = game.session_id
    transcript = {
        "mode": "speech",
        "session_id": session_id,
        "level": level,
        "audio_files": {},
        "question": None,
        "options": [],
        "whisper_model_size": globals().get("ACTIVE_WHISPER_MODEL_SIZE") or WHISPER_MODEL_SIZE,
        "whisper_device": globals().get("ACTIVE_WHISPER_DEVICE") or WHISPER_DEVICE,
    }
    started_at = time.time()

    def option_letter(index):
        letters = globals().get("LETTERS", "ABCD")
        return letters[index] if index < len(letters) else chr(65 + index)

    def audio_path_for(kind, letter=None):
        if kind == "question":
            filename = f"session_{session_id}_level_{level}_question.wav"
        else:
            filename = f"session_{session_id}_level_{level}_option_{letter}.wav"
        if SAVE_SPEECH_AUDIO:
            return audio_dir / filename
        return Path("/tmp") / filename

    print("Fetching question audio...")
    question_audio = game.fetch_audio_question()
    question_path = audio_path_for("question")
    save_speech_audio(question_audio, question_path)
    transcript["audio_files"]["question"] = str(question_path)
    maybe_display_audio(question_audio)

    option_paths = []
    option_count = len(getattr(question, "options", []) or []) or 4
    for index in range(option_count):
        letter = option_letter(index)
        print(f"Fetching option {letter} audio...")
        option_audio = game.fetch_audio_option_next()
        option_path = audio_path_for("option", letter)
        save_speech_audio(option_audio, option_path)
        transcript["audio_files"][letter] = str(option_path)
        option_paths.append((letter, option_path))
        maybe_display_audio(option_audio)

    try:
        game.refresh_state()
        refreshed_question = game.current_question
        if refreshed_question is not None:
            question = refreshed_question
    except Exception as exc:
        print(f"Could not refresh game state after speech audio delivery: {exc}")

    print("Transcribing question audio...")
    question_text = transcribe_audio_file(question_path, initial_prompt="A multiple choice trivia question.")
    question.text = question_text
    transcript["question"] = question_text
    print("Question transcript:", question_text)

    option_texts = []
    for index, (letter, option_path) in enumerate(option_paths):
        option_text = transcribe_audio_file(option_path, initial_prompt=None, is_option=True, letter=letter)
        option_texts.append(option_text)
        transcript["options"].append({"letter": letter, "audio_file": str(option_path), "text": option_text})
        print(f"Option {letter} transcript: {option_text}")

    for index, option in enumerate(question.options):
        if index < len(option_texts):
            option.text = option_texts[index]

    transcript["transcription_seconds"] = time.time() - started_at
    try:
        transcript["seconds_left_after_audio"] = seconds_available(game)
    except NameError:
        transcript["seconds_left_after_audio"] = game.time_remaining
    transcript["whisper_model_size"] = globals().get("ACTIVE_WHISPER_MODEL_SIZE") or WHISPER_MODEL_SIZE
    transcript["whisper_device"] = globals().get("ACTIVE_WHISPER_DEVICE") or WHISPER_DEVICE
    return question, transcript


if LOAD_WHISPER_IN_SHARED_MODEL_CELL:
    print("Loading Whisper last, after Qwen 7B and Qwen Math are already loaded...")
    speech_whisper_model, speech_whisper_device = load_whisper_model_for_speech()
    print(f"Shared speech model ready: Whisper {globals().get('ACTIVE_WHISPER_MODEL_SIZE')} on {speech_whisper_device}")
else:
    print("Shared speech model preload skipped. Speech loops will load Whisper before starting the timer.")



Loading Whisper last, after Qwen 7B and Qwen Math are already loaded...
Before Whisper turbo load: free=6.31GB | total=14.56GB | allocated=8.04GB | reserved=8.13GB
Loading Whisper 'turbo' on cuda...


100%|█████████████████████████████████████| 1.51G/1.51G [00:21<00:00, 75.4MiB/s]


Whisper ready in 37.6s
After Whisper turbo load: free=1.84GB | total=14.56GB | allocated=11.06GB | reserved=12.61GB
Shared speech model ready: Whisper turbo on cuda


---
## 4. Entertainment / Science / Psychology Pipeline
_Competition IDs: 0 (Entertainment), 2 (Science & Nature), 4 (Philosophy & Psychology)_

Techniques: zero-shot, few-shot, chain-of-thought, Wikipedia RAG,
DuckDuckGo hybrid RAG, multi-model ensemble.


### 4.1 Zero-Shot & Prompt Variants

In [12]:
import re
import time
import torch

#zero-shot prompt
def build_zero_shot_prompt(question_text, options):
    # Format options as A) B) C) D)
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    return (
        f"Answer the following multiple choice question. "
        f"Reply with only the letter A, B, C, or D.\n\n"
        f"Question: {question_text}\n{opts}\n\nAnswer:"
    )

def extract_letter(text):
    # 1. Clean the text
    text = text.strip()

    # 2. Look for explicit patterns like "Answer: B" or "Final Answer: [B]" near the end
    match = re.search(r"(?:FINAL ANSWER|ANSWER|OPTION):\s*([A-D])", text.upper())
    if match:
        return match.group(1)

    # 3. Fallback: Find all isolated capital letters A, B, C, D and take the LAST one
    letters = re.findall(r"\b([A-D])\b", text.upper())
    if letters:
        return letters[-1]  # Takes the final decision made by the model

    return "A"  # Default fallback guess

def answer_with_model(question, prompt_fn=build_zero_shot_prompt, max_new_tokens=128):
    # Time the response
    t0 = time.time()

    # 1. Generate your standard question string
    raw_prompt = prompt_fn(question.text, question.options)

    # 2. Format it into the model's chat structure
    messages = [{"role": "user", "content": raw_prompt}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 3. Tokenize the formatted prompt
    inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    # 4. Generate the answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=False,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )

    # 5. CRITICAL FIX: Only extract tokens generated AFTER the prompt sequence length
    prompt_length = inputs.input_ids.shape[1]
    new_generated_tokens = outputs[0][prompt_length:]

    # Decode ONLY the new answer text
    response = tokenizer.decode(new_generated_tokens, skip_special_tokens=True)

    # Extract the choice letter from the clean text response
    letter = extract_letter(response)
    elapsed = time.time() - t0

    # Map letter to option ID (with safety lower boundary check)
    idx = ord(letter) - ord("A")
    idx = min(max(0, idx), len(question.options) - 1)

    return question.options[idx].id, letter, elapsed, response

print("Answer function defined.")

Answer function defined.


### 4.2 Generic Game Loop

In [13]:
# The game loop
def play_full_game(competition_id, answer_fn, label="Model", mode="text"):
    """
    Play a complete game and return results log.
    answer_fn: callable(question) -> (option_id, letter, elapsed, raw_response)
    mode: "text" or "speech". Speech mode fetches audio and transcribes it with Whisper.
    """
    if mode not in {"text", "speech"}:
        raise ValueError('mode must be either "text" or "speech"')

    if mode == "speech":
        load_whisper_model_for_speech()

    game = client.game.start(competition_id=competition_id, mode=mode)
    print(f"\n=== Game Started: {label} | Competition {competition_id} | Mode {game.mode} | Session {game.session_id} ===")

    log = []

    while game.in_progress:
        speech_transcription = None
        if game.mode == "speech":
            q, speech_transcription = transcribe_speech_question(game)
        else:
            q = game.current_question

        if not q:
            print("No question available, there is. Ending, the game is.")
            break

        time_left = game.time_remaining
        time_left_text = f"{time_left:.1f}s" if time_left is not None else "n/a"
        current_level = game.current_level
        print(f"\n--- Level {current_level} | Time left: {time_left_text} ---")
        if speech_transcription:
            seconds_left = speech_transcription.get("seconds_left_after_audio")
            print(
                "Speech transcription:",
                f"{speech_transcription.get('transcription_seconds', 0.0):.1f}s",
                f"| {seconds_left:.1f}s left after audio" if seconds_left is not None else "| time left n/a",
                f"| Whisper {speech_transcription.get('whisper_model_size')} on {speech_transcription.get('whisper_device')}",
            )
        print(f"Q: {q.text}")
        for opt in q.options:
            print(f"   [{opt.id}] {opt.text}")

        # Get answer from the model
        try:
            option_id, letter, elapsed, raw = answer_fn(q)
        except Exception as e:
            print(f"Model error, there is: {e}. Random answer, choosing we are.")
            option_id = random.choice(q.options).id
            letter, elapsed, raw = "?", 0.0, str(e)

        print(f"   -> Chose: {letter} (in {elapsed:.2f}s)")

        # Submit answer
        result = game.answer(option_id)

        entry = {
            "level": current_level,
            "question": q.text,
            "options": [{"id": opt.id, "text": opt.text} for opt in q.options],
            "correct": result.correct,
            "timed_out": result.timed_out,
            "elapsed": elapsed,
            "chosen_letter": letter,
            "earned": result.earned_amount,
            "model_raw": raw,
            "mode": game.mode,
            "speech_transcription": speech_transcription,
        }
        log.append(entry)

        if result.timed_out:
            print("   TIMEOUT!")
            break
        elif result.correct:
            print(f"   CORRECT! Earned: ${result.earned_amount:,.0f}")
            if result.game_over:
                print("   GAME COMPLETE!")
                break
        else:
            print(f"   WRONG! Final earnings: ${result.earned_amount:,.0f}")
            break

    print(f"\n=== Game Over | Reached Level: {game.current_level} | Earnings: ${game.earned_amount:,.0f} ===")
    return log, game.current_level, game.earned_amount


### 4.3 Run Baseline (Zero-Shot)

In [14]:
# Competition IDs: 0=Entertainment, 1=Ancient History & Politics, 2=Science & Nature, 3=Maths, 4=Philosophy & Psychology, 5=News
COMP_ID = 0

baseline_log, baseline_level, baseline_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=lambda q: answer_with_model(q, build_zero_shot_prompt),
    label="QWEN Zero-Shot",
    mode="text",
)


=== Game Started: QWEN Zero-Shot | Competition 0 | Mode text | Session 340352 ===

--- Level 1 | Time left: 29.9s ---
Q: Which of the following best describes Kanye West's impact on hip-hop music?
   [0] He introduced new production styles that facilitated the emergence of non-gangster rappers.
   [1] He focused solely on dance music production.
   [2] He discouraged collaboration between rappers and producers.
   [3] He popularized gangster rap.
   -> Chose: A (in 0.50s)
   CORRECT! Earned: $100

--- Level 2 | Time left: 29.9s ---
Q: What is the primary genre of music associated with Coldplay?
   [0] Hip-hop
   [1] Jazz
   [2] Electronic dance music
   [3] Rock
   -> Chose: D (in 0.45s)
   CORRECT! Earned: $200

--- Level 3 | Time left: 29.9s ---
Q: Which of the following films directed by Ridley Scott was selected for preservation in the United States National Film Registry by the Library of Congress for being considered 'culturally, historically, or aesthetically significant'?
   [

In [15]:
# Competition IDs: 0=Entertainment, 1=Ancient History & Politics, 2=Science & Nature, 3=Maths, 4=Philosophy & Psychology, 5=News
COMP_ID = 0

baseline_log, baseline_level, baseline_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=lambda q: answer_with_model(q, build_zero_shot_prompt),
    label="QWEN Zero-Shot",
    mode="speech",
)


=== Game Started: QWEN Zero-Shot | Competition 0 | Mode speech | Session 340355 ===
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: "What is the fundamental principle behind Radiohead's approach to their music?" -> "What is the fundamental principle behind Radiohead's approach to their music"
Question transcript: What is the fundamental principle behind Radiohead's approach to their music
Cleaned transcript noise: 'Option A, sticking to a single genre without variation.' -> 'sticking to a single genre without variation'
Option A transcript: sticking to a single genre without variation
Cleaned transcript noise: 'Option B, releasing only acoustic albums.' -> 'releasing only acoustic albums'
Option B transcript: releasing only acoustic albums
Cleaned transcript noise: 'Option C, focusing solely on traditional rock instruments.' -> 'focusing solel

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   -> Chose: D (in 0.46s)
   CORRECT! Earned: $100
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: "How does Marlon Brando's portrayal of Stanley Kowalski in A Streetcar Named Desire relate to his performance in The Wild One?" -> "How does Marlon Brando's portrayal of Stanley Kowalski in A Streetcar Named Desire relate to his performance in The Wild One"
Question transcript: How does Marlon Brando's portrayal of Stanley Kowalski in A Streetcar Named Desire relate to his performance in The Wild One
Cleaned transcript noise: 'Option A. Both roles showcase his ability to portray a tough, rebellious character.' -> 'Both roles showcase his ability to portray a tough, rebellious character'
Option A transcript: Both roles showcase his ability to portray a tough, rebellious character
Cleaned transcript noise: 'Option B, both roles were comedic and ligh

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   -> Chose: A (in 0.53s)
   CORRECT! Earned: $200
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'What is the name of the protagonist in Squid Game?' -> 'What is the name of the protagonist in Squid Game'
Question transcript: What is the name of the protagonist in Squid Game
Cleaned transcript noise: 'Option a Song Ji Hoon' -> 'Song Ji Hoon'
Option A transcript: Song Ji Hoon
Cleaned transcript noise: 'Hobson B? Oh, Illnum?' -> 'Hobson B? Oh, Illnum'
Option B transcript: Hobson B? Oh, Illnum
Cleaned transcript noise: 'Option C Toh Sang Woo' -> 'Toh Sang Woo'
Option C transcript: Toh Sang Woo
Cleaned transcript noise: 'Thompson D. Juan Juno.' -> 'Thompson D. Juan Juno'
Option D transcript: Thompson D. Juan Juno

--- Level 3 | Time left: 27.3s ---
Speech transcription: 4.0s | 27.3s left after audio | Whisper turbo on cuda
Q: What is the name of 

...

   -> Chose: A (in 0.46s)
   CORRECT! Earned: $300
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'That is the fundamental principle of music theory that defines the pitch of a note.' -> 'That is the fundamental principle of music theory that defines the pitch of a note'
Question transcript: That is the fundamental principle of music theory that defines the pitch of a note
Cleaned transcript noise: 'Option A, Harmony.' -> 'Harmony'
Option A transcript: Harmony
Cleaned transcript noise: 'Option B Dumber' -> 'Dumber'
Option B transcript: Dumber
Cleaned transcript noise: 'Option C. Pitch!' -> 'Pitch'
Option C transcript: Pitch
Cleaned transcript noise: 'Option D, rhythm.' -> 'rhythm'
Option D transcript: rhythm

--- Level 4 | Time left: 27.3s ---
Speech transcription: 4.2s | 27.3s left after audio | Whisper turbo on cuda
Q: That is the fundamenta

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   -> Chose: C (in 0.46s)
   CORRECT! Earned: $500
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: "Oh, what is the primary connection between David Bowie's early school experiences and his future career as a musician?" -> "what is the primary connection between David Bowie's early school experiences and his future career as a musician"
Question transcript: what is the primary connection between David Bowie's early school experiences and his future career as a musician
Cleaned transcript noise: 'Option A, his involvement in school sports teams.' -> 'his involvement in school sports teams'
Option A transcript: his involvement in school sports teams
Cleaned transcript noise: 'Option B, his participation in school bans.' -> 'his participation in school bans'
Option B transcript: his participation in school bans
Cleaned transcript noise: 'Option C,

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   -> Chose: D (in 0.46s)
   WRONG! Final earnings: $500

=== Game Over | Reached Level: 5 | Earnings: $500 ===


### 4.4 Few-Shot & Chain-of-Thought Prompt Variants

In [16]:
# Entertainment specialized few-shot data
FEW_SHOT_EXAMPLES = [
    {
        "question": "Which movie won the Academy Award for Best Picture in 2020?",
        "options": ["A) 1917", "B) Parasite", "C) Joker", "D) Once Upon a Time in Hollywood"],
        "answer": "B"
    },
    {
        "question": "Who is widely recognized as the 'King of Pop'?",
        "options": ["A) Elvis Presley", "B) Prince", "C) Michael Jackson", "D) Madonna"],
        "answer": "C"
    }
]

def build_few_shot_prompt(question_text, options):
    shots = ""
    for ex in FEW_SHOT_EXAMPLES:
        shots += f"Question: {ex['question']}\n" + "\n".join(ex['options']) + f"\nAnswer: {ex['answer']}\n\n"
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    return (
        f"Answer multiple choice questions with only a single letter A, B, C, or D.\n\n"
        f"{shots}"
        f"Question: {question_text}\n{opts}\nAnswer:"
    )

def build_cot_prompt(question_text, options):
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    return (
        f"Answer the following question. Think briefly, then give your final answer as a single letter.\n\n"
        f"Question: {question_text}\n{opts}\n\n"
        f"Reasoning: Let me think step by step.\nFinal Answer:"
    )

print("Prompt variants successfully adjusted for Entertainment trivia parsing.")

Prompt variants successfully adjusted for Entertainment trivia parsing.


In [17]:
# Compare prompts offline on a sample - NOT via live game (save API calls)
# Manually define a test question to compare prompt styles
sample_text = "Which planet is known as the Red Planet?"

class FakeOption:
    def __init__(self, id_, text):
        self.id = id_
        self.text = text

sample_opts = [FakeOption(1,"Mars"), FakeOption(2,"Venus"), FakeOption(3,"Jupiter"), FakeOption(4,"Saturn")]

class FakeQ:
    def __init__(self):
        self.text = sample_text
        self.options = sample_opts

fq = FakeQ()

print("=== Zero-Shot ===")
print(build_zero_shot_prompt(fq.text, fq.options))
print("\n=== Few-Shot ===")
print(build_few_shot_prompt(fq.text, fq.options))
print("\n=== Chain-of-Thought ===")
print(build_cot_prompt(fq.text, fq.options))

=== Zero-Shot ===
Answer the following multiple choice question. Reply with only the letter A, B, C, or D.

Question: Which planet is known as the Red Planet?
A) Mars
B) Venus
C) Jupiter
D) Saturn

Answer:

=== Few-Shot ===
Answer multiple choice questions with only a single letter A, B, C, or D.

Question: Which movie won the Academy Award for Best Picture in 2020?
A) 1917
B) Parasite
C) Joker
D) Once Upon a Time in Hollywood
Answer: B

Question: Who is widely recognized as the 'King of Pop'?
A) Elvis Presley
B) Prince
C) Michael Jackson
D) Madonna
Answer: C

Question: Which planet is known as the Red Planet?
A) Mars
B) Venus
C) Jupiter
D) Saturn
Answer:

=== Chain-of-Thought ===
Answer the following question. Think briefly, then give your final answer as a single letter.

Question: Which planet is known as the Red Planet?
A) Mars
B) Venus
C) Jupiter
D) Saturn

Reasoning: Let me think step by step.
Final Answer:


In [19]:
# Run few-shot game - compare with baseline result above
fewshot_log, fewshot_level, fewshot_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=lambda q: answer_with_model(q, build_few_shot_prompt),
    label="Few-Shot",
    mode="text",
)


=== Game Started: Few-Shot | Competition 0 | Mode text | Session 340363 ===

--- Level 1 | Time left: 29.9s ---
Q: How does Joey's character Dr. Drake Ramoray's role on 'Days of Our Lives' end in 'Friends'?
   [0] He is promoted to a higher role
   [1] He quits the show voluntarily
   [2] He falls down an elevator shaft
   [3] He is killed off in a car accident
   -> Chose: C (in 0.60s)
   CORRECT! Earned: $100

--- Level 2 | Time left: 29.9s ---
Q: Which of the following best describes the fundamental principle of Nolan's filmmaking style?
   [0] Emphasis on special effects and visual spectacle
   [1] Relying solely on comic book source material
   [2] Exploration of complex philosophical themes
   [3] Focus on linear narrative structures
   -> Chose: C (in 0.55s)
   CORRECT! Earned: $200

--- Level 3 | Time left: 29.9s ---
Q: Which of the following best describes the concept of 'psychedelic pop' that was used to categorize Pink Floyd in the late 1960s?
   [0] Music that focuses sole

In [20]:
# Run few-shot game - compare with baseline result above
fewshot_log, fewshot_level, fewshot_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=lambda q: answer_with_model(q, build_few_shot_prompt),
    label="Few-Shot",
    mode="speech",
)


=== Game Started: Few-Shot | Competition 0 | Mode speech | Session 340365 ===
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: "What term describes Drake's achievement of having 13 Billboard Hot 100 No. 1 singles, a joint record for a male solo artist?" -> "What term describes Drake's achievement of having 13 Billboard Hot 100 No. 1 singles, a joint record for a male solo artist"
Question transcript: What term describes Drake's achievement of having 13 Billboard Hot 100 No. 1 singles, a joint record for a male solo artist
Cleaned transcript noise: 'Option A, highest grossing hip-hop touring artist.' -> 'highest grossing hip-hop touring artist'
Option A transcript: highest grossing hip-hop touring artist
Cleaned transcript noise: 'Option B, Artist of the Decade, 2010s.' -> 'Artist of the Decade, 2010s'
Option B transcript: Artist of the Decade, 

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   -> Chose: D (in 0.59s)
   CORRECT! Earned: $100
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: "Which of the following best describes the connection between Drake's personal life and his music?" -> "Which of the following best describes the connection between Drake's personal life and his music"
Question transcript: Which of the following best describes the connection between Drake's personal life and his music
Cleaned transcript noise: 'Option A, his music is primarily about political commentary.' -> 'his music is primarily about political commentary'
Option A transcript: his music is primarily about political commentary
Cleaned transcript noise: 'Option B, his music avoids mentioning personal life.' -> 'his music avoids mentioning personal life'
Option B transcript: his music avoids mentioning personal life
Cleaned transcript noise: 'Opti

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   -> Chose: C (in 0.54s)
   CORRECT! Earned: $200
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: "How did Eminem's relationship with his mother, Debbie Nelson, influence his early life and career?" -> "How did Eminem's relationship with his mother, Debbie Nelson, influence his early life and career"
Question transcript: How did Eminem's relationship with his mother, Debbie Nelson, influence his early life and career
Cleaned transcript noise: 'Option A, it provided him with financial support.' -> 'it provided him with financial support'
Option A transcript: it provided him with financial support
Cleaned transcript noise: 'Option B, it caused frequent relocation in different states.' -> 'it caused frequent relocation in different states'
Option B transcript: it caused frequent relocation in different states
Cleaned transcript noise: 'Option C, 

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   -> Chose: B (in 0.62s)
   WRONG! Final earnings: $200

=== Game Over | Reached Level: 3 | Earnings: $200 ===


### 4.5 RAG - Wikipedia (Entertainment / Science)

Uses `search_wikipedia_deep` + an entertainment-aware RAG prompt.

In [21]:
import requests
import re
import time

def search_wikipedia_deep(query, max_chars=1200):
    """
    Deeper Wikipedia extract grabber to catch tracklists, cast lists,
    and detailed table indexes missing from short summaries.
    """
    clean = re.sub(r'[^\w\s]', '', query)[:60].strip()

    # Phase 1: Search API to get the correct matching title
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        "action": "query", "list": "search",
        "srsearch": clean, "format": "json", "srlimit": 1
    }
    try:
        r = requests.get(search_url, params=search_params, timeout=5).json()
        results = r.get("query", {}).get("search", [])
        if not results:
            return ""
        title = results[0]["title"]

        # Phase 2: Request full un-summarized text section extract
        content_params = {
            "action": "query", "prop": "extracts",
            "explaintext": 1, "titles": title, "format": "json", "exintro": 0
        }
        resp = requests.get(search_url, params=content_params, timeout=5).json()
        pages = resp["query"]["pages"]
        page_id = list(pages.keys())[0]

        extract = pages[page_id].get("extract", "")
        return extract[:max_chars]
    except Exception:
        return ""

def extract_entertainment_query(question_text, options):
    """
    Concatenates target choices to the question search query
    so negative constraint tracking works properly.
    """
    # Isolate key elements like text in quotes (e.g. "Born to Die")
    quoted_terms = re.findall(r'"([^"]*)"', question_text)
    options_string = " ".join([o.text for o in options])

    # Strip common filler stop phrases
    clean_q = re.sub(r'(Which of these|is not|featured on|the standard version of|the following|correct answer)', '', question_text, flags=re.IGNORECASE)

    if quoted_terms:
        return f'"{quoted_terms[0]}" {options_string}'[:90]
    return f"{clean_q.strip()} {options_string}"[:90]

def build_entertainment_rag_prompt(question_text, options, context=""):
    """
    Advanced prompt directing process of elimination for negative properties (NOT, EXCEPT).
    """
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    ctx_block = f"Background Context Information:\n{context}\n\n" if context else ""

    return (
        f"{ctx_block}"
        f"Task: Solve this multiple choice entertainment trivia question based ONLY on the text above.\n"
        f"CRITICAL RULES:\n"
        f"1. If the question contains words like 'NOT', 'FALSE', or 'EXCEPT', use process of elimination. Eliminate any options explicitly verified by the context text and pick the one outlier that remains.\n"
        f"2. Output strictly a single capital letter matching the answer (A, B, C, or D).\n\n"
        f"Question: {question_text}\n"
        f"Options:\n{opts}\n\n"
        f"Final Answer (Letter Only):"
    )

def answer_with_rag(question):
    # Pass options array to feed query synthesis loop
    query = extract_entertainment_query(question.text, question.options)
    context = search_wikipedia_deep(query)

    if context:
        print(f"   [RAG Active] Retrieved context data frame for search parameters.")
    else:
        print("   [RAG Warning] Flying blind without structural text records.")

    return answer_with_model(
        question,
        lambda q_text, opts: build_entertainment_rag_prompt(q_text, opts, context),
        max_new_tokens=16 # Kept short to prevent model from writing chat justifications
    )

### 4.6 RAG - Wikipedia with News Query Extractor

Shares `search_wikipedia_deep`; swaps in a news-oriented query builder and prompt.
Useful for competition 5 warm-up runs against Wikipedia.

In [22]:
def build_news_rag_prompt(question_text, options, context=""):
    """
    Optimized for News and Events: Forces the model to carefully inspect
    historical timelines, official roles, and factual details from the Wikipedia context.
    """
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    ctx_block = f"Background Context Information (Wikipedia Extract):\n{context}\n\n" if context else ""

    return (
        f"{ctx_block}"
        f"Task: Solve this current events and news multiple choice question using the background text above.\n"
        f"CRITICAL RULES:\n"
        f"1. Pay close attention to exact dates, years, official political titles, and specific country/geographic actions mentioned in the text.\n"
        f"2. Output strictly a single capital letter matching the correct choice (A, B, C, or D).\n\n"
        f"Question: {question_text}\n"
        f"Options:\n{opts}\n\n"
        f"Final Answer (Letter Only):"
    )

def answer_with_rag(question):
    # Call the new news query extractor instead of entertainment
    query = extract_news_query(question.text, question.options)
    print(f"   [Wikipedia News Search] Query: '{query}'")

    # Keep using your deep lookup engine exactly as it was
    context = search_wikipedia_deep(query)

    if context:
        print(f"   [RAG Active] Retrieved context data frame from Wikipedia.")
    else:
        print("   [RAG Warning] Flying blind without structural text records.")

    return answer_with_model(
        question,
        lambda q_text, opts: build_news_rag_prompt(q_text, opts, context),
        max_new_tokens=16
    )

### 4.7 Run Wikipedia RAG Game

In [ ]:
# Run RAG game
rag_log, rag_level, rag_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=answer_with_rag,
    label="Model + Wikipedia RAG",
    mode="text",
)


=== Game Started: Model + Wikipedia RAG | Competition 0 | Mode text | Session 340381 ===

--- Level 1 | Time left: 29.9s ---
Q: Which of the following best describes James Cameron's early career before directing his first feature film?
   [0] He was a professional athlete
   [1] He was a film critic
   [2] He was a janitor at a high school
   [3] He was a successful truck driver
Model error, there is: name 'extract_news_query' is not defined. Random answer, choosing we are.
   -> Chose: ? (in 0.00s)
   CORRECT! Earned: $100

--- Level 2 | Time left: 29.9s ---
Q: What is the term used for the words of an opera?
   [0] Libretto
   [1] Chorus
   [2] Aria
   [3] Recitative
Model error, there is: name 'extract_news_query' is not defined. Random answer, choosing we are.
   -> Chose: ? (in 0.00s)
   WRONG! Final earnings: $100

=== Game Over | Reached Level: 2 | Earnings: $100 ===


In [ ]:
# Run RAG game
rag_log, rag_level, rag_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=answer_with_rag,
    label="Model + Wikipedia RAG",
    mode="speech",
)


=== Game Started: Model + Wikipedia RAG | Competition 0 | Mode speech | Session 340390 ===
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'What is the fundamental principle of The Godfather that sets it apart from other gangster films?' -> 'What is the fundamental principle of The Godfather that sets it apart from other gangster films'
Question transcript: What is the fundamental principle of The Godfather that sets it apart from other gangster films
Cleaned transcript noise: 'Option A, the use of advanced special effects and editing techniques.' -> 'the use of advanced special effects and editing techniques'
Option A transcript: the use of advanced special effects and editing techniques
Cleaned transcript noise: 'Option B, the focus on family and loyalty over violence.' -> 'the focus on family and loyalty over violence'
Option B transcript: 

### 4.8 Hybrid RAG - DuckDuckGo + Wikipedia

Tries DuckDuckGo first, falls back to Wikipedia. Best for pop-culture/entertainment.

In [27]:
# Multi-Source RAG Setup - DuckDuckGo Live Search + Wikipedia Fallback
import requests
import re

def search_duckduckgo(query, max_chars=600):
    """
    Queries DuckDuckGo's free API for an instant abstract summary.
    Perfect for pop-culture entities, famous tracks, actors, and media questions.
    """
    clean_query = query.strip()
    url = f"https://api.duckduckgo.com/?q={requests.utils.quote(clean_query)}&format=json&no_html=1"
    try:
        response = requests.get(url, timeout=4)
        if response.status_code == 200:
            data = response.json()
            # DuckDuckGo provides 'AbstractText' for broad definitions
            abstract = data.get("AbstractText", "")
            if abstract:
                return abstract[:max_chars]

            # Alternative fallback: look inside RelatedTopics list snippets
            related = data.get("RelatedTopics", [])
            if related and "Text" in related[0]:
                return related[0]["Text"][:max_chars]
    except Exception:
        pass
    return ""

def search_wikipedia(query, max_chars=600):
    """Search Wikipedia and return a summary snippet as a safety fallback."""
    clean = re.sub(r'[^\w\s]', '', query)[:60]
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{clean.replace(' ', '_')}"
    try:
        r = requests.get(url, timeout=4)
        if r.status_code == 200:
            extract = r.json().get("extract", "")
            if extract:
                return extract[:max_chars]
    except Exception:
        pass

    # Secondary deep title search fallback
    try:
        params = {
            "action": "query", "list": "search",
            "srsearch": query, "format": "json", "srlimit": 1
        }
        r = requests.get("https://en.wikipedia.org/w/api.php", params=params, timeout=4)
        results = r.json().get("query", {}).get("search", [])
        if results:
            title = results[0]["title"]
            return search_wikipedia(title, max_chars)
    except Exception:
        pass

    return ""

def extract_key_terms_with_options(question_text, options):
    """
    Combines question entities with candidate choices.
    This guarantees that the search checks for the tracks/choices explicitly.
    """
    # Isolate any quoted strings first (e.g. "Born to Die")
    quoted = re.findall(r'"([^"]*)"', question_text)
    options_str = " ".join([o.text for o in options])

    # Strip basic filler text
    clean_q = re.sub(r'(Which of these|is not|featured on|the standard version of|the following|correct answer)', '', question_text, flags=re.IGNORECASE)

    if quoted:
        return f'"{quoted[0]}" {options_str}'[:90]
    return f"{clean_q.strip()} {options_str}"[:90]

def build_hybrid_rag_prompt(question_text, options, context=""):
    """
    A smart prompt directing process of elimination when a context string is found.
    """
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    ctx_block = f"Background Context Document:\n{context}\n\n" if context else ""
    return (
        f"{ctx_block}"
        f"Task: Answer the multiple-choice trivia question based on the background context provided above.\n"
        f"Rule: If the question contains words like 'NOT', 'EXCEPT', or 'FALSE', eliminate options matched by the context and pick the outlier.\n\n"
        f"Question: {question_text}\n"
        f"Options:\n{opts}\n\n"
        f"Final Answer (Letter Only):"
    )

def answer_with_rag(question):
    """
    Ensemble Retrieval Engine: Queries DuckDuckGo first,
    then falls back to Wikipedia if no result is returned.
    """
    query = extract_key_terms_with_options(question.text, question.options)

    # Step 1: Try Live Web Summary via DuckDuckGo
    context = search_duckduckgo(query)
    source_used = "DuckDuckGo"

    # Step 2: Fall back to Wikipedia if DuckDuckGo came up empty
    if not context:
        context = search_wikipedia(query)
        source_used = "Wikipedia"

    if context:
        print(f"   [RAG Active] Context fetched via {source_used}: '{context[:70]}...'")
    else:
        print("   [RAG Warning] Both sources empty! Flying blind via zero-shot memory.")

    return answer_with_model(
        question,
        lambda q_text, opts: build_hybrid_rag_prompt(q_text, opts, context),
        max_new_tokens=16
    )

print("Hybrid RAG engine successfully compiled! DuckDuckGo + Wikipedia are active.")

Hybrid RAG engine successfully compiled! DuckDuckGo + Wikipedia are active.


In [40]:
# Run the Multi-Source Hybrid RAG on the Entertainment competition
rag_log, rag_level, rag_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=answer_with_rag,
    label="Qwen2.5 + Hybrid RAG (DDG + Wiki)",
    mode="text",
)


=== Game Started: Qwen2.5 + Hybrid RAG (DDG + Wiki) | Competition 0 | Mode text | Session 340461 ===

--- Level 1 | Time left: 29.9s ---
Q: What is the primary reason Joey Tribbiani pursued acting as a career?
   [0] To escape his father's profession
   [1] To become a pipefitter like his father
   [2] To work as a model
   [3] To follow in his father's footsteps
   [RAG Warning] Both sources empty! Flying blind via zero-shot memory.
   -> Chose: A (in 0.56s)
   CORRECT! Earned: $100

--- Level 2 | Time left: 29.9s ---
Q: How does the film 'Lawrence of Arabia' relate to the historical figure T. E. Lawrence?
   [0] It accurately portrays all of Lawrence's actions and decisions
   [1] It is a fictional story with no basis in reality
   [2] It focuses solely on Lawrence's archaeological work
   [3] It is a dramatized version of Lawrence's life and experiences
   [RAG Warning] Both sources empty! Flying blind via zero-shot memory.
   -> Chose: B (in 0.53s)
   WRONG! Final earnings: $100



In [33]:
# Run the Multi-Source Hybrid RAG on the Entertainment competition
rag_log, rag_level, rag_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=answer_with_rag,
    label="Qwen2.5 + Hybrid RAG (DDG + Wiki)",
    mode="speech",
)


=== Game Started: Qwen2.5 + Hybrid RAG (DDG + Wiki) | Competition 0 | Mode speech | Session 340420 ===
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'What was the main theme of the film The Terminator that James Cameron wrote and directed?' -> 'What was the main theme of the film The Terminator that James Cameron wrote and directed'
Question transcript: What was the main theme of the film The Terminator that James Cameron wrote and directed
Cleaned transcript noise: 'Option A, cybernetic warfare and the dangers of artificial intelligence.' -> 'cybernetic warfare and the dangers of artificial intelligence'
Option A transcript: cybernetic warfare and the dangers of artificial intelligence
Cleaned transcript noise: 'Option B alien invasions' -> 'alien invasions'
Option B transcript: alien invasions
Cleaned transcript noise: 'Option C, Romantic 

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   -> Chose: B (in 0.55s)
   WRONG! Final earnings: $0

=== Game Over | Reached Level: 1 | Earnings: $0 ===


### 4.9 Multi-Model Ensemble (Majority Vote)

In [34]:
# Ensemble: majority vote across models
def answer_ensemble(question):
    votes = {}
    details = []

    models_to_use = [
        ("ZeroShot", lambda q: answer_with_model(q, build_zero_shot_prompt)),
        ("FewShot",  lambda q: answer_with_model(q, build_few_shot_prompt)),
        ("CoT",      lambda q: answer_with_model(q, build_cot_prompt)),
    ]

    for name, fn in models_to_use:
        try:
            opt_id, letter, elapsed, raw = fn(question)
            votes[letter] = votes.get(letter, 0) + 1
            details.append((name, letter, opt_id, elapsed))
            print(f"   [{name}] voted: {letter}")
        except Exception as e:
            print(f"   [{name}] failed: {e}")

    if not votes:
        # All models failed, we choose random answer
        chosen = random.choice(question.options)
        return chosen.id, "?", 0.0, "All models failed"

    # Find most voted letter
    best_letter = max(votes, key=votes.get)
    print(f"   [ENSEMBLE] Majority vote -> {best_letter} ({votes[best_letter]}/{len(models_to_use)} votes)")

    # Find option ID for the winning letter
    idx = min(ord(best_letter) - ord("A"), len(question.options) - 1)
    chosen_id = question.options[idx].id
    avg_elapsed = sum(d[3] for d in details) / len(details)

    return chosen_id, best_letter, avg_elapsed, str(votes)

print("Ensemble function ready")

Ensemble function ready


### 4.10 Run Majority Vote on Entertainment

In [ ]:
# Run ensemble game, we shall
ensemble_log, ensemble_level, ensemble_earned = play_full_game(
    competition_id=0,
    answer_fn=answer_ensemble,
    label="Multi-Model Ensemble",
    mode="text",
)


=== Game Started: Multi-Model Ensemble | Competition 0 | Mode text | Session 340497 ===

--- Level 1 | Time left: 29.9s ---
Q: What is the term used for the words of an opera?
   [0] Aria
   [1] Recitative
   [2] Libretto
   [3] Chorus
   [ZeroShot] voted: C
   [FewShot] voted: C
   [CoT] voted: C
   [ENSEMBLE] Majority vote -> C (3/3 votes)
   -> Chose: C (in 0.53s)
   CORRECT! Earned: $100

--- Level 2 | Time left: 29.9s ---
Q: Which of the following best describes the relationship between Batman and the Joker in The Dark Knight?
   [0] Batman and the Joker are allies
   [1] Batman and the Joker are mortal enemies
   [2] Batman and the Joker are romantic partners
   [3] Batman and the Joker are friends
   [ZeroShot] voted: B
   [FewShot] voted: B
   [CoT] voted: B
   [ENSEMBLE] Majority vote -> B (3/3 votes)
   -> Chose: B (in 0.53s)
   CORRECT! Earned: $200

--- Level 3 | Time left: 29.9s ---
Q: Which of the following films was Stanley Kubrick known for pioneering the use of?
   [0

In [ ]:
# Run ensemble game, we shall
ensemble_log, ensemble_level, ensemble_earned = play_full_game(
    competition_id=0,
    answer_fn=answer_ensemble,
    label="Multi-Model Ensemble",
    mode="speech",
)


=== Game Started: Multi-Model Ensemble | Competition 0 | Mode speech | Session 340627 ===
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'Which method would you use to depict the vast desert landscapes in Lawrence of Arabia?' -> 'Which method would you use to depict the vast desert landscapes in Lawrence of Arabia'
Question transcript: Which method would you use to depict the vast desert landscapes in Lawrence of Arabia
Cleaned transcript noise: 'Option A, hands-drawn animation.' -> 'hands-drawn animation'
Option A transcript: hands-drawn animation
Cleaned transcript noise: 'Option B, computer-generated imagery, CGI.' -> 'computer-generated imagery, CGI'
Option B transcript: computer-generated imagery, CGI
Cleaned transcript noise: 'Option C, painting the desert scenes on a large canvas.' -> 'painting the desert scenes on a large canvas'
Opti

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: D
   [FewShot] voted: D
   [CoT] voted: D
   [ENSEMBLE] Majority vote -> D (3/3 votes)
   -> Chose: D (in 0.51s)
   CORRECT! Earned: $100
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: "Which term best describes Frank Sinatra's impact on American popular culture?" -> "Which term best describes Frank Sinatra's impact on American popular culture"
Question transcript: Which term best describes Frank Sinatra's impact on American popular culture
Cleaned transcript noise: 'Option A, pioneering singer.' -> 'pioneering singer'
Option A transcript: pioneering singer
Cleaned transcript noise: 'Option B, ground-breaking actor.' -> 'ground-breaking actor'
Option B transcript: ground-breaking actor
Cleaned transcript noise: 'Option C, Innovative Dancer.' -> 'Innovative Dancer'
Option C transcript: Innovative Dancer
Cleaned transcript n

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: A
   [FewShot] voted: A
   [CoT] voted: A
   [ENSEMBLE] Majority vote -> A (3/3 votes)
   -> Chose: A (in 0.52s)
   CORRECT! Earned: $200
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: "What is the fundamental principle behind Stanley Kubrick's filmmaking?" -> "What is the fundamental principle behind Stanley Kubrick's filmmaking"
Question transcript: What is the fundamental principle behind Stanley Kubrick's filmmaking
Cleaned transcript noise: 'Option A, emphasizing quick and efficient production methods.' -> 'emphasizing quick and efficient production methods'
Option A transcript: emphasizing quick and efficient production methods
Cleaned transcript noise: 'Option B, focusing on extensive research and meticulous attention to detail.' -> 'focusing on extensive research and meticulous attention to detail'
Option B transcr

...

   [ZeroShot] voted: C
   [FewShot] voted: C
   [CoT] voted: C
   [ENSEMBLE] Majority vote -> C (3/3 votes)
   -> Chose: C (in 0.56s)
   CORRECT! Earned: $8,000
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: "What is the fundamental principle of Madonna's music career according to her own statements?" -> "What is the fundamental principle of Madonna's music career according to her own statements"
Question transcript: What is the fundamental principle of Madonna's music career according to her own statements
Cleaned transcript noise: 'Option A. To achieve commercial success.' -> 'To achieve commercial success'
Option A transcript: To achieve commercial success
Cleaned transcript noise: "Option B, they'll focus solely on dance pop." -> "they'll focus solely on dance pop"
Option B transcript: they'll focus solely on dance pop
Cleaned transcript n

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: C
   [FewShot] voted: C
   [CoT] voted: C
   [ENSEMBLE] Majority vote -> C (3/3 votes)
   -> Chose: C (in 0.52s)
   CORRECT! Earned: $16,000
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'Which of the following individuals is not considered a fifth beetle?' -> 'Which of the following individuals is not considered a fifth beetle'
Question transcript: Which of the following individuals is not considered a fifth beetle
Cleaned transcript noise: 'Option A, George Martin.' -> 'George Martin'
Option A transcript: George Martin
Cleaned transcript noise: 'Option B, Pete Best.' -> 'Pete Best'
Option B transcript: Pete Best
Cleaned transcript noise: 'Option C, Brian Epstein.' -> 'Brian Epstein'
Option C transcript: Brian Epstein
Cleaned transcript noise: 'Thompson D. Tony Sheridan.' -> 'Thompson D. Tony Sheridan'
Option D transcrip

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: A
   [FewShot] voted: A
   [CoT] voted: A
   [ENSEMBLE] Majority vote -> A (3/3 votes)
   -> Chose: A (in 0.54s)
   WRONG! Final earnings: $16,000

=== Game Over | Reached Level: 10 | Earnings: $16,000 ===


### 4.11 Run Majority Vote on Science

In [58]:
# Run ensemble game, we shall
ensemble_log, ensemble_level, ensemble_earned = play_full_game(
    competition_id=2,
    answer_fn=answer_ensemble,
    label="Multi-Model Ensemble",
    mode="text",
)


=== Game Started: Multi-Model Ensemble | Competition 2 | Mode text | Session 341005 ===

--- Level 1 | Time left: 29.9s ---
Q: During a presentation on astronomy, Professor Williams discussed various measurements in space. Which objects was she referring to when she talked about how astronomical units (AU) were determined?
   [0] the Sun and Earth
   [1] the Sun and the closest star
   [2] the inner and outer planets
   [3] the Moon and Earth
   [ZeroShot] voted: A
   [FewShot] voted: A
   [CoT] voted: A
   [ENSEMBLE] Majority vote -> A (3/3 votes)
   -> Chose: A (in 0.58s)
   CORRECT! Earned: $100

--- Level 2 | Time left: 29.9s ---
Q: In order to carry out the functions of life, organisms must
   [0] have at least one cell.
   [1] be able to move.
   [2] have more than one type of cell.
   [3] be exposed to sunlight.
   [ZeroShot] voted: A
   [FewShot] voted: A
   [CoT] voted: A
   [ENSEMBLE] Majority vote -> A (3/3 votes)
   -> Chose: A (in 0.54s)
   CORRECT! Earned: $200

--- Leve

In [59]:
# Run ensemble game, we shall
ensemble_log, ensemble_level, ensemble_earned = play_full_game(
    competition_id=2,
    answer_fn=answer_ensemble,
    label="Multi-Model Ensemble",
    mode="speech",
)


=== Game Started: Multi-Model Ensemble | Competition 2 | Mode speech | Session 341021 ===
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'Which interpretation of quantum mechanics is known for its emphasis on the role of the observer in wave function collapse?' -> 'Which interpretation of quantum mechanics is known for its emphasis on the role of the observer in wave function collapse'
Question transcript: Which interpretation of quantum mechanics is known for its emphasis on the role of the observer in wave function collapse
Cleaned transcript noise: 'Option A, objective columns, interpretation.' -> 'objective columns, interpretation'
Option A transcript: objective columns, interpretation
Cleaned transcript noise: 'Option B Many Worlds Interpretation' -> 'Many Worlds Interpretation'
Option B transcript: Many Worlds Interpretation
Cleaned tra

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: D
   [FewShot] voted: D
   [CoT] voted: D
   [ENSEMBLE] Majority vote -> D (3/3 votes)
   -> Chose: D (in 0.51s)
   CORRECT! Earned: $100
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'Which of the following best describes a scientific theory?' -> 'Which of the following best describes a scientific theory'
Question transcript: Which of the following best describes a scientific theory
Cleaned transcript noise: 'Option A. A scientific theory is an explanation that can be repeatedly tested and has corroborating evidence.' -> 'A scientific theory is an explanation that can be repeatedly tested and has corroborating evidence'
Option A transcript: A scientific theory is an explanation that can be repeatedly tested and has corroborating evidence
Cleaned transcript noise: 'Option B, a scientific theory is a law that describes the

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: A
   [FewShot] voted: A
   [CoT] voted: A
   [ENSEMBLE] Majority vote -> A (3/3 votes)
   -> Chose: A (in 0.53s)
   CORRECT! Earned: $200
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'In the 1600s, a scientist named Anton van Leeuwenhoek worked toward improving microscopes and created his own lenses. With his microscopes, Leeuwenhoek discovered bacteria and muscle fibers. Which type of job benefited most from these discoveries?' -> 'In the 1600s, a scientist named Anton van Leeuwenhoek worked toward improving microscopes and created his own lenses. With his microscopes, Leeuwenhoek discovered bacteria and muscle fibers. Which type of job benefited most from these discoveries'
Question transcript: In the 1600s, a scientist named Anton van Leeuwenhoek worked toward improving microscopes and created his own lenses. With his

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: D
   [FewShot] voted: D
   [CoT] voted: D
   [ENSEMBLE] Majority vote -> D (3/3 votes)
   -> Chose: D (in 0.57s)
   CORRECT! Earned: $300
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: "Which trade is common in gases that contribute to Earth's greenhouse effect?" -> "Which trade is common in gases that contribute to Earth's greenhouse effect"
Question transcript: Which trade is common in gases that contribute to Earth's greenhouse effect
Cleaned transcript noise: 'Option A, a tendency to lose electrons.' -> 'a tendency to lose electrons'
Option A transcript: a tendency to lose electrons
Cleaned transcript noise: 'Option B, the ability to trapeze.' -> 'the ability to trapeze'
Option B transcript: the ability to trapeze
Cleaned transcript noise: 'Option C, a tendency to exist as diatomic molecules.' -> 'a tendency to exist a

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: C
   [FewShot] voted: C
   [CoT] voted: C
   [ENSEMBLE] Majority vote -> C (3/3 votes)
   -> Chose: C (in 0.51s)
   WRONG! Final earnings: $300

=== Game Over | Reached Level: 4 | Earnings: $300 ===


### 4.12 Run Majority Vote on Philosophy

In [62]:
# Run ensemble game, we shall
ensemble_log, ensemble_level, ensemble_earned = play_full_game(
    competition_id=4,
    answer_fn=answer_ensemble,
    label="Multi-Model Ensemble",
    mode="text",
)


=== Game Started: Multi-Model Ensemble | Competition 4 | Mode text | Session 341058 ===

--- Level 1 | Time left: 29.9s ---
Q: Which of the following best describes Judith Butler's field of study?
   [0] Philosophy
   [1] History
   [2] Physics
   [3] Biology
   [ZeroShot] voted: A
   [FewShot] voted: A
   [CoT] voted: A
   [ENSEMBLE] Majority vote -> A (3/3 votes)
   -> Chose: A (in 0.55s)
   CORRECT! Earned: $100

--- Level 2 | Time left: 29.9s ---
Q: What term describes the principle of treating others the way you would want to be treated?
   [0] Golden Rule
   [1] Silver Rule
   [2] Platinum Rule
   [3] Iron Rule
   [ZeroShot] voted: A
   [FewShot] voted: A
   [CoT] voted: A
   [ENSEMBLE] Majority vote -> A (3/3 votes)
   -> Chose: A (in 0.55s)
   CORRECT! Earned: $200

--- Level 3 | Time left: 29.9s ---
Q: In 'No One Is Talking About This', the protagonist's engagement with social media primarily reflects which psychological concept?
   [0] Existentialism
   [1] Behavioral condit

In [63]:
# Run ensemble game, we shall
ensemble_log, ensemble_level, ensemble_earned = play_full_game(
    competition_id=4,
    answer_fn=answer_ensemble,
    label="Multi-Model Ensemble",
    mode="speech",
)


=== Game Started: Multi-Model Ensemble | Competition 4 | Mode speech | Session 341069 ===
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'Which of the following best describes the core principle of Confucianism?' -> 'Which of the following best describes the core principle of Confucianism'
Question transcript: Which of the following best describes the core principle of Confucianism
Cleaned transcript noise: 'Option A, encouraging the abandonment of traditional values.' -> 'encouraging the abandonment of traditional values'
Option A transcript: encouraging the abandonment of traditional values
Cleaned transcript noise: 'Option B, focusing on the worship of a single deity.' -> 'focusing on the worship of a single deity'
Option B transcript: focusing on the worship of a single deity
Cleaned transcript noise: 'Option C, emphasizing military power

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: D
   [FewShot] voted: D
   [CoT] voted: D
   [ENSEMBLE] Majority vote -> D (3/3 votes)
   -> Chose: D (in 0.51s)
   CORRECT! Earned: $100
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'What does the term credence refer to in Bayesian epistemology?' -> 'What does the term credence refer to in Bayesian epistemology'
Question transcript: What does the term credence refer to in Bayesian epistemology
Cleaned transcript noise: 'Option A, the degree of certainty one has about a belief on a scale from zero to one.' -> 'the degree of certainty one has about a belief on a scale from zero to one'
Option A transcript: the degree of certainty one has about a belief on a scale from zero to one
Cleaned transcript noise: 'Option B, the process of changing beliefs based on new evidence.' -> 'the process of changing beliefs based on new ev

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: A
   [FewShot] voted: A
   [CoT] voted: A
   [ENSEMBLE] Majority vote -> A (3/3 votes)
   -> Chose: A (in 0.54s)
   CORRECT! Earned: $200
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'Which of the following best describes the term virtue signaling?' -> 'Which of the following best describes the term virtue signaling'
Question transcript: Which of the following best describes the term virtue signaling
Cleaned transcript noise: 'Option A. Expressing a moral viewpoint to gain social approval.' -> 'Expressing a moral viewpoint to gain social approval'
Option A transcript: Expressing a moral viewpoint to gain social approval
Cleaned transcript noise: 'Option B, a method to solve moral dilemmas.' -> 'a method to solve moral dilemmas'
Option B transcript: a method to solve moral dilemmas
Cleaned transcript noise: 'Option C, the

...

   [ZeroShot] voted: A
   [FewShot] voted: A
   [CoT] voted: A
   [ENSEMBLE] Majority vote -> A (3/3 votes)
   -> Chose: A (in 0.53s)
   CORRECT! Earned: $16,000
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'What does the demandingness, objection critique, and utilitarianism?' -> 'What does the demandingness, objection critique, and utilitarianism'
Question transcript: What does the demandingness, objection critique, and utilitarianism
Cleaned transcript noise: 'Option A, the importance of maintaining social hierarchies.' -> 'the importance of maintaining social hierarchies'
Option A transcript: the importance of maintaining social hierarchies
Cleaned transcript noise: 'Option B, the necessity of following divine commands.' -> 'the necessity of following divine commands'
Option B transcript: the necessity of following divine commands
Cleaned

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: D
   [FewShot] voted: D
   [CoT] voted: D
   [ENSEMBLE] Majority vote -> D (3/3 votes)
   -> Chose: D (in 0.58s)
   CORRECT! Earned: $32,000
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'In the context of blood brotherhood, what was the limitation set by different culture regarding the number of blood brotherhoods man could have at one time?' -> 'In the context of blood brotherhood, what was the limitation set by different culture regarding the number of blood brotherhoods man could have at one time'
Question transcript: In the context of blood brotherhood, what was the limitation set by different culture regarding the number of blood brotherhoods man could have at one time
Cleaned transcript noise: 'Option A. Three.' -> 'Three'
Option A transcript: Three
Cleaned transcript noise: 'Option B! One!' -> 'One'
Option B trans

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


   [ZeroShot] voted: B
   [FewShot] voted: B
   [CoT] voted: B
   [ENSEMBLE] Majority vote -> B (3/3 votes)
   -> Chose: B (in 0.52s)
   WRONG! Final earnings: $32,000

=== Game Over | Reached Level: 11 | Earnings: $32,000 ===


---
## 5. History Pipeline
_Competition ID: 1 (Ancient History & Politics)_

Techniques: PyTerrier BM25 + sentence-embedding reranker + optional cross-encoder,
direct logit scoring with shuffled option orders, agentic tool router.


In [64]:
comp_id = 1  # Ancient History & Politics

### 5.1 Wikipedia Retrieval Helpers

In [65]:
import json
import re
from urllib.parse import quote, urlencode
from urllib.error import HTTPError
from urllib.request import Request, urlopen
import time

WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_USER_AGENT = "PoliMillionaireNLP/1.0 student project"
WIKIPEDIA_REQUEST_DELAY_SECONDS = 0.8
WIKIPEDIA_429_BACKOFF_SECONDS = 4.0
WIKIPEDIA_MAX_RETRIES = 2
MAX_WIKIPEDIA_SEARCH_QUERIES = 2
_LAST_WIKIPEDIA_REQUEST_TIME = 0.0
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "by", "for", "from",
    "has", "have", "how", "in", "is", "it", "its", "of", "on", "or", "that",
    "the", "their", "there", "these", "this", "those", "to", "was", "were",
    "what", "when", "where", "which", "who", "why", "with", "according", "article",
    "considered", "important", "goal", "goals", "main", "primary", "following"
}


def question_to_text(question) -> str:
    """Accept a string, a dict, or a millionaire_client Question object."""
    if hasattr(question, "text"):
        return str(question.text)
    if isinstance(question, dict) and "text" in question:
        return str(question["text"])
    return str(question)


def normalize_wikipedia_text(text: str) -> str:
    """Clean a plain Wikipedia extract enough for later NLP steps."""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text: str) -> list[str]:
    return [token for token in re.findall(r"[a-z0-9]+", str(text).lower()) if len(token) > 1]


def expand_term(token: str) -> set[str]:
    """Tiny synonym/variant helper for common historical wording traps."""
    variants = {token}
    if token == "roman":
        variants.update({"rome", "romans"})
    elif token in {"rome", "romans"}:
        variants.add("roman")
    return variants


def extract_keywords(text: str, limit: int = 10) -> list[str]:
    keywords = []
    seen = set()
    for token in tokenize(text):
        if token in STOPWORDS or token in seen:
            continue
        keywords.append(token)
        seen.add(token)
        if len(keywords) >= limit:
            break
    return keywords


def capital_context_phrases(question_text: str) -> list[str]:
    """Build focused phrases like 'Roman marriage' from capitalized topic words."""
    words = re.findall(r"[A-Za-z][A-Za-z'-]*", question_text)
    phrases = []
    for index, word in enumerate(words):
        if not word[:1].isupper() or word.lower() in STOPWORDS:
            continue
        phrase_words = [word]
        for next_word in words[index + 1:index + 4]:
            if next_word.lower() in STOPWORDS:
                break
            phrase_words.append(next_word)
        if len(phrase_words) > 1:
            phrases.append(" ".join(phrase_words))
    return phrases


def raw_option_text(option) -> str:
    """Read option text without depending on later notebook cells."""
    if hasattr(option, "text"):
        return str(option.text)
    if isinstance(option, dict):
        return str(option.get("text", ""))
    return str(option)


def question_options(question, options=None) -> list:
    """Return answer options from an explicit argument or a Question object."""
    if options is not None:
        return list(options)
    if hasattr(question, "options"):
        return list(question.options)
    if isinstance(question, dict) and "options" in question:
        return list(question["options"])
    return []


def dedupe_queries(queries: list[str]) -> list[str]:
    deduped = []
    seen = set()
    for query in queries:
        normalized = normalize_wikipedia_text(query).lower()
        if normalized and normalized not in seen:
            deduped.append(query)
            seen.add(normalized)
    return deduped


def build_base_wikipedia_search_queries(question) -> list[str]:
    """Create focused question-only Wikipedia queries."""
    question_text = question_to_text(question)
    cleaned = re.sub(r"\baccording to (?:the )?(?:article|text|passage)\b", " ", question_text, flags=re.IGNORECASE)
    cleaned = normalize_wikipedia_text(cleaned)
    keywords = extract_keywords(cleaned, limit=10)

    capital_words = []
    for word in re.findall(r"[A-Za-z][A-Za-z'-]*", cleaned):
        normalized_word = word.strip("'-")
        lowered = normalized_word.lower()
        if normalized_word[:1].isupper() and lowered not in STOPWORDS and len(lowered) > 2:
            capital_words.append(normalized_word)

    queries = []
    if len(capital_words) >= 2:
        queries.append(" ".join(capital_words[:4]))
    queries.extend(capital_context_phrases(cleaned))
    if keywords:
        queries.append(" ".join(keywords[:6]))
    if len(keywords) >= 2:
        queries.append(" ".join(keywords[:2]))
    queries.append(cleaned)
    queries.append(question_text)
    return dedupe_queries(queries)


def build_option_wikipedia_search_queries(question, options=None) -> list[str]:
    """Create one concise search query per option, balanced across all choices."""
    question_text = question_to_text(question)
    cleaned = re.sub(r"\baccording to (?:the )?(?:article|text|passage)\b", " ", question_text, flags=re.IGNORECASE)
    cleaned = normalize_wikipedia_text(cleaned)
    question_keywords = extract_keywords(cleaned, limit=6)
    queries = []
    for option in question_options(question, options):
        option_keywords = extract_keywords(raw_option_text(option), limit=6)
        if option_keywords:
            queries.append(" ".join((question_keywords[:4] + option_keywords[:4])[:8]))
    return dedupe_queries(queries)




def wikipedia_request(params: dict, timeout: float = 6.0, deadline_monotonic=None) -> dict:
    global _LAST_WIKIPEDIA_REQUEST_TIME

    url = f"{WIKIPEDIA_API}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": WIKIPEDIA_USER_AGENT})

    for attempt in range(WIKIPEDIA_MAX_RETRIES + 1):
        if deadline_monotonic is not None and time.monotonic() >= deadline_monotonic:
            raise TimeoutError("Wikipedia request skipped because the question deadline was reached")

        elapsed_since_last = time.monotonic() - _LAST_WIKIPEDIA_REQUEST_TIME
        sleep_for = WIKIPEDIA_REQUEST_DELAY_SECONDS - elapsed_since_last
        if sleep_for > 0:
            if deadline_monotonic is not None:
                remaining = deadline_monotonic - time.monotonic()
                if remaining <= 0:
                    raise TimeoutError("Wikipedia delay skipped because the question deadline was reached")
                sleep_for = min(sleep_for, remaining)
            time.sleep(sleep_for)

        request_timeout = timeout
        if deadline_monotonic is not None:
            remaining = deadline_monotonic - time.monotonic()
            if remaining <= 0:
                raise TimeoutError("Wikipedia request skipped because the question deadline was reached")
            request_timeout = min(timeout, max(0.25, remaining))

        try:
            with urlopen(request, timeout=request_timeout) as response:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic()
                data = json.loads(response.read().decode("utf-8"))
                return data

        except HTTPError as exc:
            if exc.code == 429:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic() - WIKIPEDIA_REQUEST_DELAY_SECONDS
                raise TimeoutError("Wikipedia rate limited; skipping live retry inside timed game")
            if attempt >= WIKIPEDIA_MAX_RETRIES:
                _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic() - WIKIPEDIA_REQUEST_DELAY_SECONDS
                raise

            retry_after = exc.headers.get("Retry-After")
            try:
                wait_seconds = float(retry_after) if retry_after else WIKIPEDIA_429_BACKOFF_SECONDS * (attempt + 1)
            except ValueError:
                wait_seconds = WIKIPEDIA_429_BACKOFF_SECONDS * (attempt + 1)

            if deadline_monotonic is not None:
                remaining = deadline_monotonic - time.monotonic()
                if remaining <= 0 or wait_seconds >= remaining:
                    _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic() - WIKIPEDIA_REQUEST_DELAY_SECONDS
                    raise TimeoutError("Wikipedia rate-limit backoff would exceed the question deadline")
                wait_seconds = min(wait_seconds, remaining)

            print(f"Wikipedia rate limit hit. Waiting {wait_seconds:.1f}s before retry {attempt + 1}/{WIKIPEDIA_MAX_RETRIES}...")
            time.sleep(wait_seconds)
            _LAST_WIKIPEDIA_REQUEST_TIME = time.monotonic()



def search_wikipedia(query: str, limit: int = 5, timeout: float = 6.0, deadline_monotonic=None) -> list[dict]:
    """Search Wikipedia and return candidate pages for one query string."""
    query = normalize_wikipedia_text(query)
    data = wikipedia_request(
        {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": limit,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
        deadline_monotonic=deadline_monotonic,
    )

    results = data.get("query", {}).get("search", [])
    formatted_results = [
        {
            "title": item.get("title", ""),
            "page_id": item.get("pageid"),
            "snippet": normalize_wikipedia_text(re.sub(r"<[^>]+>", " ", item.get("snippet", ""))),
            "query": query,
            "search_rank": rank,
        }
        for rank, item in enumerate(results, start=1)
    ]
    return formatted_results


def collect_wikipedia_candidates(question, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None, options=None, deadline_monotonic=None) -> list[dict]:
    """Search capped focused queries and deduplicate candidate pages by title."""
    candidates_by_title = {}
    base_queries = build_base_wikipedia_search_queries(question)
    option_queries = build_option_wikipedia_search_queries(question, options=options)
    if max_search_queries is None:
        max_search_queries = MAX_WIKIPEDIA_SEARCH_QUERIES
    if max_search_queries is not None:
        base_budget = max(1, int(max_search_queries * 0.6))
        option_budget = max(0, max_search_queries - base_budget)
        search_queries = dedupe_queries(base_queries[:base_budget] + option_queries[:option_budget])
        if len(search_queries) < max_search_queries:
            search_queries = dedupe_queries(search_queries + base_queries + option_queries)[:max_search_queries]
    else:
        search_queries = dedupe_queries(base_queries + option_queries)
    for query in search_queries:
        if deadline_monotonic is not None and time.monotonic() >= deadline_monotonic:
            print("Wikipedia search budget exhausted; using candidates collected so far.")
            break
        try:
            results = search_wikipedia(query, limit=per_query_limit, timeout=timeout, deadline_monotonic=deadline_monotonic)
        except Exception as exc:
            print(f"Wikipedia search skipped for {query!r}: {exc}")
            continue

        for result in results:
            title_key = result["title"].lower()
            if title_key not in candidates_by_title:
                candidates_by_title[title_key] = result
            else:
                candidates_by_title[title_key]["search_rank"] = min(
                    candidates_by_title[title_key]["search_rank"],
                    result["search_rank"],
                )
    return list(candidates_by_title.values())


def core_question_terms(question) -> list[str]:
    """Terms that must anchor retrieval: quoted terms and named entities in the question."""
    question_text = question_to_text(question)
    terms = []
    for phrase in re.findall(r"['\"]([^'\"]{3,80})['\"]", question_text):
        terms.extend(tokenize(phrase))
    for word in re.findall(r"\b[A-Z][A-Za-z0-9'-]{2,}\b", question_text):
        lowered = word.lower().strip("'-")
        if lowered not in STOPWORDS:
            terms.append(lowered)
    return list(dict.fromkeys(terms))[:8]


def candidate_relevance_score(candidate: dict, question, options=None) -> float:
    """Score title/snippet overlap with question keywords; penalize very generic one-word titles."""
    question_text = question_to_text(question)
    keywords = extract_keywords(question_text, limit=10)
    option_keywords = []
    for option in question_options(question, options):
        option_keywords.extend(extract_keywords(raw_option_text(option), limit=5))
    candidate_text = f"{candidate.get('title', '')} {candidate.get('snippet', '')}"
    candidate_terms = set(tokenize(candidate_text))

    matched = 0
    for keyword in keywords:
        if expand_term(keyword) & candidate_terms:
            matched += 1

    overlap = matched / max(1, len(keywords))
    title_terms = tokenize(candidate.get("title", ""))
    core_terms = core_question_terms(question)
    core_overlap = len([term for term in core_terms if expand_term(term) & candidate_terms]) / max(1, len(core_terms)) if core_terms else 1.0
    option_overlap = len(set(option_keywords) & candidate_terms) / max(1, len(set(option_keywords))) if option_keywords else 0.0
    rank_bonus = 1.0 / max(1, candidate.get("search_rank", 1))
    generic_penalty = 0.35 if len(title_terms) == 1 and len(keywords) > 1 else 0.0
    missing_core_penalty = 0.85 if core_terms and core_overlap == 0 else 0.0
    generic_title_penalty = 0.30 if title_terms and core_terms and not (set(title_terms) & set(core_terms)) and len(set(title_terms) & set(keywords)) <= 1 else 0.0

    return (1.6 * overlap) + (0.20 * option_overlap) + (0.80 * core_overlap) + (0.25 * rank_bonus) - generic_penalty - missing_core_penalty - generic_title_penalty


def rank_wikipedia_candidates(question, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None, options=None, deadline_monotonic=None) -> list[dict]:
    candidates = collect_wikipedia_candidates(question, per_query_limit=per_query_limit, timeout=timeout, max_search_queries=max_search_queries, options=options, deadline_monotonic=deadline_monotonic)
    for candidate in candidates:
        candidate["candidate_score"] = candidate_relevance_score(candidate, question, options=options)
    return sorted(candidates, key=lambda item: item["candidate_score"], reverse=True)


def fetch_wikipedia_extract(title: str, timeout: float = 6.0, deadline_monotonic=None) -> dict:
    """Fetch a Wikipedia page as a plain-text document."""
    data = wikipedia_request(
        {
            "action": "query",
            "prop": "extracts|info",
            "explaintext": 1,
            "exsectionformat": "plain",
            "inprop": "url",
            "titles": title,
            "format": "json",
            "utf8": 1,
            "redirects": 1,
        },
        timeout=timeout,
        deadline_monotonic=deadline_monotonic,
    )

    pages = data.get("query", {}).get("pages", {})
    page = next(iter(pages.values()), {}) if pages else {}
    document = {
        "title": page.get("title", title),
        "page_id": page.get("pageid"),
        "url": page.get("fullurl") or f"https://en.wikipedia.org/wiki/{quote(title.replace(' ', '_'))}",
        "text": normalize_wikipedia_text(page.get("extract", "")),
    }
    return document


def get_wikipedia_documents_for_question(question, top_n: int = 5, per_query_limit: int = 5, timeout: float = 6.0, max_search_queries=None, options=None, deadline_monotonic=None) -> list[dict]:
    """
    Return the top N related Wikipedia documents for a question.

    This avoids the trap of trusting only Wikipedia's first result for the full question.
    """
    query = question_to_text(question)
    candidates = rank_wikipedia_candidates(question, per_query_limit=per_query_limit, timeout=timeout, max_search_queries=max_search_queries, options=options, deadline_monotonic=deadline_monotonic)
    documents = []

    min_candidate_score = float(globals().get("MIN_WIKIPEDIA_CANDIDATE_SCORE", 0.0))
    for candidate in candidates[:top_n]:
        if candidate.get("candidate_score", 0.0) < min_candidate_score:
            print(f"Wikipedia candidate skipped for low relevance: {candidate.get('title')!r} score={candidate.get('candidate_score', 0.0):.3f}")
            continue
        if deadline_monotonic is not None and time.monotonic() >= deadline_monotonic:
            print("Wikipedia fetch budget exhausted; using documents fetched so far.")
            break
        try:
            document = fetch_wikipedia_extract(candidate["title"], timeout=timeout, deadline_monotonic=deadline_monotonic)
        except Exception as exc:
            print(f"Wikipedia page skipped for {candidate['title']!r}: {exc}")
            continue

        document["query"] = query
        document["matched_query"] = candidate.get("query")
        document["search_rank"] = candidate.get("search_rank")
        document["candidate_score"] = candidate.get("candidate_score", 0.0)
        document["snippet"] = candidate.get("snippet", "")
        document["search_results"] = candidates
        documents.append(document)

    return documents



### 5.2 Chunk Retrieval & Reranking (BM25 + Sentence Embeddings + Cross-Encoder)

In [66]:
import math
import time
import torch
import pyterrier as pt
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import shutil
import tempfile


def split_sentences(text: str) -> list[str]:
    """Sentence splitter for clean Wikipedia text."""
    text = normalize_wikipedia_text(text)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [sentence.strip() for sentence in sentences if len(sentence.strip()) >= 40]


def build_rag_chunks(documents: list[dict], sentences_per_chunk: int = 5, overlap: int = 2) -> list[dict]:
    """Split the top-N Wikipedia documents into overlapping evidence chunks."""
    chunks = []
    step = max(1, sentences_per_chunk - overlap)

    for doc_index, doc in enumerate(documents):
        sentences = split_sentences(doc.get("text", ""))
        for start in range(0, len(sentences), step):
            chunk_sentences = sentences[start:start + sentences_per_chunk]
            if not chunk_sentences:
                break
            chunk_text = " ".join(chunk_sentences)
            if len(chunk_text) < 120:
                continue
            chunks.append(
                {
                    "doc_index": doc_index,
                    "chunk_index": len(chunks),
                    "title": doc.get("title", ""),
                    "url": doc.get("url", ""),
                    "text": chunk_text,
                    "document_score": float(doc.get("candidate_score", 0.0)),
                }
            )
            if start + sentences_per_chunk >= len(sentences):
                break

    return chunks


def lexical_similarity(query: str, text: str) -> float:
    """Fallback score if sklearn is unavailable."""
    query_terms = set(extract_keywords(query, limit=20))
    text_terms = set(tokenize(text))
    if not query_terms or not text_terms:
        return 0.0
    overlap = len(query_terms & text_terms) / len(query_terms)
    return overlap


SENTENCE_EMBEDDING_MODEL_ID = globals().get(
    "SENTENCE_EMBEDDING_MODEL_ID",
    "sentence-transformers/all-MiniLM-L6-v2",
)
_SENTENCE_EMBEDDING_CACHE = globals().setdefault("_SENTENCE_EMBEDDING_CACHE", {})


def normalize_sentence_embedding_model_id(model_name: str | None = None) -> str:
    """Use one cache key for equivalent MiniLM model names across pipelines."""
    model_name = model_name or SENTENCE_EMBEDDING_MODEL_ID
    if model_name == "all-MiniLM-L6-v2":
        return SENTENCE_EMBEDDING_MODEL_ID
    return model_name


def load_sentence_embedding_model(model_name: str = SENTENCE_EMBEDDING_MODEL_ID):
    """Load a small sentence embedding model once and reuse it across pipelines."""
    model_name = normalize_sentence_embedding_model_id(model_name)
    if model_name not in _SENTENCE_EMBEDDING_CACHE:
        device = globals().get("SENTENCE_EMBEDDING_DEVICE", "cpu")
        _SENTENCE_EMBEDDING_CACHE[model_name] = SentenceTransformer(model_name, device=device)
    return _SENTENCE_EMBEDDING_CACHE[model_name]


CROSS_ENCODER_MODEL_ID = globals().get("CROSS_ENCODER_MODEL_ID", "cross-encoder/ms-marco-MiniLM-L-6-v2")
_CROSS_ENCODER_CACHE = globals().setdefault("_CROSS_ENCODER_CACHE", {})


def load_cross_encoder_model(model_name: str = CROSS_ENCODER_MODEL_ID):
    """Load an optional stronger reranker once and reuse it across pipelines."""
    if model_name not in _CROSS_ENCODER_CACHE:
        device = globals().get("CROSS_ENCODER_DEVICE", globals().get("SENTENCE_EMBEDDING_DEVICE", "cpu"))
        _CROSS_ENCODER_CACHE[model_name] = CrossEncoder(model_name, device=device)
    return _CROSS_ENCODER_CACHE[model_name]


def rerank_chunks_with_cross_encoder(question_text: str, ranked: list[dict], top_k: int = 8):
    """Rerank top chunks with a cross-encoder; return None if unavailable."""
    if not globals().get("USE_CROSS_ENCODER_RERANKER", False) or len(ranked) <= 1:
        return None

    candidate_count = min(len(ranked), max(top_k, globals().get("CROSS_ENCODER_RERANK_TOP_N", 12)))
    cross_weight = float(globals().get("CROSS_ENCODER_RERANK_WEIGHT", 0.55))
    candidates = ranked[:candidate_count]

    try:
        model = load_cross_encoder_model()
        pairs = [(question_text, f"{item.get('title', '')} {item.get('text', '')}") for item in candidates]
        cross_scores = model.predict(pairs)
    except Exception as exc:
        print(f"Cross-encoder reranker skipped, using sentence embedding/BM25 order: {exc}")
        return None

    cross_scores = [float(score) for score in cross_scores]
    min_cross = min(cross_scores, default=0.0)
    max_cross = max(cross_scores, default=1.0)
    cross_span = max(max_cross - min_cross, 1e-9)
    max_retrieval_score = max((float(item.get("retrieval_score", 0.0)) for item in candidates), default=1.0) or 1.0

    reranked = []
    for item, raw_cross_score in zip(candidates, cross_scores):
        enriched = dict(item)
        normalized_retrieval = float(enriched.get("retrieval_score", 0.0)) / max_retrieval_score
        normalized_cross = (raw_cross_score - min_cross) / cross_span
        enriched["cross_encoder_score"] = raw_cross_score
        enriched["pre_rerank_retrieval_score"] = enriched.get("retrieval_score")
        enriched["retrieval_score"] = ((1.0 - cross_weight) * normalized_retrieval) + (cross_weight * normalized_cross)
        enriched["retrieval_method"] = f"{enriched.get('retrieval_method', 'retrieval')}+cross_encoder"
        reranked.append(enriched)

    reranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
    return reranked[:top_k]


def rerank_chunks_with_sentence_embeddings(question_text: str, ranked: list[dict], top_k: int = 8) -> list[dict]:
    """Rerank the strongest lexical/BM25 chunks using semantic sentence similarity."""
    cross_encoder_ranked = rerank_chunks_with_cross_encoder(question_text, ranked, top_k=top_k)
    if cross_encoder_ranked is not None:
        return cross_encoder_ranked

    if not globals().get("USE_SENTENCE_EMBEDDING_RERANKER", True) or len(ranked) <= 1:
        return ranked[:top_k]

    candidate_count = min(len(ranked), max(top_k, globals().get("EMBEDDING_RERANK_TOP_N", 20)))
    embedding_weight = float(globals().get("EMBEDDING_RERANK_WEIGHT", 0.35))
    candidates = ranked[:candidate_count]

    try:

        model = load_sentence_embedding_model()
        chunk_texts = [f"{item.get('title', '')} {item.get('text', '')}" for item in candidates]
        question_embedding = model.encode([question_text], normalize_embeddings=True, convert_to_numpy=True)[0]
        chunk_embeddings = model.encode(chunk_texts, normalize_embeddings=True, convert_to_numpy=True)
        semantic_scores = np.matmul(chunk_embeddings, question_embedding)
    except Exception as exc:
        print(f"Sentence embedding reranker skipped, keeping BM25 order: {exc}")
        return ranked[:top_k]

    max_retrieval_score = max((float(item.get("retrieval_score", 0.0)) for item in candidates), default=1.0) or 1.0
    reranked = []
    for item, semantic_score in zip(candidates, semantic_scores):
        enriched = dict(item)
        normalized_retrieval = float(enriched.get("retrieval_score", 0.0)) / max_retrieval_score
        semantic_score = float(semantic_score)
        enriched["semantic_score"] = semantic_score
        enriched["pre_rerank_retrieval_score"] = enriched.get("retrieval_score")
        enriched["retrieval_score"] = ((1.0 - embedding_weight) * normalized_retrieval) + (embedding_weight * semantic_score)
        enriched["retrieval_method"] = f"{enriched.get('retrieval_method', 'retrieval')}+sentence_embedding"
        reranked.append(enriched)

    reranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
    return reranked[:top_k]


def ensure_pyterrier_started():
    """Import and initialize PyTerrier once for BM25 retrieval."""
    

    started = True
    if hasattr(pt, "started"):
        started = pt.started()
    elif hasattr(pt, "java") and hasattr(pt.java, "started"):
        started = pt.java.started()

    if not started:
        if hasattr(pt, "init"):
            pt.init()
        elif hasattr(pt, "java") and hasattr(pt.java, "init"):
            pt.java.init()

    return pt


def retrieve_rag_chunks_with_tfidf_fallback(question_text: str, chunks: list[dict], top_k: int = 8) -> list[dict]:
    """Fallback retriever used only when PyTerrier is unavailable."""
    chunk_texts = [f"{chunk['title']} {chunk['text']}" for chunk in chunks]

    try:

        vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        matrix = vectorizer.fit_transform([question_text] + chunk_texts)
        similarities = cosine_similarity(matrix[0:1], matrix[1:]).flatten()
        method = "tfidf_cosine_fallback"
    except Exception:
        similarities = [lexical_similarity(question_text, text) for text in chunk_texts]
        method = "lexical_overlap_fallback"

    ranked = []
    for chunk, similarity in zip(chunks, similarities):
        score = float(similarity) + 0.08 * chunk.get("document_score", 0.0)
        enriched = dict(chunk)
        enriched["retrieval_score"] = score
        enriched["retrieval_method"] = method
        ranked.append(enriched)

    ranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
    return rerank_chunks_with_sentence_embeddings(question_text, ranked, top_k=top_k)


def retrieve_rag_chunks(question, documents: list[dict], top_k: int = 8) -> list[dict]:
    """Retrieve the strongest chunks with PyTerrier BM25 over the Wikipedia chunks."""
    question_text = question_to_text(question)
    bm25_query = " ".join(extract_keywords(question_text, limit=30)) or question_text
    chunks = build_rag_chunks(documents)
    if not chunks:
        return []
    if not globals().get("USE_PYTERRIER_BM25", True):
        return retrieve_rag_chunks_with_tfidf_fallback(question_text, chunks, top_k=top_k)

    try:

        pt = ensure_pyterrier_started()
        index_dir = tempfile.mkdtemp(prefix="pt_rag_chunks_")
        try:
            indexer = pt.IterDictIndexer(index_dir, meta={"docno": 32}, overwrite=True)
            index_ref = indexer.index(
                {
                    "docno": str(index),
                    "text": f"{chunk.get('title', '')} {chunk.get('text', '')}",
                }
                for index, chunk in enumerate(chunks)
            )
            if hasattr(pt, "terrier") and hasattr(pt.terrier, "Retriever"):
                retriever = pt.terrier.Retriever(index_ref, wmodel="BM25", metadata=["docno"])
            else:
                retriever = pt.BatchRetrieve(index_ref, wmodel="BM25", metadata=["docno"])
            results = retriever.search(bm25_query)
        finally:
            shutil.rmtree(index_dir, ignore_errors=True)

        score_by_docno = {
            str(row.docno): float(row.score)
            for row in results.itertuples(index=False)
        }
        max_bm25 = max(score_by_docno.values(), default=0.0)
        if max_bm25 <= 0:
            return retrieve_rag_chunks_with_tfidf_fallback(question_text, chunks, top_k=top_k)

        ranked = []
        for index, chunk in enumerate(chunks):
            raw_bm25 = score_by_docno.get(str(index), 0.0)
            normalized_bm25 = raw_bm25 / max_bm25 if max_bm25 else 0.0
            score = normalized_bm25 + 0.08 * chunk.get("document_score", 0.0)
            enriched = dict(chunk)
            enriched["retrieval_score"] = score
            enriched["bm25_score"] = raw_bm25
            enriched["retrieval_method"] = "pyterrier_bm25"
            ranked.append(enriched)

        ranked.sort(key=lambda item: item["retrieval_score"], reverse=True)
        return rerank_chunks_with_sentence_embeddings(question_text, ranked, top_k=top_k)
    except Exception as exc:
        print(f"PyTerrier BM25 retrieval skipped, using fallback retrieval instead: {exc}")
        return retrieve_rag_chunks_with_tfidf_fallback(question_text, chunks, top_k=top_k)


def build_rag_context(hits: list[dict], max_chars: int = 4200) -> str:
    """Format retrieved chunks as compact evidence for generation."""
    blocks = []
    used = 0
    for index, hit in enumerate(hits, start=1):
        block = f"[Evidence {index} | {hit['title']}] {hit['text']}"
        if used + len(block) > max_chars:
            block = block[:max(0, max_chars - used)]
        if block.strip():
            blocks.append(block)
            used += len(block)
        if used >= max_chars:
            break
    return "\n\n".join(blocks)


def rank_answer_sentences(question, hits: list[dict], max_sentences: int = 5) -> list[str]:
    """Extract the most relevant evidence sentences for a extractive answer."""
    question_text = question_to_text(question)
    question_terms = set(tokenize(question_text))
    purpose_terms = {"goal", "purpose", "reason", "important", "considered", "used", "use", "tool", "primarily", "primary", "fundamental", "institution"}
    wants_purpose = bool(question_terms & purpose_terms)
    candidates = []
    seen = set()

    for hit_index, hit in enumerate(hits):
        for sentence_index, sentence in enumerate(split_sentences(hit.get("text", ""))):
            key = sentence.lower()
            if key in seen:
                continue
            seen.add(key)
            sentence_terms = set(tokenize(sentence))
            sentence_for_score = f"{hit.get('title', '')} {sentence}"
            score = lexical_similarity(question_text, sentence_for_score) + 0.15 * hit.get("retrieval_score", 0.0)
            if wants_purpose:
                score += 0.25 * len(sentence_terms & purpose_terms)
            if hit.get("title", "").lower() in sentence.lower():
                score += 0.05
            candidates.append((score, hit_index, sentence_index, sentence))

    candidates.sort(key=lambda item: item[0], reverse=True)
    return [sentence for _, _, _, sentence in candidates[:max_sentences]]


def extractive_rag_answer(question, hits: list[dict], max_sentences: int = 5) -> str:
    """Create a concise explanation paragraph from retrieved evidence sentences."""
    sentences = rank_answer_sentences(question, hits, max_sentences=max_sentences)
    if not sentences:
        return "I could not find enough evidence in the retrieved Wikipedia documents to answer confidently."
    return " ".join(sentences)


_ANSWER_MODEL_CACHE = globals().setdefault("_ANSWER_MODEL_CACHE", {})
if "tokenizer" in globals() and "model" in globals():
    _ANSWER_MODEL_CACHE.setdefault(MODEL_ID, (tokenizer, model))

def generate_rag_answer(question, hits: list[dict], model_name: str = MODEL_ID, max_new_tokens: int = 180) -> str:
    """Generate an explanatory RAG answer with the local answer model."""
    

    tokenizer, model = load_answer_model(model_name)
    question_text = question_to_text(question)
    context = build_rag_context(hits)

    messages = [
        {
            "role": "system",
            "content": "Answer the question using only the retrieved evidence. Be concise and do not invent facts.",
        },
        {
            "role": "user",
            "content": f"Evidence:\n{context}\n\nQuestion: {question_text}\n\nAnswer:",
        },
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\nAnswer:"

    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def answer_question_with_rag(
    question,
    documents: list[dict],
    top_k_chunks: int = 12,
    use_local_generator: bool = True,
    generator_model: str = MODEL_ID,
) -> dict:
    """
    Ask the question from RAG using the top-N documents, without using answer options.

    Returns an explanatory answer plus the retrieved evidence chunks.
    """
    hits = retrieve_rag_chunks(question, documents, top_k=top_k_chunks)
    method = "extractive_rag"

    if use_local_generator:
        try:
            answer = generate_rag_answer(question, hits, model_name=generator_model)
            method = f"answer_model_rag:{generator_model}"
        except Exception as exc:
            print(f"Local generator skipped, using extractive RAG instead: {exc}")
            answer = extractive_rag_answer(question, hits)
    else:
        answer = extractive_rag_answer(question, hits)

    return {
        "question": question_to_text(question),
        "answer": answer,
        "method": method,
        "evidence_chunks": hits,
    }

### 5.3 Answer Model Loader

In [67]:
import torch


def load_answer_model(model_name: str = MODEL_ID):
    """Return the shared answer model loaded in the setup cell.

    All pipelines use the same Qwen model instance. This helper intentionally
    does not call from_pretrained, so later pipeline cells cannot allocate a
    second copy of the 7B model by accident.
    """
    requested_model = model_name or MODEL_ID
    shared_model_id = MODEL_ID

    if requested_model in _ANSWER_MODEL_CACHE:
        return _ANSWER_MODEL_CACHE[requested_model]

    if requested_model == shared_model_id and "tokenizer" in globals() and "model" in globals():
        _ANSWER_MODEL_CACHE[requested_model] = (tokenizer, model)
        return tokenizer, model

    raise RuntimeError(
        f"Model {requested_model} is not loaded. Run the shared model loader cell first, "
        f"or change MODEL_ID there to {requested_model} and rerun from setup."
    )

### 5.4 Preload & Warm Up

In [68]:
import time
import torch

# Run this BEFORE starting a timed game.
# Hugging Face auth is no longer required for this notebook.



start_time = time.time()
print(f"Preloading {MODEL_ID} before the timed game...")
try:
    answer_tokenizer, answer_model = load_answer_model(MODEL_ID)
except Exception as exc:
    raise RuntimeError(
        f"Could not load answer model {MODEL_ID}. Restart the Colab runtime, run only the setup cells, "
        "or choose a smaller model if GPU memory is tight."
    ) from exc
ACTIVE_MODEL_ID = next((name for name, cached in _ANSWER_MODEL_CACHE.items() if cached == (answer_tokenizer, answer_model)), MODEL_ID)

print(f"Active model: {ACTIVE_MODEL_ID}")
if globals().get("USE_SENTENCE_EMBEDDING_RERANKER", False):
    print(f"Preloading sentence embedding model {SENTENCE_EMBEDDING_MODEL_ID}...")
    sentence_embedding_model = load_sentence_embedding_model()
if globals().get("USE_CROSS_ENCODER_RERANKER", False):
    print(f"Preloading cross-encoder reranker {CROSS_ENCODER_MODEL_ID}...")
    cross_encoder_model = load_cross_encoder_model()

# Small warm-up generation so first real RAG answer does not pay setup cost.
warmup_messages = [
    {"role": "system", "content": "Answer shortly."},
    {"role": "user", "content": "Say I wanna be a PoliMillionaire."},
]
warmup_prompt = answer_tokenizer.apply_chat_template(warmup_messages, tokenize=False, add_generation_prompt=True)
warmup_device = next(answer_model.parameters()).device
warmup_inputs = answer_tokenizer(warmup_prompt, return_tensors="pt").to(warmup_device)
with torch.inference_mode():
    _ = answer_model.generate(
        **warmup_inputs,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=answer_tokenizer.eos_token_id,
    )

print(f"Answer and retrieval models are loaded and warmed up in {time.time() - start_time:.1f}s.")
print("Now start the game / run the RAG answer cell.")

Preloading Qwen/Qwen2.5-7B-Instruct before the timed game...
Active model: Qwen/Qwen2.5-7B-Instruct


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Answer and retrieval models are loaded and warmed up in 1.1s.
Now start the game / run the RAG answer cell.


### 5.5 MCQ Prompt Builder

In [69]:
SYSTEM_PROMPT = """
You are an expert multiple choice quiz solver.

Carefully analyze the question.

Return ONLY the single best answer option.

Do not explain your reasoning.
Do not output extra text.
"""

def build_mcq_prompt(question):

    choices = []

    for i, opt in enumerate(question.options):
        letter = chr(ord("A") + i)
        choices.append(f"{letter}. {option_text(opt)}")

    joined = "\n".join(choices)

    return f"""
Question:
{question.text}

Options:
{joined}

Reply with ONLY one letter: A, B, C, or D.
"""

### 5.6 Direct Logit Scoring & Option Matching

In [70]:
import hashlib
import random
import torch
import torch.nn.functional as F
import numpy as np

LETTERS = "ABCD"


def option_text(option) -> str:
    return option.text if hasattr(option, "text") else option["text"]


def option_id(option) -> int:
    return option.id if hasattr(option, "id") else option["id"]


def parse_option_choice(text: str, option_count: int = 4):
    """Parse A-D or 0-3 from the model output."""
    cleaned = str(text).strip().upper()

    letter_match = re.search(r"\b([A-D])\b", cleaned)
    if letter_match:
        index = LETTERS.index(letter_match.group(1))
        return index if index < option_count else None

    digit_match = re.search(r"\b([0-3])\b", cleaned)
    if digit_match:
        index = int(digit_match.group(1))
        return index if index < option_count else None

    return None


def parse_option_choice_with_text(text: str, options):
    """Parse the chosen option, preferring an exact option-text mention over a possibly wrong letter."""
    normalized_output = normalize_match_text(text)
    text_matches = []
    for index, option in enumerate(options):
        normalized_option = normalize_match_text(option_text(option))
        if normalized_option and normalized_option in normalized_output:
            text_matches.append(index)
    if len(text_matches) == 1:
        return text_matches[0]
    return parse_option_choice(text, option_count=len(options))



def choose_option_direct_shuffled_logits(question, options, model_name: str = MODEL_ID) -> dict:
    """Average next-letter logit scores across shuffled option orders."""

    vote_count = max(3, int(globals().get("DIRECT_MODEL_VOTES", 3)))
    options = list(options)

    question_text = question_to_text(question)
    tokenizer, model = load_answer_model(model_name)
    device = next(model.parameters()).device

    score_lists = {index: [] for index in range(len(options))}
    vote_details = []

    for vote_number in range(vote_count):
        order = list(range(len(options)))

        if vote_number > 0:
            seed = int(
                hashlib.sha256(
                    f"{question_text}|logits|{vote_number}".encode("utf-8")
                ).hexdigest()[:12],
                16,
            )
            rng = random.Random(seed)
            rng.shuffle(order)

            if order == list(range(len(options))) and len(order) > 1:
                order = order[1:] + order[:1]

        option_lines = "\n".join(
            f"{LETTERS[display_index]}. {option_text(options[original_index])}"
            for display_index, original_index in enumerate(order)
        )

        user_prompt = f"""
Question:
{question_text}

Options:
{option_lines}

Reply with ONLY one letter: A, B, C, or D.
"""

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        if hasattr(tokenizer, "apply_chat_template"):
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = SYSTEM_PROMPT + "\n\n" + user_prompt

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1536,
        ).to(device)

        with torch.inference_mode():
            outputs = model(**inputs)
            log_probs = F.log_softmax(outputs.logits[0, -1], dim=-1)

        display_scores = []

        for display_index, original_index in enumerate(order):
            letter = LETTERS[display_index]
            variant_scores = []

            for variant in (letter, f" {letter}", f"{letter}.", f" {letter}."):
                token_ids = tokenizer.encode(variant, add_special_tokens=False)
                if token_ids:
                    variant_scores.append(float(log_probs[token_ids[0]].detach().cpu()))

            best_score = max(variant_scores) if variant_scores else float("-inf")

            score_lists[original_index].append(best_score)
            display_scores.append({
                "display_letter": letter,
                "answer_index": original_index,
                "logprob": best_score,
            })

        display_scores.sort(key=lambda item: item["logprob"], reverse=True)

        vote_details.append({
            "vote_number": vote_number + 1,
            "display_order": order,
            "ranked": display_scores,
        })

    averaged_scores = []

    for index, scores in score_lists.items():
        averaged_scores.append({
            "answer_index": index,
            "letter": LETTERS[index],
            "avg_logprob": sum(scores) / max(1, len(scores)),
            "logprobs": scores,
        })

    ranked = sorted(
        averaged_scores,
        key=lambda item: item["avg_logprob"],
        reverse=True,
    )

    selected_index = ranked[0]["answer_index"]
    margin = ranked[0]["avg_logprob"] - ranked[1]["avg_logprob"] if len(ranked) > 1 else 0.0
    selected_option = options[selected_index]

    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": selected_index,
        "letter": LETTERS[selected_index],
        "model_output": "shuffled_logit_scores:" + ", ".join(
            f"{item['letter']}={item['avg_logprob']:.3f}" for item in ranked
        ),
        "selection_source": f"direct_model_shuffled_logit_scoring_{vote_count}",
        "direct_logit_scores": ranked,
        "direct_logit_margin": margin,
        "direct_votes": vote_details,
        "option_scores": [],
    }


def choose_option_direct_voted(question, options, model_name: str = MODEL_ID) -> dict:
    """Fast direct answer with strict MCQ voting."""
    vote_count = int(globals().get("DIRECT_MODEL_VOTES", 1))
    vote_count = max(1, vote_count)

    tokenizer, model = load_answer_model(model_name)
    user_prompt = build_mcq_prompt(question)


    device = next(model.parameters()).device
    votes = []

    for _ in range(vote_count):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        if hasattr(tokenizer, "apply_chat_template"):
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = SYSTEM_PROMPT + "\n\n" + user_prompt

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1536,
        ).to(device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=2,
                do_sample=False,
                temperature=0.0,
                top_p=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        model_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        selected_index = parse_option_choice_with_text(model_output, options)

        if selected_index is not None:
            votes.append({
                "answer_index": selected_index,
                "letter": LETTERS[selected_index],
                "model_output": model_output,
            })

    if not votes:
        raise ValueError("Could not parse any direct model vote")

    counts = {}
    first_seen = {}

    for order, vote in enumerate(votes):
        index = vote["answer_index"]
        counts[index] = counts.get(index, 0) + 1
        first_seen.setdefault(index, order)

    selected_index = sorted(
        counts,
        key=lambda index: (-counts[index], first_seen[index]),
    )[0]

    selected_option = options[selected_index]

    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": selected_index,
        "letter": LETTERS[selected_index],
        "model_output": " | ".join(vote["model_output"] for vote in votes),
        "selection_source": f"direct_model_voted_{len(votes)}",
        "direct_votes": votes,
        "option_scores": [],
    }





def normalize_match_text(text: str) -> str:
    """Normalize text for cheap exact/near-exact option matching."""
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", str(text).lower())).strip()


def score_options_against_evidence(question, options, hits: list[dict]) -> list[dict]:
    """Score each option directly against retrieved evidence using fast lexical/exact matching."""
    question_text = question_to_text(question)
    evidence_texts = [f"{hit.get('title', '')} {hit.get('text', '')}" for hit in hits if hit.get("text")]
    evidence_blob = " ".join(evidence_texts)
    normalized_evidence = normalize_match_text(evidence_blob)
    question_terms = set(extract_keywords(question_text, limit=24))
    evidence_sentences = split_sentences(evidence_blob) if evidence_blob else []
    try:
        core_terms = core_question_terms(question)
    except NameError:
        core_terms = []
    evidence_terms = set(tokenize(evidence_blob)) if evidence_blob else set()
    evidence_core_overlap = len([term for term in core_terms if expand_term(term) & evidence_terms]) / max(1, len(core_terms)) if core_terms else 1.0
    topic_relevance_multiplier = 1.0 if evidence_core_overlap > 0 else 0.35
    scored = []

    lexical_scores = []
    for option in options:
        option_query = f"{question_text} {option_text(option)}"
        if evidence_texts:
            lexical_scores.append(max(lexical_similarity(option_query, evidence_text) for evidence_text in evidence_texts))
        else:
            lexical_scores.append(0.0)

    semantic_scores = [0.0 for _ in options]
    if evidence_texts and globals().get("USE_OPTION_EMBEDDING_SCORER", True):
        try:

            model = load_sentence_embedding_model()
            option_queries = [f"{question_text} {option_text(option)}" for option in options]
            option_embeddings = model.encode(option_queries, normalize_embeddings=True, convert_to_numpy=True)
            evidence_embeddings = model.encode(evidence_texts, normalize_embeddings=True, convert_to_numpy=True)
            similarities = np.matmul(option_embeddings, evidence_embeddings.T)
            semantic_scores = [float(row.max()) for row in similarities]
        except Exception as exc:
            print(f"Option embedding scorer skipped, using lexical option scores only: {exc}")

    for index, option in enumerate(options):
        lexical_score = float(lexical_scores[index])
        semantic_score = float(semantic_scores[index])
        normalized_option = normalize_match_text(option_text(option))
        option_terms = [term for term in normalized_option.split() if len(term) > 2]
        exact_match = bool(normalized_option and normalized_option in normalized_evidence)
        support_sentence_score = 0.0
        support_sentence = ""
        for sentence in evidence_sentences:
            normalized_sentence = normalize_match_text(sentence)
            if not normalized_option or normalized_option not in normalized_sentence:
                continue
            sentence_terms = set(tokenize(sentence))
            overlap = len(question_terms & sentence_terms) / max(1, len(question_terms))
            score = 0.75 + overlap
            if "only once" in normalized_sentence or "rarely" in normalized_sentence:
                score -= 0.75
            if score > support_sentence_score:
                support_sentence_score = score
                support_sentence = sentence[:260]
        term_coverage = len([term for term in option_terms if term in normalized_evidence]) / max(1, len(option_terms))
        exact_boost = float(globals().get("EXACT_OPTION_MATCH_BOOST", 1.8)) if exact_match else 0.0
        coverage_boost = float(globals().get("OPTION_TERM_COVERAGE_WEIGHT", 0.35)) * term_coverage
        combined_score = ((0.45 * lexical_score) + (0.55 * semantic_score) + exact_boost + coverage_boost + support_sentence_score) * topic_relevance_multiplier
        scored.append(
            {
                "answer_id": option_id(option),
                "answer_text": option_text(option),
                "answer_index": index,
                "letter": LETTERS[index],
                "lexical_score": lexical_score,
                "semantic_score": semantic_score,
                "exact_match": exact_match,
                "term_coverage": term_coverage,
                "support_sentence_score": support_sentence_score,
                "support_sentence": support_sentence,
                "evidence_core_overlap": evidence_core_overlap,
                "combined_score": combined_score,
            }
        )

    scored.sort(key=lambda item: item["combined_score"], reverse=True)
    return scored




### 5.7 Full RAG + Agentic Tool Router Pipeline

In [71]:
def is_numeric_question(question) -> bool:
    q = question.text.lower()
    option_texts = " ".join(option_text(opt) for opt in question.options)

    numeric_words = [
        "population", "year", "date", "century", "how many",
        "number", "estimated", "amount", "percentage", "million",
        "billion", "km", "meters", "age"
    ]

    has_digit_option = bool(re.search(r"\d", option_texts))
    has_numeric_word = any(word in q for word in numeric_words)

    return has_digit_option or has_numeric_word


def answer_one_question_with_pipeline(question, game=None) -> dict:
    start = time.monotonic()
    seconds_left_start = seconds_available(game) if game is not None else None
    direct_option_match = None
    direct_model_seconds = 0.0

    if seconds_left_start is not None and seconds_left_start < MIN_SECONDS_FOR_ANY_MODEL:
        selected = fallback_option(question)
        return {
            "question": question.text,
            "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
            "documents": [],
            "rag_answer": "Skipped pipeline because too little time remained.",
            "rag_method": "hard_time_guard",
            "evidence_chunks": [],
            "option_match": {
                "answer_id": option_id(selected),
                "answer_text": option_text(selected),
                "answer_index": 0,
                "letter": "A",
                "model_output": "hard_time_guard",
                "selection_source": "hard_time_guard",
                "option_scores": [],
            },
            "elapsed_seconds": 0.0,
            "timings": {},
            "seconds_left_start": seconds_left_start,
            "seconds_left_end": seconds_left_start,
        }

    if USE_DIRECT_MODEL_FIRST and (
        seconds_left_start is None
        or seconds_left_start >= QUESTION_TIME_BUFFER + MIN_SECONDS_FOR_DIRECT_MODEL
    ):
        direct_start = time.monotonic()
        try:
            if USE_DIRECT_LOGIT_SCORING:
                direct_option_match = choose_option_direct_shuffled_logits(
                    question,
                    question.options,
                    model_name=MODEL_ID,
                )
                if direct_option_match.get("direct_logit_margin", 0.0) < DIRECT_LOGIT_CONFIDENCE_MARGIN:
                    print(
                        f"Low direct logit margin "
                        f"({direct_option_match.get('direct_logit_margin', 0.0):.3f}); trying prompt voting."
                    )
                    voted_match = choose_option_direct_voted(
                        question,
                        question.options,
                        model_name=MODEL_ID,
                    )
                    voted_match["logit_match"] = direct_option_match
                    direct_option_match = voted_match
            else:
                direct_option_match = choose_option_direct_voted(
                    question,
                    question.options,
                    model_name=MODEL_ID,
                )

            direct_end = time.monotonic()
            direct_model_seconds = direct_end - direct_start

        except Exception as exc:
            print(f"Direct model answer failed; falling back to Wikipedia: {exc}")


    if direct_option_match is not None:
        current_seconds_left = seconds_available(game) if game is not None else seconds_left_start
        direct_margin = direct_logit_margin(direct_option_match)
        use_tool_router = globals().get("USE_AGENTIC_TOOL_ROUTER", True)
        skip_margin = float(globals().get("DIRECT_TOOL_SKIP_MARGIN", DIRECT_LOGIT_CONFIDENCE_MARGIN))
        verify_direct = globals().get("VERIFY_DIRECT_WITH_WIKIPEDIA", True)
        has_tool_time = current_seconds_left is None or current_seconds_left >= QUESTION_TIME_BUFFER + MIN_SECONDS_TO_VERIFY_DIRECT
        high_confidence_direct = use_tool_router and direct_margin >= skip_margin
        should_skip_tools = high_confidence_direct or not verify_direct or not has_tool_time

        if should_skip_tools:
            if high_confidence_direct:
                router_reason = f"high_direct_margin:{direct_margin:.3f}>={skip_margin:.3f}"
            elif not verify_direct:
                router_reason = "verification_disabled"
            else:
                router_reason = "not_enough_time_for_tool_call"
            direct_option_match["selection_source"] = "agentic_router_direct_answer"
            direct_option_match["tool_router_reason"] = router_reason
            direct_option_match["direct_logit_margin"] = direct_margin
            after_match = time.monotonic()
            return {
                "question": question.text,
                "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
                "documents": [],
                "rag_answer": f"Tool router skipped live Wikipedia ({router_reason}); used direct {MODEL_ID} answer.",
                "rag_method": "agentic_router_direct_answer",
                "evidence_chunks": [],
                "option_match": direct_option_match,
                "elapsed_seconds": after_match - start,
                "timings": {
                    "direct_model_seconds": direct_model_seconds,
                    "wikipedia_seconds": 0.0,
                    "rag_generation_seconds": 0.0,
                    "option_matching_seconds": after_match - start - direct_model_seconds,
                    "pre_submit_pipeline_seconds": after_match - start,
                },
                "seconds_left_start": seconds_left_start,
                "seconds_left_end": seconds_available(game) if game is not None else None,
            }

        print(f"Tool router: direct answer was low confidence (margin={direct_margin:.3f}); calling Wikipedia tool path.")

    current_seconds_left = seconds_available(game) if game is not None else seconds_left_start
    if current_seconds_left is not None:
        final_model_reserve = MIN_SECONDS_FOR_FINAL_MODEL
        retrieval_budget = max(0.0, current_seconds_left - QUESTION_TIME_BUFFER - final_model_reserve)
        retrieval_budget = min(WIKIPEDIA_TIME_BUDGET, retrieval_budget)
    else:
        retrieval_budget = WIKIPEDIA_TIME_BUDGET

    retrieval_deadline = time.monotonic() + max(0.0, retrieval_budget)

    if USE_LIVE_WIKIPEDIA:
        docs = get_wikipedia_documents_for_question(
            question,
            top_n=TOP_N_DOCS,
            per_query_limit=PER_QUERY_LIMIT,
            timeout=WIKIPEDIA_TIMEOUT,
            max_search_queries=MAX_SEARCH_QUERIES,
            options=question.options,
            deadline_monotonic=retrieval_deadline,
        )
    else:
        docs = []

    after_wikipedia = time.monotonic()
    seconds_left_after_wiki = seconds_available(game) if game is not None else None

    if seconds_left_after_wiki is not None and seconds_left_after_wiki <= QUESTION_TIME_BUFFER:
        if direct_option_match is not None:
            option_match = direct_option_match
            option_match["selection_source"] = "direct_model_after_wikipedia_time_guard"
        else:
            option_match = score_fallback_option_match(
                question,
                [],
                "submit_time_guard_after_wikipedia",
            )
        after_match = time.monotonic()
        return {
            "question": question.text,
            "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
            "documents": [
                {
                    "title": doc.get("title"),
                    "url": doc.get("url"),
                    "candidate_score": doc.get("candidate_score"),
                    "matched_query": doc.get("matched_query"),
                }
                for doc in docs
            ],
            "rag_answer": "Skipped chunk retrieval because submit buffer was reached.",
            "rag_method": "submit_time_guard_after_wikipedia",
            "evidence_chunks": [],
            "option_match": option_match,
            "elapsed_seconds": after_match - start,
            "timings": {
                "wikipedia_seconds": after_wikipedia - start,
                "pre_submit_pipeline_seconds": after_match - start,
            },
            "seconds_left_start": seconds_left_start,
            "seconds_left_end": seconds_available(game) if game is not None else None,
        }

    if not docs:
        if direct_option_match is not None:
            option_match = direct_option_match
            option_match["selection_source"] = "direct_model_no_wikipedia_docs"
        else:
            option_match = choose_option_direct_voted(
                question,
                question.options,
                model_name=MODEL_ID,
            )
            option_match["selection_source"] = "direct_model_no_wikipedia_docs_retry"

        after_match = time.monotonic()
        if direct_option_match is not None:
            evidence_scores = option_match.get("option_scores", [])
            evidence_is_strong = False

            if evidence_scores and len(evidence_scores) >= 2:
                top = float(evidence_scores[0].get("combined_score", 0.0))
                second = float(evidence_scores[1].get("combined_score", 0.0))
                evidence_is_strong = (
                top >= MIN_EVIDENCE_SCORE_TO_TRUST
                and (top - second) >= EVIDENCE_CONFIDENCE_MARGIN
                )

            if not evidence_is_strong:
                option_match = direct_option_match
                option_match["selection_source"] = "direct_model_preferred_over_weak_rag"
        return {
            "question": question.text,
            "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
            "documents": [],
            "rag_answer": f"No Wikipedia documents; used direct {MODEL_ID} answer.",
            "rag_method": "no_docs_direct_model",
            "evidence_chunks": [],
            "option_match": option_match,
            "elapsed_seconds": after_match - start,
            "timings": {
                "direct_model_seconds": direct_model_seconds,
                "wikipedia_seconds": after_wikipedia - start - direct_model_seconds,
                "pre_submit_pipeline_seconds": after_match - start,
            },
            "seconds_left_start": seconds_left_start,
            "seconds_left_end": seconds_available(game) if game is not None else None,
     }

    seconds_left_before_rag = seconds_available(game) if game is not None else None
    allow_rag_generator = GENERATE_RAG_DRAFT and (
        seconds_left_before_rag is None
        or seconds_left_before_rag >= QUESTION_TIME_BUFFER + MIN_SECONDS_FOR_FINAL_MODEL
    )

    rag_result = answer_question_with_rag(
        question,
        docs,
        top_k_chunks=TOP_K_CHUNKS,
        use_local_generator=allow_rag_generator,
        generator_model=MODEL_ID,
    )

    after_rag = time.monotonic()
    seconds_left_after_rag = seconds_available(game) if game is not None else None

    evidence_hits_for_scoring = [{"title": "RAG answer", "text": rag_result.get("answer", "")}] + rag_result["evidence_chunks"]

    evidence_match = score_fallback_option_match(
        question,
        evidence_hits_for_scoring,
        "wikipedia_evidence_score",
    )

    evidence_scores = evidence_match.get("option_scores", [])
    evidence_is_strong = False

    if evidence_scores and len(evidence_scores) >= 2:
        top = float(evidence_scores[0].get("combined_score", 0.0))
        second = float(evidence_scores[1].get("combined_score", 0.0))
        evidence_is_strong = (
            top >= MIN_EVIDENCE_SCORE_TO_TRUST
            and (top - second) >= EVIDENCE_CONFIDENCE_MARGIN
        )


    if direct_option_match is not None and is_numeric_question(question):
        option_match = direct_option_match
        option_match["selection_source"] = "direct_model_numeric_question"

    elif direct_option_match is not None:
        option_match = reconcile_direct_and_evidence(direct_option_match, evidence_match)

    elif evidence_is_strong:
        option_match = evidence_match
        option_match["selection_source"] = "strong_wikipedia_evidence"

    else:
        option_match = evidence_match
        option_match["selection_source"] = "fallback_wikipedia_evidence"

    after_match = time.monotonic()
    elapsed = after_match - start

    return {
        "question": question.text,
        "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
        "documents": [
            {
                "title": doc.get("title"),
                "url": doc.get("url"),
                "candidate_score": doc.get("candidate_score"),
                "matched_query": doc.get("matched_query"),
            }
            for doc in docs
        ],
        "rag_answer": rag_result["answer"],
        "rag_method": rag_result["method"],
        "evidence_chunks": [
            {
                "title": hit.get("title"),
                "retrieval_score": hit.get("retrieval_score"),
                "bm25_score": hit.get("bm25_score"),
                "semantic_score": hit.get("semantic_score"),
                "cross_encoder_score": hit.get("cross_encoder_score"),
                "pre_rerank_retrieval_score": hit.get("pre_rerank_retrieval_score"),
                "retrieval_method": hit.get("retrieval_method"),
                "text": hit.get("text", "")[:900],
            }
            for hit in rag_result["evidence_chunks"][:5]
        ],
        "option_match": option_match,
        "elapsed_seconds": elapsed,
        "timings": {
            "direct_model_seconds": direct_model_seconds,
            "wikipedia_seconds": after_wikipedia - start - direct_model_seconds,
            "rag_generation_seconds": after_rag - after_wikipedia,
            "option_matching_seconds": after_match - after_rag,
            "pre_submit_pipeline_seconds": elapsed,
        },
        "seconds_left_start": seconds_left_start,
        "seconds_left_end": seconds_available(game) if game is not None else None,
    }

### 5.8 Run History Game

In [72]:
import json
import os
import time
import types
from datetime import datetime, timezone
from pathlib import Path
import torch

# =========================
# ACTUAL GAME: free-Colab direct model + optional RAG verification
# =========================
# Run setup/function cells first: imports/login/client, cell 8, cell 10, cell 13.
# This cell preloads the answer model before starting the timed game, then starts the game.


 # ---- Hyperparameters ----
RUN_ACTUAL_GAME = True
COMPETITION_ID = comp_id
MAX_QUESTIONS = None          # Use 1 or 2 for a small test; None = play until game over.
SUBMIT_ANSWERS = True         # True = send answers to API. False = dry run, no submission.

MAX_SEARCH_QUERIES = 2        # Free Colab/timed game: keep live Wikipedia small to avoid 429s.
PER_QUERY_LIMIT = 2           # Enough fallback candidates without burning the whole timer.
TOP_N_DOCS = 3                # Smaller evidence set for a 30-second question window.



WIKIPEDIA_TIMEOUT = 2.0       # Seconds per Wikipedia API request.
WIKIPEDIA_DELAY_SECONDS = 1.0 # Respect Wikipedia's rate limit.
WIKIPEDIA_BACKOFF_SECONDS = 0.5
WIKIPEDIA_RETRIES = 0
MIN_WIKIPEDIA_CANDIDATE_SCORE = 0.25 # Fetch more plausible pages; evidence scorer filters them later.
TOP_K_CHUNKS = 12          # Keep the final evidence set small for the 30-second timer.
USE_LIVE_WIKIPEDIA = True    # Call Wikipedia during the timed game.
USE_PYTERRIER_BM25 = True      # Timed game: TF-IDF fallback is much faster for small per-question chunks.
USE_SENTENCE_EMBEDDING_RERANKER = True # Free Colab: avoid CPU embedding latency during timed play.
USE_OPTION_EMBEDDING_SCORER = True    # Timed game: use fast lexical option scores unless you have time.
USE_CROSS_ENCODER_RERANKER = True # Timed game: cross-encoder was slower/worse in practice.
USE_DIRECT_MODEL_FIRST = True   # Cheap first opinion; retrieved evidence can verify or correct it.
USE_DIRECT_LOGIT_SCORING = True    # Use stable direct next-token logit scoring for no-evidence fallback.
DIRECT_LOGIT_CONFIDENCE_MARGIN = 1.0 # Low margin triggers vote/Wikipedia fallback.
USE_AGENTIC_TOOL_ROUTER = True # Direct answer first; call Wikipedia only when confidence/time says it is worth it.
DIRECT_TOOL_SKIP_MARGIN = 1.25 # If direct logit margin reaches this, skip tool calls and submit.
VERIFY_DIRECT_WITH_WIKIPEDIA = True # Router may use Wikipedia for low-confidence direct answers.
DIRECT_MODEL_VOTES = 3        # Used only by optional generated-vote helpers, not the default fallback.
GENERATE_RAG_DRAFT = False  # Timed game: avoid slow explanatory generation before choosing an option.
SENTENCE_EMBEDDING_DEVICE = "cpu" # Keep GPU memory for the answer model; use "cuda" only if you have room.
CROSS_ENCODER_DEVICE = "cpu"
EMBEDDING_RERANK_TOP_N = 20    # Rerank this many top BM25/fallback chunks semantically.
EMBEDDING_RERANK_WEIGHT = 0.35 # Higher means semantic similarity influences ranking more.
CROSS_ENCODER_RERANK_TOP_N = 4
CROSS_ENCODER_RERANK_WEIGHT = 0.55
QUESTION_TIME_BUFFER = 6.0    # Submit before this many seconds remain.
MIN_SECONDS_TO_ATTEMPT = 4.0  # If less time remains, submit fallback option 0.
WIKIPEDIA_TIME_BUDGET = 5.0   # Hard cap for Wikipedia search + page fetch.
MIN_SECONDS_FOR_FINAL_MODEL = 8.0 # Require this much extra time, after the submit buffer, before final answer model.
MIN_SECONDS_FOR_ANY_MODEL = 3.0   # Below this, submit fallback immediately.
MIN_SECONDS_FOR_DIRECT_MODEL = 4.0
MIN_SECONDS_TO_VERIFY_DIRECT = 10.0
EVIDENCE_OVERRIDE_MARGIN = 0.20
EVIDENCE_CONFIDENCE_MARGIN = 0.12
MIN_EVIDENCE_SCORE_TO_TRUST = 0.45
MIN_DIRECT_MARGIN_FOR_WEAK_EVIDENCE = 2.50
EXACT_OPTION_MATCH_BOOST = 1.8
OPTION_TERM_COVERAGE_WEIGHT = 0.35

DELAY_SUBMIT_FOR_WIKI_COOLDOWN = False # Never wait on purpose inside a 30-second question.
TARGET_SUBMIT_ELAPSED_SECONDS = 22.0
MIN_SECONDS_LEFT_AT_SUBMIT = 5.0      # Safety margin for network/server latency.
MAX_SUBMIT_WAIT_SECONDS = 0

PRELOAD_ANSWER_MODEL = True
SAVE_RUN_LOG = True
RUN_LOG_DIR = "/content/gdrive/MyDrive/NLP_assignment/test3_rag_game_runs"
VERBOSE = True

# ---- End hyperparameters ----






def preload_answer_model_for_game():
    globals()["WIKIPEDIA_REQUEST_DELAY_SECONDS"] = WIKIPEDIA_DELAY_SECONDS
    globals()["WIKIPEDIA_429_BACKOFF_SECONDS"] = WIKIPEDIA_BACKOFF_SECONDS
    globals()["WIKIPEDIA_MAX_RETRIES"] = WIKIPEDIA_RETRIES
    globals()["MAX_WIKIPEDIA_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
    globals()["USE_PYTERRIER_BM25"] = USE_PYTERRIER_BM25
    print(f"Preloading {MODEL_ID} before starting timed game...")
    start = time.time()
    try:
        tokenizer, model = load_answer_model(MODEL_ID)
    except Exception as exc:
        raise RuntimeError(
            f"Could not load answer model {MODEL_ID}. Restart the Colab runtime, run only the setup cells, "
            "or choose a smaller model if GPU memory is tight. The timed game was not started."
        ) from exc
    active_model_id = next((name for name, cached in _ANSWER_MODEL_CACHE.items() if cached == (tokenizer, model)), MODEL_ID)
    model_device = next(model.parameters()).device
    model_dtype = next(model.parameters()).dtype
    print(f"Loaded model: {active_model_id} | device={model_device} | dtype={model_dtype}")


    warmup_messages = [
        {"role": "system", "content": "Answer briefly."},
        {"role": "user", "content": "Say ready."},
    ]
    warmup_prompt = tokenizer.apply_chat_template(warmup_messages, tokenize=False, add_generation_prompt=True)
    device = next(model.parameters()).device
    inputs = tokenizer(warmup_prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        _ = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    print(f"Direct model ready in {time.time() - start:.1f}s. Starting game only after this point.")
    if globals().get("USE_SENTENCE_EMBEDDING_RERANKER", True):
        print(f"Preloading sentence embedding model {SENTENCE_EMBEDDING_MODEL_ID}...")
        load_sentence_embedding_model()
        print("Sentence embedding reranker ready.")
    if globals().get("USE_CROSS_ENCODER_RERANKER", False):
        print(f"Preloading cross-encoder reranker {CROSS_ENCODER_MODEL_ID}...")
        load_cross_encoder_model()
        print("Cross-encoder reranker ready.")


def seconds_available(game) -> float:
    remaining = game.time_remaining
    if remaining is None:
        return 30.0
    return max(0.0, float(remaining))


def fallback_option(question):
    return question.options[0]


def score_fallback_option_match(question, hits: list[dict], reason: str) -> dict:
    """Choose the highest evidence-score option without another answer-model call."""
    option_scores = score_options_against_evidence(question, question.options, hits)
    best_score = option_scores[0] if option_scores else {"answer_index": 0}
    selected_option = question.options[best_score["answer_index"]]
    return {
        "answer_id": option_id(selected_option),
        "answer_text": option_text(selected_option),
        "answer_index": best_score["answer_index"],
        "letter": LETTERS[best_score["answer_index"]],
        "model_output": reason,
        "selection_source": "timed_option_evidence_score_fallback",
        "selected_option_score": best_score,
        "option_scores": option_scores,
    }


def reconcile_direct_and_evidence(direct_match: dict, evidence_match: dict) -> dict:
    """Keep direct model unless evidence strongly supports another option."""
    if not direct_match:
        return evidence_match
    if not evidence_match or not evidence_match.get("option_scores"):
        direct = dict(direct_match)
        direct["selection_source"] = "direct_model_no_evidence"
        return direct

    scores = evidence_match["option_scores"]
    top = scores[0]
    direct_score = next((item for item in scores if item["answer_index"] == direct_match["answer_index"]), None)
    direct_value = float(direct_score.get("combined_score", 0.0)) if direct_score else 0.0
    top_value = float(top.get("combined_score", 0.0))
    second_value = float(scores[1].get("combined_score", 0.0)) if len(scores) > 1 else 0.0
    evidence_margin = top_value - second_value
    margin = top_value - direct_value
    direct_margin = direct_logit_margin(direct_match)
    direct_is_high_confidence = direct_margin >= DIRECT_LOGIT_CONFIDENCE_MARGIN
    direct_is_safe_with_weak_evidence = direct_margin >= globals().get("MIN_DIRECT_MARGIN_FOR_WEAK_EVIDENCE", 1.20)
    top_exact_with_margin = bool(top.get("exact_match")) and evidence_margin >= EVIDENCE_CONFIDENCE_MARGIN
    top_is_trustworthy = top_exact_with_margin or (
        top_value >= MIN_EVIDENCE_SCORE_TO_TRUST and evidence_margin >= EVIDENCE_OVERRIDE_MARGIN
    )
    should_override = top["answer_index"] != direct_match["answer_index"] and top_is_trustworthy and margin >= EVIDENCE_OVERRIDE_MARGIN

    if should_override:
        chosen = dict(evidence_match)
        chosen["selection_source"] = "wikipedia_evidence_overrode_direct_model"
        chosen["direct_model_match"] = direct_match
        chosen["evidence_override_margin"] = margin
        chosen["evidence_score_margin"] = evidence_margin
        return chosen

    chosen = dict(direct_match)
    if top["answer_index"] == direct_match["answer_index"] and top_is_trustworthy:
        chosen["selection_source"] = "direct_model_confirmed_by_wikipedia_evidence"
    elif direct_is_safe_with_weak_evidence:
        chosen["selection_source"] = "direct_model_high_confidence_weak_evidence"
    elif direct_is_high_confidence:
        chosen["selection_source"] = "direct_model_acceptable_margin_weak_evidence"
    else:
        chosen = dict(evidence_match)
        chosen["selection_source"] = "weak_direct_model_deferred_to_evidence_score"
        chosen["direct_model_match"] = direct_match
        chosen["evidence_override_margin"] = margin
        chosen["evidence_score_margin"] = evidence_margin
        return chosen
    chosen["option_scores"] = scores
    chosen["selected_option_score"] = direct_score
    chosen["evidence_top_option"] = top
    chosen["evidence_override_margin"] = margin
    chosen["evidence_score_margin"] = evidence_margin
    chosen["direct_logit_margin"] = direct_margin
    return chosen


def direct_logit_margin(direct_match: dict) -> float:
    """Return the direct logit margin, including when a generated vote wrapped it."""
    if not direct_match:
        return 0.0
    if "direct_logit_margin" in direct_match:
        return float(direct_match.get("direct_logit_margin", 0.0))
    logit_match = direct_match.get("logit_match") or {}
    return float(logit_match.get("direct_logit_margin", 0.0))


def wait_before_submit_for_cooldown(game) -> float:
    """Wait after computing the answer so Wikipedia gets cooldown time before next question."""
    if not DELAY_SUBMIT_FOR_WIKI_COOLDOWN:
        return 0.0

    current_remaining = seconds_available(game)
    target_remaining = max(MIN_SECONDS_LEFT_AT_SUBMIT, 30.0 - TARGET_SUBMIT_ELAPSED_SECONDS)
    wait_seconds = current_remaining - target_remaining
    wait_seconds = min(MAX_SUBMIT_WAIT_SECONDS, max(0.0, wait_seconds))

    if wait_seconds > 0:
        print(f"Answer ready. Waiting {wait_seconds:.1f}s before submit to give Wikipedia API cooldown time.")
        time.sleep(wait_seconds)

    return wait_seconds



def play_actual_rag_game(mode="text"):
    globals()["WIKIPEDIA_REQUEST_DELAY_SECONDS"] = WIKIPEDIA_DELAY_SECONDS
    globals()["WIKIPEDIA_429_BACKOFF_SECONDS"] = WIKIPEDIA_BACKOFF_SECONDS
    globals()["WIKIPEDIA_MAX_RETRIES"] = WIKIPEDIA_RETRIES
    globals()["MAX_WIKIPEDIA_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
    globals()["USE_PYTERRIER_BM25"] = USE_PYTERRIER_BM25

    if mode not in {"text", "speech"}:
        raise ValueError('mode must be either "text" or "speech"')

    if PRELOAD_ANSWER_MODEL:
        preload_answer_model_for_game()
    if mode == "speech":
        load_whisper_model_for_speech()

    # Keep the Wikipedia cooldown state across setup/game questions so live requests do not trip 429s.

    if not RUN_ACTUAL_GAME:
        print("RUN_ACTUAL_GAME is False. Set it to True to start a real timed game.")
        return None, None

    game = client.game.start(competition_id=COMPETITION_ID, mode=mode)
    run_log = {
        "session_id": game.session_id,
        "competition_id": COMPETITION_ID,
        "started_at": datetime.now(timezone.utc).isoformat(),
        "hyperparameters": {
            "top_n_docs": TOP_N_DOCS,
            "per_query_limit": PER_QUERY_LIMIT,
            "max_search_queries": MAX_SEARCH_QUERIES,
            "wikipedia_timeout": WIKIPEDIA_TIMEOUT,
            "wikipedia_delay_seconds": WIKIPEDIA_DELAY_SECONDS,
            "wikipedia_backoff_seconds": WIKIPEDIA_BACKOFF_SECONDS,
            "wikipedia_retries": WIKIPEDIA_RETRIES,
            "min_wikipedia_candidate_score": MIN_WIKIPEDIA_CANDIDATE_SCORE,
            "use_live_wikipedia": USE_LIVE_WIKIPEDIA,
            "use_pyterrier_bm25": USE_PYTERRIER_BM25,
            "top_k_chunks": TOP_K_CHUNKS,
            "use_option_embedding_scorer": USE_OPTION_EMBEDDING_SCORER,
            "use_cross_encoder_reranker": USE_CROSS_ENCODER_RERANKER,
            "cross_encoder_model": CROSS_ENCODER_MODEL_ID,
            "use_direct_model_first": USE_DIRECT_MODEL_FIRST,
            "use_direct_logit_scoring": USE_DIRECT_LOGIT_SCORING,
            "direct_logit_confidence_margin": DIRECT_LOGIT_CONFIDENCE_MARGIN,
            "use_agentic_tool_router": USE_AGENTIC_TOOL_ROUTER,
            "direct_tool_skip_margin": DIRECT_TOOL_SKIP_MARGIN,
            "verify_direct_with_wikipedia": VERIFY_DIRECT_WITH_WIKIPEDIA,
            "direct_model_votes": DIRECT_MODEL_VOTES,
            "min_seconds_for_direct_model": MIN_SECONDS_FOR_DIRECT_MODEL,
            "min_seconds_to_verify_direct": MIN_SECONDS_TO_VERIFY_DIRECT,
            "evidence_override_margin": EVIDENCE_OVERRIDE_MARGIN,
            "evidence_confidence_margin": EVIDENCE_CONFIDENCE_MARGIN,
            "min_evidence_score_to_trust": MIN_EVIDENCE_SCORE_TO_TRUST,
            "min_direct_margin_for_weak_evidence": MIN_DIRECT_MARGIN_FOR_WEAK_EVIDENCE,
            "exact_option_match_boost": EXACT_OPTION_MATCH_BOOST,
            "option_term_coverage_weight": OPTION_TERM_COVERAGE_WEIGHT,
            "generate_rag_draft": GENERATE_RAG_DRAFT,
            "question_time_buffer": QUESTION_TIME_BUFFER,
            "min_seconds_to_attempt": MIN_SECONDS_TO_ATTEMPT,
            "delay_submit_for_wiki_cooldown": DELAY_SUBMIT_FOR_WIKI_COOLDOWN,
            "target_submit_elapsed_seconds": TARGET_SUBMIT_ELAPSED_SECONDS,
            "min_seconds_left_at_submit": MIN_SECONDS_LEFT_AT_SUBMIT,
            "max_submit_wait_seconds": MAX_SUBMIT_WAIT_SECONDS,
            "answer_model": MODEL_ID,
            "submit_answers": SUBMIT_ANSWERS,
            "game_mode": mode,
            "whisper_model_size": globals().get("ACTIVE_WHISPER_MODEL_SIZE") or globals().get("WHISPER_MODEL_SIZE"),
            "whisper_device": globals().get("ACTIVE_WHISPER_DEVICE") or globals().get("WHISPER_DEVICE"),
        },
        "questions": [],
    }

    print(f"Started game session {game.session_id}. Competition {COMPETITION_ID}. Mode {game.mode}.")
    question_count = 0
    correct_count = 0

    while game.in_progress:
        speech_transcription = None
        if game.mode == "speech":
            question, speech_transcription = transcribe_speech_question(game)
        else:
            question = game.current_question
        if question is None:
            print("No active question returned by server.")
            break

        # Do not reset the Wikipedia cooldown here; the server rate limit continues across questions.

        question_count += 1
        current_level = game.current_level
        time_left = seconds_available(game)

        print("\n" + "=" * 80)
        print(f"Question {question_count} | Level {current_level} | {time_left:.1f}s left")
        if speech_transcription:
            seconds_left = speech_transcription.get("seconds_left_after_audio")
            print(
                "Speech transcription:",
                f"{speech_transcription.get('transcription_seconds', 0.0):.1f}s",
                f"| {seconds_left:.1f}s left after audio" if seconds_left is not None else "| time left n/a",
                f"| Whisper {speech_transcription.get('whisper_model_size')} on {speech_transcription.get('whisper_device')}",
            )
        print(question.text)
        for index, opt in enumerate(question.options):
            print(f"  {LETTERS[index]}. [{option_id(opt)}] {option_text(opt)}")
        print("=" * 80)

        if time_left < MIN_SECONDS_TO_ATTEMPT:
            selected = fallback_option(question)
            prediction = {
                "question": question.text,
                "options": [{"id": option_id(opt), "text": option_text(opt)} for opt in question.options],
                "documents": [],
                "rag_answer": "Skipped RAG because not enough time remained.",
                "rag_method": "fallback_time_guard",
                "evidence_chunks": [],
                "option_match": {
                    "answer_id": option_id(selected),
                    "answer_text": option_text(selected),
                    "answer_index": 0,
                    "letter": "A",
                    "model_output": "fallback_time_guard",
                },
                "elapsed_seconds": 0.0,
                "seconds_left_start": time_left,
                "seconds_left_end": time_left,
            }
        else:
            prediction = answer_one_question_with_pipeline(question, game=game)

        selected_id = prediction["option_match"]["answer_id"]
        selected_text = prediction["option_match"]["answer_text"]
        selected_letter = prediction["option_match"]["letter"]

        if VERBOSE:
            print("\nRAG answer:")
            print(prediction["rag_answer"])
            print("\nClosest option:", f"{selected_letter}. [{selected_id}] {selected_text}")
            print("Matcher output:", prediction["option_match"].get("model_output"))
            timings = prediction.get("timings", {})
            if timings:
                print(
                    "Timing:",
                    f"direct={timings.get('direct_model_seconds', 0):.2f}s",
                    f"wiki={timings.get('wikipedia_seconds', 0):.2f}s",
                    f"rag={timings.get('rag_generation_seconds', 0):.2f}s",
                    f"match={timings.get('option_matching_seconds', 0):.2f}s",
                    f"total={timings.get('pre_submit_pipeline_seconds', prediction['elapsed_seconds']):.2f}s",
                )
            print(f"Elapsed: {prediction['elapsed_seconds']:.2f}s | Time left now: {seconds_available(game):.1f}s")

        result_payload = None
        if SUBMIT_ANSWERS:
            submission_wait_seconds = wait_before_submit_for_cooldown(game)
            prediction["submission_wait_seconds"] = submission_wait_seconds
            if submission_wait_seconds:
                print(f"Time left after cooldown wait: {seconds_available(game):.1f}s")

            if seconds_available(game) <= QUESTION_TIME_BUFFER:
                print("Warning: low time before submit; submitting selected option immediately.")
            result = game.answer(selected_id)
            result_payload = {
                "correct": result.correct,
                "timed_out": result.timed_out,
                "game_over": result.game_over,
                "earned_amount": result.earned_amount,
            }
            correct_count += int(bool(result.correct))

            if result.correct:
                print(f"Correct. Earned: {result.earned_amount}")
            elif result.timed_out:
                print(f"Timed out. Earned: {result.earned_amount}")
            else:
                print(f"Wrong. Earned: {result.earned_amount}")
        else:
            print("Dry run: answer not submitted.")

        run_log["questions"].append(
            {
                "number": question_count,
                "level": current_level,
                "mode": game.mode,
                "speech_transcription": speech_transcription,
                "prediction": prediction,
                "result": result_payload,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
        )

        if result_payload and result_payload.get("game_over"):
            break
        if MAX_QUESTIONS is not None and question_count >= MAX_QUESTIONS:
            print("MAX_QUESTIONS reached; stopping.")
            break
        if not SUBMIT_ANSWERS:
            break

    run_log["finished_at"] = datetime.now(timezone.utc).isoformat()
    run_log["questions_answered"] = question_count
    run_log["correct_count"] = correct_count
    run_log["final_earned_amount"] = game.earned_amount

    if SAVE_RUN_LOG:
        log_dir = Path(RUN_LOG_DIR)
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f"test3_rag_game_{game.session_id}.json"
        with open(log_path, "w", encoding="utf-8") as handle:
            json.dump(run_log, handle, indent=2, ensure_ascii=False)
        print("Run log saved to:", log_path)

    print("\nGame summary")
    print("Questions answered:", question_count)
    print("Correct answers:", correct_count)
    print("Final earnings:", game.earned_amount)
    return game, run_log



In [75]:
final_game, final_run_log = play_actual_rag_game(mode="text")

Preloading Qwen/Qwen2.5-7B-Instruct before starting timed game...
Loaded model: Qwen/Qwen2.5-7B-Instruct | device=cuda:0 | dtype=torch.bfloat16
Direct model ready in 0.5s. Starting game only after this point.
Preloading sentence embedding model sentence-transformers/all-MiniLM-L6-v2...
Sentence embedding reranker ready.
Preloading cross-encoder reranker cross-encoder/ms-marco-MiniLM-L-6-v2...
Cross-encoder reranker ready.
Started game session 341129. Competition 1. Mode text.

Question 1 | Level 1 | 29.9s left
What strategy did Alexander use to secure the loyalty of the Iranian upper classes after his conquests?
  A. [0] He married Persian princesses and adopted Persian customs
  B. [1] He expelled all Iranians from the Persian Empire
  C. [2] He ignored the Iranian nobility and focused only on Greek settlers
  D. [3] He executed all Iranian nobles who opposed him

RAG answer:
Tool router skipped live Wikipedia (high_direct_margin:22.333>=1.250); used direct Qwen/Qwen2.5-7B-Instruct an

In [78]:
final_game, final_run_log = play_actual_rag_game(mode="speech")

Preloading Qwen/Qwen2.5-7B-Instruct before starting timed game...
Loaded model: Qwen/Qwen2.5-7B-Instruct | device=cuda:0 | dtype=torch.bfloat16


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Direct model ready in 0.5s. Starting game only after this point.
Preloading sentence embedding model sentence-transformers/all-MiniLM-L6-v2...
Sentence embedding reranker ready.
Preloading cross-encoder reranker cross-encoder/ms-marco-MiniLM-L-6-v2...
Cross-encoder reranker ready.
Started game session 341153. Competition 1. Mode speech.
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'Which term describes the unusual octa-style design of the Parthenon?' -> 'Which term describes the unusual octa-style design of the Parthenon'
Question transcript: Which term describes the unusual octa-style design of the Parthenon
Cleaned transcript noise: 'Option A, a single row of columns on the front and back.' -> 'a single row of columns on the front and back'
Option A transcript: a single row of columns on the front and back
Cleaned transcript noise: 'Option

---
## 6. News Pipeline
_Competition ID: 5 (News & Current Events)_

Techniques: Serper Google News API -> article scraping, Bing RSS + FAISS semantic
fallback, LLM-generated search keywords, date-window filtering.

### 6.1 News Retrieval Helpers (Serper, Bing RSS, FAISS)

In [79]:
import concurrent.futures
import json
import builtins
import os
import re
import textwrap
import urllib.parse
import warnings
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from latex2sympy2 import latex2sympy

import faiss
import requests
import trafilatura
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer


# Keep notebook output clean while the helper libraries load and run.
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter(action="ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore")


SERPER_API_KEY = os.getenv("SERPER_API_KEY", "efc501e31e6f819db6f5c2546f86f9c239aebade")
USE_SEARCH_DATE_FILTER = True
SEARCH_DATE_INTERVAL_DAYS = 2


REQUEST_HEADERS = {
    "User-Agent": "Chrome/120.0",
    "Accept": "text/html",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://www.google.com/",
    "DNT": "1",
    "Upgrade-Insecure-Requests": "1",
}


def clean_bing_redirect(url):
    """Return the real page URL when Bing wraps it in a redirect link."""
    if "bing.com" not in url or "url=" not in url.lower():
        return url

    parsed_url = urllib.parse.urlparse(url)
    query_args = urllib.parse.parse_qs(parsed_url.query)
    return query_args.get("url", [url])[0]


def read_article_body(url):
    """Try to pull readable article text from a URL."""
    resolved_url = clean_bing_redirect(url)

    # First try trafilatura because it usually strips menus, ads, and sidebars cleanly.
    try:
        html = trafilatura.fetch_url(resolved_url)
        if html:
            article_text = trafilatura.extract(html)
            if article_text and len(article_text) > 200:
                return article_text[:8000]
    except Exception:
        pass

    # If extraction fails, fall back to a simple BeautifulSoup paragraph scrape.
    try:
        response = requests.get(resolved_url, headers=REQUEST_HEADERS, timeout=4, allow_redirects=True)
        if response.status_code != 200:
            return ""

        soup = BeautifulSoup(response.text, "html.parser")
        for node in soup(["script", "style", "nav", "header", "footer", "aside"]):
            node.extract()

        useful_lines = []
        for paragraph in soup.find_all(["p", "li"]):
            line = paragraph.get_text(strip=True)
            if len(line) > 30:
                useful_lines.append(line)

        return " ".join(useful_lines)[:8000]
    except Exception:
        return ""


def make_date_filter(question_text):
    """Build a Serper date filter around a YYYY-MM-DD date found in the question."""
    match = re.search(r"\b(202\d)-(\d{2})-(\d{2})\b", question_text)
    if not match:
        return ""

    year, month, day = match.groups()
    try:
        target_day = datetime.strptime(f"{year}-{month}-{day}", "%Y-%m-%d")
        span = max(0, int(SEARCH_DATE_INTERVAL_DAYS))
        start_date = (target_day - timedelta(days=span)).strftime("%m/%d/%Y")
        end_date = (target_day + timedelta(days=span)).strftime("%m/%d/%Y")
        return f"cdr:1,cd_min:{start_date},cd_max:{end_date}"
    except Exception:
        return ""


def describe_date_window(date_window):
    match = re.search(r"cd_min:([^,]+),cd_max:([^,]+)", date_window or "")
    if not match:
        return ""
    return f"{match.group(1)} to {match.group(2)}"


def gather_primary_news(search_text, date_window=""):
    print(f"\nQuery trace: Search phrase: '{search_text}'")
    if date_window:
        print(f"        Date interval: {describe_date_window(date_window)}")

    if not SERPER_API_KEY:
        print("Query trace: SERPER_API_KEY not found - skipping primary lookup.")
        return ""

    request_body = {"q": search_text, "num": 6}
    if date_window:
        request_body["tbs"] = date_window

    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}

    try:
        response = requests.post(
            "https://google.serper.dev/news",
            headers=headers,
            data=json.dumps(request_body),
            timeout=10,
        )
        if response.status_code != 200:
            return ""

        results = response.json().get("news", [])

        # If the date-filtered query is too narrow, retry once without the date window.
        if not results and date_window:
            print("        Query trace: Date interval search empty; retrying broad search.")
            request_body.pop("tbs", None)
            wide_response = requests.post(
                "https://google.serper.dev/news",
                headers=headers,
                data=json.dumps(request_body),
                timeout=10,
            )
            results = wide_response.json().get("news", [])

        # News search can miss fresh pages, so use regular web search as one more fallback.
        if not results:
            print("        Query trace: News channel empty; checking general web results.")
            web_response = requests.post(
                "https://google.serper.dev/search",
                headers=headers,
                data=json.dumps(request_body),
                timeout=10,
            )
            results = web_response.json().get("organic", [])

        top_links = [item.get("link") for item in results[:2] if item.get("link")]
        article_text_by_url = {}

        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
            futures = {executor.submit(read_article_body, link): link for link in top_links}
            for future in concurrent.futures.as_completed(futures):
                link = futures[future]
                try:
                    article_text_by_url[link] = future.result()
                except Exception:
                    article_text_by_url[link] = ""

        context_parts = []
        for item in results[:4]:
            title = item.get("title", "")
            date_label = item.get("date", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")

            block = f"Title: {title}\nDate: {date_label}\nSummary: {snippet}\n"
            if article_text_by_url.get(link):
                block += f"EXTENDED FULL TEXT: {article_text_by_url[link][:2500]}...\n"
            context_parts.append(block)

        return "\n".join(context_parts)
    except Exception as error:
        print(f"Query trace: Primary lookup error: {error}")
        return ""


class SecondaryNewsIndex:
    """Small Bing RSS + FAISS fallback used when Serper does not find enough evidence."""

    def __init__(self):
        print("Secondary index: Loading Bing RSS + vector fallback...")
        if "load_sentence_embedding_model" in globals():
            self.encoder = load_sentence_embedding_model()
        else:
            sentence_model_id = globals().get("SENTENCE_EMBEDDING_MODEL_ID", "sentence-transformers/all-MiniLM-L6-v2")
            cache = globals().setdefault("_SENTENCE_EMBEDDING_CACHE", {})
            if sentence_model_id not in cache:
                cache[sentence_model_id] = SentenceTransformer(sentence_model_id)
            self.encoder = cache[sentence_model_id]
        self.rss_cache = {}

    def make_chunks(self, text, chunk_size=600, overlap=150):
        clean_text = re.sub(r"\s+", " ", text)
        sentences = re.split(r"(?<=[.])\s+", clean_text)

        chunks = []
        current_chunk = ""
        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= chunk_size:
                current_chunk += " " + sentence
            else:
                if current_chunk.strip():
                    chunks.append(current_chunk.strip())
                current_chunk = sentence

        if current_chunk.strip():
            chunks.append(current_chunk.strip())
        return chunks

    def search_backup_index(self, search_terms, semantic_query, top_k=6):
        search_words = search_terms.split()
        search_variants = [search_terms]
        if len(search_words) > 3:
            search_variants.append(" ".join(search_words[:-1]))
        if len(search_words) > 2:
            search_variants.append(" ".join(search_words[:2]))

        candidate_chunks = []
        seen_urls = set()
        headers = {"User-Agent": "Mozilla/5.0"}

        for query in search_variants:
            if not query.strip():
                continue

            if query in self.rss_cache:
                candidate_chunks.extend(self.rss_cache[query])
                break

            chunks_for_query = []
            try:
                encoded_query = urllib.parse.quote(query)
                rss_url = f"https://www.bing.com/news/search?q={encoded_query}&format=rss"
                response = requests.get(rss_url, headers=headers, timeout=10)
                if response.status_code != 200:
                    continue

                root = ET.fromstring(response.text)
                items = root.findall(".//channel/item")

                feed_items = []
                for item in items:
                    link_node = item.find("link")
                    link = link_node.text if link_node is not None else ""
                    if link and link not in seen_urls:
                        seen_urls.add(link)
                        feed_items.append((item, link))
                    if len(feed_items) >= 3:
                        break

                def read_feed_item(feed_item):
                    item, link = feed_item
                    title_node = item.find("title")
                    desc_node = item.find("description")

                    title = title_node.text if title_node is not None else ""
                    description = desc_node.text if desc_node is not None else ""
                    description = re.sub("<[^<]+>", " ", description)

                    article_text = read_article_body(link)
                    combined_text = f"{title}. {description}. {article_text}"
                    return self.make_chunks(combined_text)

                with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
                    for chunks in executor.map(read_feed_item, feed_items):
                        chunks_for_query.extend(chunks or [])

                self.rss_cache[query] = chunks_for_query
                candidate_chunks.extend(chunks_for_query)
                if candidate_chunks:
                    break
            except Exception:
                continue

        if not candidate_chunks:
            return ""

        # Remove near-duplicate chunks before embedding them.
        deduped_chunks = []
        chunk_signatures = set()
        for chunk in candidate_chunks:
            signature = chunk[:100].strip()
            if signature not in chunk_signatures:
                chunk_signatures.add(signature)
                deduped_chunks.append(chunk)

        if not deduped_chunks:
            return ""

        embeddings = self.encoder.encode(deduped_chunks, convert_to_numpy=True)
        index = faiss.IndexFlatL2(embeddings.shape[1])
        index.add(embeddings)

        question_vector = self.encoder.encode([semantic_query], convert_to_numpy=True)
        _, nearest_ids = index.search(question_vector, min(top_k, len(deduped_chunks)))

        selected_chunks = [deduped_chunks[index_id] for index_id in nearest_ids[0][:6]]
        return "\n\n".join(selected_chunks)


secondary_news_index = SecondaryNewsIndex()


def wrap_output_text(text, indent="", subsequent_indent=None):
    subsequent_indent = indent if subsequent_indent is None else subsequent_indent
    return textwrap.fill(
        str(text or ""),
        width=100,
        initial_indent=indent,
        subsequent_indent=subsequent_indent,
        break_long_words=False,
        break_on_hyphens=False,
    )


def display_question(question, question_count, level, seconds_left=None):
    border = "=" * 80
    time_label = "" if seconds_left is None else f" | {seconds_left:.1f}s left"

    print("\n" + border)
    print(f"Question {question_count} | Level {level}{time_label}")
    print(wrap_output_text(question.text))

    for index, option in enumerate(question.options):
        answer_letter = chr(65 + index)
        line = f"{answer_letter}: {option.id}: {option.text}"
        print(wrap_output_text(line, indent="  ", subsequent_indent="     "))

    print(border)


def pull_eval_summary(text):
    match = re.search(r"Eval:\s*(.+?)(?:\n|FINAL ANSWER:|$)", str(text or ""), re.IGNORECASE | re.DOTALL)
    if not match:
        return ""
    return re.sub(r"\s+", " ", match.group(1)).strip()


def make_search_keywords(question_text, options):
    prompt = f"""[INST] You are a specialist in news-search query construction.
Create a focused Google News query of no more than 5 words that can locate the exact article.

Rules:
1. Use only the core event, distinctive proper nouns, and main subjects from the question.
2. Do not include or borrow wording from the answer choices.
3. Return bare noun keywords separated by spaces. No verbs, punctuation, or explanation.

Generate keywords for this question:
{question_text}
Keywords: [/INST]"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=25,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    keywords = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    keywords = re.sub(r"\[/?INST\]|Keywords?:", " ", keywords, flags=re.IGNORECASE)
    keywords = re.sub(r"[^\w\s\-']", " ", keywords)
    return re.sub(r"\s+", " ", keywords).strip()


def read_final_letter(text):
    upper_text = text.upper()
    if "FINAL ANSWER: NONE" in upper_text:
        return "N"

    match = re.search(r"FINAL ANSWER:\s*([ABCD])", text, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    letters = re.findall(r"\b([ABCD])\b", upper_text)
    return letters[-1] if letters else "A"


def ask_news_model(context, question):
    prompt = f"""[INST] You are a careful current-events analyst answering a multiple-choice question from retrieved news context.

Context:
{context}

Question:
{question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}

Instructions:
1. Choose the option best supported by the evidence.
2. If the question asks for what is NOT, EXCEPT, false, missing, or denied, choose the option that the text excludes or fails to support.
3. If several options appear as nested locations or entities, choose the most specific one.
4. Combine details across snippets when that is necessary to identify the correct option.
5. If the context does not support an answer, output 'FINAL ANSWER: NONE'. Do not guess.
6. After 'FINAL ANSWER:' output only A, B, C, D, or NONE. Do not write the option text.

Use exactly this format:
Eval: justify the selected option in at most 15 words
FINAL ANSWER: A, B, C, D, or NONE
[/INST]Eval: """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    reply = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    reply = re.sub(r"\[/?INST\]", " ", reply, flags=re.IGNORECASE)
    reply = re.sub(r"\s+", " ", reply).strip()
    return re.sub(r"\s*FINAL ANSWER:", "\nFINAL ANSWER:", reply, flags=re.IGNORECASE).strip()


def select_answer(question):
    if len(question.options) < 4:
        return question.options[0].id, "A", "fallback"

    search_terms = make_search_keywords(question.text, question.options)
    date_filter = make_date_filter(question.text) if USE_SEARCH_DATE_FILTER else ""

    answer_letter = "N"
    model_reply = ""

    # Primary path: look for direct news evidence first.
    context = gather_primary_news(search_terms, date_filter)
    if context.strip():
        print("\n" + "-" * 40)
        print("Source packet A: Primary Serper evidence:")
        print(context)
        print("-" * 40 + "\n")

        model_reply = ask_news_model(context, question)
        answer_letter = read_final_letter(model_reply)
        print(f"Model read A:\n{model_reply}")

    # Secondary path: if the model says NONE, gather RSS evidence and rank it semantically.
    if answer_letter == "N":
        print("\nSecondary pass: Primary answer was NONE; checking fallback evidence...")
        option_text = " ".join(option.text for option in question.options)
        semantic_query = f"{question.text} {option_text}"
        backup_evidence = secondary_news_index.search_backup_index(
            search_terms=search_terms,
            semantic_query=semantic_query,
            top_k=6,
        )

        if backup_evidence.strip():
            print("\n" + "-" * 40)
            print("Source packet B: Bing RSS fallback evidence:")
            print(backup_evidence)
            print("-" * 40 + "\n")

            model_reply = ask_news_model(backup_evidence, question)
            answer_letter = read_final_letter(model_reply)
            print(f"Model read B:\n{model_reply}")
        else:
            print("\nSecondary pass: No fallback evidence found.")

    # Keep the caller from crashing when neither retrieval path supports an answer.
    if answer_letter == "N":
        print("\nFinal fallback: No supported answer found; using option A.")
        answer_letter = "A"

    try:
        choice_index = ["A", "B", "C", "D"].index(answer_letter)
    except ValueError:
        choice_index = 0
        answer_letter = "A"

    return question.options[choice_index].id, answer_letter, model_reply

Secondary index: Loading Bing RSS + vector fallback...


### 6.2 News Game Loop

In [80]:
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError


def play_game(competition_id=5, mode="text"):
    if mode not in {"text", "speech"}:
        raise ValueError('mode must be either "text" or "speech"')
    if mode == "speech":
        load_whisper_model_for_speech()

    API_URL = 'http://131.175.15.22:51111/'
    client = MillionaireClient(API_URL)
    user = client.login('gary', '13790229')
    print(f"Session user: {user.username}")

    game = client.game.start(competition_id=competition_id, mode=mode)
    question_count = 0
    correct_answers = 0

    while game.in_progress:
        speech_transcription = None
        if game.mode == "speech":
            question, speech_transcription = transcribe_speech_question(game)
        else:
            question = game.current_question
        if question is None:
            break

        question_count += 1
        seconds_left = getattr(game, "time_remaining", None)
        if speech_transcription:
            print(
                "Speech transcription:",
                f"{speech_transcription.get('transcription_seconds', 0.0):.1f}s",
                f"| {speech_transcription.get('seconds_left_after_audio', seconds_left) or 0:.1f}s left after audio",
                f"| Whisper {speech_transcription.get('whisper_model_size')} on {speech_transcription.get('whisper_device')}",
            )
        display_question(question, question_count, game.current_level, seconds_left)

        t0 = time.time()
        option_id, answer_letter, answer_trace = select_answer(question)
        t1 = time.time()

        chosen_offset = builtins.max(0, ord(answer_letter) - ord("A")) if answer_letter in "ABCD" else 0
        chosen_text = question.options[chosen_offset].text if chosen_offset < len(question.options) else ""
        print("-" * 80)
        print(wrap_output_text(f"Selected answer: {answer_letter}: {option_id}: {chosen_text}"))
        brief_note = pull_eval_summary(answer_trace)
        if brief_note:
            print(wrap_output_text(f"Brief rationale: {brief_note}"))
        print(f"Processing time: {t1-t0:.2f}s")

        try:
            result = game.answer(option_id)
            if result.correct:
                correct_answers += 1
            print(f"Outcome: {'CORRECT' if result.correct else 'WRONG'} | Correct answers: {correct_answers}")
        except TimeoutError:
            print("Outcome: TIMED OUT | generation took more than thirty seconds")
            break
        except RateLimitError:
            print("Rate limited; waiting five seconds before retrying submit.")
            time.sleep(5)
            result = game.answer(option_id)
            if result.correct:
                correct_answers += 1
            print(f"Outcome: {'CORRECT' if result.correct else 'WRONG'} | Correct answers: {correct_answers}")

        if result.game_over:
            break
        time.sleep(1)

    print(f"Final correct answers: {correct_answers}")
    game.correct_answers = correct_answers
    return game


### 6.3 Run News Game

In [84]:
game = play_game(competition_id=5, mode="text")

Session user: gary

Question 1 | Level 1 | 29.9s left
What strain of hantavirus was identified as the cause of the outbreak on the MV Hondius, as reported
on 2026-05-11?
  A: 0: Andes
  B: 1: Sin Nombre
  C: 2: Sangassou
  D: 3: Seoul

Query trace: Search phrase: 'Hantavirus MV_Hondius 2026-05-11 outbreak'
        Date interval: 05/09/2026 to 05/13/2026


ERROR:trafilatura.downloads:not a 200 response: 401 for URL https://www.reuters.com/business/healthcare-pharmaceuticals/evacuation-passengers-virus-hit-cruise-ship-be-completed-monday-2026-05-11/
ERROR:trafilatura.downloads:download error: https://www.nature.com/articles/d41586-026-01512-w HTTPSConnectionPool(host='idp.nature.com', port=443): Max retries exceeded with url: https://idp.nature.com/transit?redirect_uri=https%3A%2F%2Fwww.nature.com%2Farticles%2Fd41586-026-01512-w&code=67476d09-2b37-4c0a-b3cf-ce156271b21e (Caused by ResponseError('too many redirects'))



----------------------------------------
Source packet A: Primary Serper evidence:
Title: Hantavirus-hit ship sets sail for Netherlands as final passengers evacuated in Tenerife
Date: 3 weeks ago
Summary: GRANADILLA DE ABONA, Spain, May 11 (Reuters) - The hantavirus-hit ​MV Hondius departed the Spanish island of Tenerife for the Netherlands on Monday as the...
EXTENDED FULL TEXT: Evacuation of cruise ship passengers draws to a close, lengthy quarantines begin Two passengers from France and US test positive for hantavirus after evacuation Spain queries US positive test, says passengers were asymptomatic Officials say hantavirus is less contagious than COVID, risk to public considered low Reporting by Reuters bureaus; Writing by Raju Gopalakrishnan, David Latona, Aislinn Laing and Charlie Devereux; Editing by Gareth Jones, Hugh Lawson and Rosalba O'Brien Our Standards:The Thomson Reuters Trust Principles., opens new tab Corina is a Madrid-based business reporter focusing on coverage of 


----------------------------------------
Source packet B: Bing RSS fallback evidence:
Initial genetic analysis of the ‘MV Hondius’ hantavirus outbreak confirms it belongs to the Andes strain and rules out mutations. The hantavirus from the MV Hondius outbreak has been sequenced from samples taken from one of the infected individuals. The results confirm that it is the Andes strain, the most virulent and ....

What we know about the MV Hondius hantavirus outbreak. As Australian travellers linked to the MV Hondius hantavirus outbreak land in Western Australia, we look at what we do and don't know about the virus.. What we know about the MV Hondius hantavirus outbreak Fri 15 May 2026 at 2:11pm A few short weeks ago, many Australians had never heard the word hantavirus. But a deadly outbreak on the MV Hondius has catapulted the infection into the spotlight. So far, 11 confirmed cases of hantavirus have been linked to the cluster, with three recorded deaths and a fourth person in intensive

ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.thelegaladvocate.com/first-dry/Taiwan-President-Reaffirms-Sovereignty-Stance-Amid-USChina-Summit-Talks-12-8290
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.thelegaladvocate.com/expert-time/Taiwan-President-Reaffirms-Sovereignty-Stance-Amid-USChina-Summit-Talks-12-8290



----------------------------------------
Source packet A: Primary Serper evidence:
Title: Taiwan President Reaffirms Sovereignty Stance Amid US-China Summit Talks - EBITDA Margin Trends
Date: 2 weeks ago
Summary: We provide continuous equity market coverage with emphasis on earnings analysis and investor sentiment. Taiwan's President Lai Ching‑te has delivered his...

Title: Taiwan President Reaffirms Sovereignty Stance Amid US-China Summit Talks - Estimate Dispersion
Date: 2 weeks ago
Summary: The platform tracks real-time market developments, including stock price movements, analyst updates, and earnings-driven volatility across key sectors.

Title: Taiwan’s president reaffirms sovereignty, pledges no provocation after US-China talks
Date: 2 weeks ago
Summary: Taipei, Taiwan – President Lai Ching-te delivered a resolute message regarding Taiwan's national status, asserting that the island nation will neither...

Title: Taiwan's President Reaffirms Sovereignty Stance Amid Geopolitica

ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None



----------------------------------------
Source packet A: Primary Serper evidence:
Title: MAGA's favorite strongman might be on the brink of defeat - Vox
Date: Apr 8, 2026
Summary: But Hungarian elections are decidedly unfair, in that the system is structured to give the incumbent government so many advantages that the ...
EXTENDED FULL TEXT: Under normal circumstances, an election in Hungary — a landlocked Central European country of less than 10 million — would not be a major world event. But for the past 16 years, Hungary has not been a normal country.
MAGA’s favorite strongman might be on the brink of defeat
We’re about to find out whether an authoritarian can lose at the ballot box.
After Prime Minister Viktor Orbán won a massive victory in Hungary’s 2010 election, he almost immediately began changing the country’s system of government to ensure he would never lose again. He has rigged the electoral rules to favor his Fidesz party, consolidated control over 80 percent to 90 perce


----------------------------------------
Source packet B: Bing RSS fallback evidence:
Hungarian PM threatens to oust Orbán-era president. Hungary's president has refused Prime Minister Péter Magyar's demands to step aside, setting up a constitutional clash..

After winning elections in April that ended 16 years of authoritarian rule, Mr Magyar ordered high-ranking officials appointed by the ousted Fidesz party to step down. They included Tamás Sulyok, the president, whom Mr Magyar told to resign by the night of Sunday, May 31, after repeatedly branding him “Orban’s puppet”. After Mr Sulyok refused, Mr Magyar said on Monday that he would use the supermajority he won in April to rewrite Hungary’s constitution so the president could be removed.

The president is currently appointed by Hungary’s parliament and has largely ceremonial powers – but does have a role in reviewing legislation. Mr Sulyok had responded to Mr Magyar’s repeated calls for his resignation by notifying the Venice Comm

In [85]:
game = play_game(competition_id=5, mode="speech")

Session user: gary
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'According to the article published on 2026-05-15, which Swiss agency announced it would open long-sealed files on Josef Menge?' -> 'According to the article published on 2026-05-15, which Swiss agency announced it would open long-sealed files on Josef Menge'
Question transcript: According to the article published on 2026-05-15, which Swiss agency announced it would open long-sealed files on Josef Menge
Cleaned transcript noise: 'Option A, Swiss Red Cross.' -> 'Swiss Red Cross'
Option A transcript: Swiss Red Cross
Cleaned transcript noise: 'Option B, Swiss Federal Intelligence Service.' -> 'Swiss Federal Intelligence Service'
Option B transcript: Swiss Federal Intelligence Service
Cleaned transcript noise: 'Option C, Swiss Federal Police.' -> 'Swiss Federal Police'
Option C tran

ERROR:trafilatura.downloads:not a 200 response: 401 for URL https://www.reuters.com/world/europe/least-three-died-ukraine-drone-attack-moscow-region-governor-says-2026-05-17/



----------------------------------------
Source packet A: Primary Serper evidence:
Title: Ukraine conducts large-scale drone strikes on Russia, killing 4 and wounding a dozen others
Date: 2 weeks ago
Summary: Russian officials say overnight Ukrainian drone strikes killed at least four people including three near Moscow. A dozen others were wounded.
EXTENDED FULL TEXT: Ukraine conducts large-scale drone strikes on Russia, killing 4 and wounding a dozen others
Ukraine conducts large-scale drone strikes on Russia, killing 4 and wounding a dozen others
KYIV, Ukraine (AP) — One of Ukraine’s largest drone strikes on Russia killed at least four people, including three near Moscow, and wounded a dozen others, local authorities said Sunday. Debris fell on Russia’s largest airport without causing damage.
Ukrainian President Volodymyr Zelenskyy confirmed the drone strikes, saying that they were “entirely justified.” Russia has repeatedly launched similar attacks on Ukraine’s capital and other ci

---
## 7. Maths Pipeline
_Competition ID: 3 (Maths)_

Techniques: Qwen 2.5-7B-Instruct shared planner, Qwen 2.5-Math-1.5B-Instruct solver, SymPy/latex2sympy2 calculator tools, bounded generation, final-answer wrap-up.

In [86]:
# Maths model roles.
# Planner uses the shared Qwen 2.5-7B model already loaded as `model` / `tokenizer`.
# Solver uses the Qwen Math 1.5B model loaded once as `math_model` / `math_tokenizer`.
planner_tokenizer, planner_model = load_answer_model(MODEL_ID)
math_tokenizer, math_model = load_math_model(MATH_MODEL_ID)

# True uses the shared planner and calculator tools.
# False skips the toolbox and answers directly with Qwen Math.
USE_TOOL = False   # it has better performance without tool
MATHS_COMPETITION_ID = 3
maths_log = globals().get("maths_log", [])

print(f"Use tool is {USE_TOOL}")
print(f"Planner model is {MODEL_ID}")
print(f"Math model is {MATH_MODEL_ID}")

Use tool is False
Planner model is Qwen/Qwen2.5-7B-Instruct
Math model is Qwen/Qwen2.5-Math-1.5B-Instruct


### 7.1 SymPy Tool Router

In [87]:
import re, time, json, math, torch
from transformers import StoppingCriteria, StoppingCriteriaList
import sympy as sp
from fractions import Fraction
from sympy import N, symbols, simplify
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor

TRANSFORMS = standard_transformations + (implicit_multiplication_application, convert_xor)

class TimeLimitStoppingCriteria(StoppingCriteria):
    def __init__(self, start_time: float, time_limit: float):
        self.start_time = start_time
        self.time_limit = time_limit

    def __call__(self, input_ids, scores, **kwargs):
        return time.time() - self.start_time >= self.time_limit
SYM_LOCALS = {'x': sp.symbols('x'), 'y': sp.symbols('y'), 'z': sp.symbols('z'), 'a': sp.symbols('a'), 'b': sp.symbols('b'), 'k': sp.symbols('k'), 't': sp.symbols('t'), 'n': sp.symbols('n'), 'pi': sp.pi, 'e': sp.E, 'E': sp.E}

TOOL_DESCRIPTIONS = '''
You are a ROUTER, not a solver. Your job is to choose a tool or no tool.
You may think briefly, but do not solve the full problem or choose the final option.
After thinking, output FINAL_JSON: followed by exactly ONE minified JSON object.
The JSON object must have no duplicate keys and no text after it.

Tools:
- sympy_simplify: simplify/evaluate expressions, radicals, powers, fractions, rational denominator.
- sympy_solve: equations, systems, roots, root sums/products/differences.
- sympy_calculus: derivative, integral, limit, critical points.
- sympy_matrix: characteristic polynomial, eigenvalues, trace, determinant.
- probability_stats: expected value, binomial, hypergeometric/draws without replacement, Bayes, normal quartile sigma, r^2, combinations, broken-stick triangle probability.
- none: conceptual/theorem/topology/group/sampling questions with no computation tool needed.

Output schemas only:
- No tool: {"action":"none"}
- Simplify tool: {"action":"tool","tool":"sympy_simplify","input":{"latex":REAL_LATEX_OR_EMPTY,"expression":REAL_EXPRESSION_OR_EMPTY}}
- Equation tool: {"action":"tool","tool":"sympy_solve","input":{"equations":[REAL_EQUATION_STRINGS],"variables":[REAL_VARIABLES],"target":REAL_TARGET}}
- Calculus tool: {"action":"tool","tool":"sympy_calculus","input":{"operation":REAL_OPERATION,"expression":REAL_EXPRESSION,"variable":REAL_VARIABLE}}
- Matrix tool: {"action":"tool","tool":"sympy_matrix","input":{"operation":REAL_OPERATION,"polynomial":REAL_POLYNOMIAL,"variable":REAL_VARIABLE}}
- Probability/statistics tool: {"action":"tool","tool":"probability_stats","input":{"operation":REAL_OPERATION,"values":REAL_VALUES_OBJECT}}

Use only values literally present in the real question, or values directly named by the real question.
For probability/chance/random/selected/exactly/at least/without replacement questions, prefer probability_stats.
For simplify/evaluate/radical/fraction/rational-denominator questions, prefer sympy_simplify.
If the question is only a conceptual rule/theorem/definition question, output {"action":"none"}.
If you are unsure how to build a valid tool input, output {"action":"none"}.
Format:
THINK: one short sentence about whether a tool is useful.
FINAL_JSON: {"action":"none"} OR {"action":"tool",...}
'''

def normalize_math_text(s: str) -> str:
    s = str(s).replace('^', '**').replace(chr(8722), '-').replace(chr(8211), '-').replace(chr(955), 'l')
    s = re.sub(r'(?<![A-Za-z])e\s*\^', 'E^', s)
    return s

def parse_math_expr(s: str, local_dict=None):
    local = dict(SYM_LOCALS)
    if local_dict: local.update(local_dict)
    return parse_expr(normalize_math_text(s), transformations=TRANSFORMS, local_dict=local)

def parse_json_object(raw: str):
    start = raw.find('{')
    if start < 0: return None
    depth = 0
    in_str = False
    escape = False
    for i, ch in enumerate(raw[start:], start):
        if in_str:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"': in_str = True
            elif ch == '{': depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    try: return json.loads(raw[start:i+1])
                    except Exception: return None
    return None

def parse_tool_plan(raw: str):
    if 'FINAL_JSON:' in raw:
        raw = raw.split('FINAL_JSON:', 1)[1]
    parsed = parse_json_object(raw)
    if parsed:
        return parsed

    def grab_string(key):
        m = re.search(r'"\s*' + re.escape(key) + r'\s*"\s*:\s*"([^"]*)"', raw, re.IGNORECASE)
        return m.group(1).strip() if m else ''

    action = grab_string('action')
    tool = grab_string('tool')
    if not action and re.search(r'"action"\s*:\s*"?\s*tool', raw, re.IGNORECASE):
        action = 'tool'
    if not action and re.search(r'"action"\s*:\s*"?\s*none', raw, re.IGNORECASE):
        action = 'none'
    if not tool:
        m_tool = re.search(r'"\s*tool\s*"\s*:\s*"?\s*([A-Za-z_.]+)', raw, re.IGNORECASE)
        if m_tool: tool = m_tool.group(1).strip()

    payload = {}
    eq_m = re.search(r'"equations"\s*:\s*\[([^\]]*)\]', raw, re.IGNORECASE | re.S)
    if eq_m:
        payload['equations'] = re.findall(r'"([^"]+)"', eq_m.group(1))
    vars_m = re.search(r'"variables"\s*:\s*\[([^\]]*)\]', raw, re.IGNORECASE | re.S)
    if vars_m:
        payload['variables'] = re.findall(r'"([^"]+)"', vars_m.group(1))
    for key in ['operation', 'expression', 'latex', 'variable', 'point', 'target', 'polynomial']:
        val = grab_string(key)
        if val: payload[key] = val

    if action or tool or payload:
        return {'action': action or 'tool', 'tool': tool, 'input': payload}
    return None

def llm_chat(tokenizer, model, messages, max_new_tokens=120, max_time=None, assistant_prefix=''):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if assistant_prefix:
        prompt += assistant_prefix
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]
    kwargs = dict(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    if max_time is not None: kwargs['max_time'] = max_time
    with torch.no_grad():
        outputs = model.generate(**kwargs)
    return assistant_prefix + tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

def extract_option_letter(raw: str):
    boxed = re.findall(r'\\boxed\{([A-D])\}', raw, re.IGNORECASE)
    if boxed: return boxed[-1].upper(), 'boxed'
    m = re.search(r'(?:FINAL ANSWER|answer is|answer|option|choice)\s*:?\s*([A-D])\b', raw, re.IGNORECASE)
    if m: return m.group(1).upper(), 'text_match'
    letters = re.findall(r'\b([A-D])\b', raw.upper())
    return (letters[-1], 'fallback') if letters else ('C', 'fallback')

def extract_answer_letter(raw: str):
    boxed = re.findall(r'\\boxed\{\s*([A-D])\s*\}', raw, re.IGNORECASE)
    if boxed:
        return boxed[-1].upper()
    m = re.search(r'(?i)\b(?:final answer|answer is|answer|option|choice)\s*[:\-]?\s*([A-D])\b', raw)
    if m:
        return m.group(1).upper()
    return None

def run_sympy_solve(payload: dict) -> str:
    variables = payload.get('variables') or ['x']
    vars_ = [sp.symbols(str(v)) for v in variables]
    local = {str(v): sym for v, sym in zip(variables, vars_)}
    equations = payload.get('equations') or []
    exprs = []
    for eq in equations:
        eq_text = str(eq)
        if '==' in eq_text:
            left, right = eq_text.split('==', 1)
        elif '=' in eq_text:
            left, right = eq_text.split('=', 1)
        else:
            left, right = eq_text, '0'
        exprs.append(sp.Eq(parse_math_expr(left, local), parse_math_expr(right, local)))
    target = str(payload.get('target', '')).lower()
    if not exprs: return 'No equation was provided.'
    sol = sp.solve(exprs, vars_[0] if len(vars_) == 1 else vars_, dict=False)
    result = f'solutions={sol}'
    if len(vars_) == 1 and isinstance(sol, list) and len(sol) == 2:
        r1, r2 = sol[0], sol[1]
        if 'positive_difference' in target or 'difference' in target:
            result += f'; positive_difference={sp.simplify(abs(r1-r2))}; approx={float(N(abs(r1-r2))):.8g}'
        if 'sum' in target: result += f'; sum={sp.simplify(r1+r2)}'
        if 'product' in target: result += f'; product={sp.simplify(r1*r2)}'
    return result

def run_sympy_simplify(payload: dict) -> str:
    try:
        latex = str(payload.get('latex', '')).strip()
        expr_text = str(payload.get('expression', '')).strip()
        if latex:
            expr = latex2sympy(latex)
        elif expr_text:
            expr = parse_math_expr(expr_text)
        else:
            return 'No expression or latex was provided.'
        simplified = sp.radsimp(sp.simplify(expr))
        return f'expr={expr}; simplified={simplified}; approx={float(N(simplified)):.8g}'
    except Exception as e:
        return f'sympy_simplify failed: {type(e).__name__}: {e}'

def run_sympy_calculus(payload: dict) -> str:
    op = str(payload.get('operation', '')).lower()
    var = sp.symbols(str(payload.get('variable', 'x')))
    expr = parse_math_expr(payload.get('expression', '0'), {str(var): var})
    if op == 'derivative': return f'derivative={sp.simplify(sp.diff(expr, var))}'
    if op == 'integral': return f'integral={sp.simplify(sp.integrate(expr, var))}'
    if op == 'limit':
        point = parse_math_expr(payload.get('point', '0'), {str(var): var})
        direction = payload.get('direction', '+-')
        return f'limit={sp.limit(expr, var, point, dir=direction)}'
    if op == 'critical_points':
        xs, ys = sp.symbols('x y')
        crit = sp.solve([sp.diff(expr, xs), sp.diff(expr, ys)], [xs, ys], dict=True)
        return f'critical_points={crit}'
    return 'Unknown calculus operation.'

def run_sympy_matrix(payload: dict) -> str:
    op = str(payload.get('operation', '')).lower()
    var_name = str(payload.get('variable', 'l'))
    l = sp.symbols(var_name)
    if op in ['char_poly', 'trace_det_eigen']:
        poly = parse_math_expr(payload.get('polynomial', '0'), {var_name: l})
        roots = sp.solve(sp.Eq(poly, 0), l)
        return f'eigenvalues={roots}; trace=sum_eigenvalues={sp.simplify(sum(roots)) if roots else None}; det=p(0)={sp.simplify(poly.subs(l, 0))}'
    return 'Unknown matrix operation.'

def run_probability_stats(payload: dict) -> str:
    op = str(payload.get('operation', '')).lower()
    v = payload.get('values') or {}
    try:
        if op == 'expected_value':
            outcomes = v.get('outcomes') or []
            ev = sum(float(o.get('prob', 0)) * float(o.get('value', 0)) for o in outcomes)
            return f'expected_value={ev}'
        if op == 'binomial':
            n, p = int(v['n']), float(v['p']); mode = v.get('mode', 'exactly'); k = int(v['k'])
            rng = [k] if mode == 'exactly' else range(k, n+1) if mode == 'at_least' else range(0, k+1)
            prob = sum(math.comb(n, r)*(p**r)*((1-p)**(n-r)) for r in rng)
            return f'binomial_probability={prob}; fraction={Fraction(prob).limit_denominator()}'
        if op in ['hypergeometric', 'without_replacement']:
            population = int(v['population'])
            success_population = int(v['success_population'])
            draws = int(v['draws'])
            successes = int(v['successes'])
            prob = (math.comb(success_population, successes) * math.comb(population - success_population, draws - successes)) / math.comb(population, draws)
            return f'hypergeometric_probability={prob}; fraction={Fraction(prob).limit_denominator()}'
        if op == 'bayes':
            prior, hit, false_pos = float(v['prior']), float(v['hit']), float(v['false_positive'])
            post = hit*prior/(hit*prior + false_pos*(1-prior))
            return f'bayes_posterior={post}'
        if op == 'normal_quartile_sigma':
            mean, value, z = float(v['mean']), float(v['value']), float(v.get('z', -0.67448975))
            return f'sigma={abs((value-mean)/z)}'
        if op == 'correlation_r2':
            r1, r2 = float(v['r1']), float(v['r2'])
            return f'r_squared_ratio={(r1*r1)/(r2*r2)}'
        if op == 'combinations':
            n, k = int(v['n']), int(v['k'])
            return f'combination={math.comb(n, k)}'
        if op == 'broken_stick_triangle':
            return 'broken_stick_triangle_probability=1/4=0.25=25%'
    except Exception as e:
        return f'probability_stats failed: {type(e).__name__}: {e}'
    return 'Unknown probability/statistics operation.'

def run_tool_call(tool_name: str, payload: dict) -> str:
    try:
        if tool_name == 'sympy_simplify': return run_sympy_simplify(payload)
        if tool_name == 'sympy_solve': return run_sympy_solve(payload)
        if tool_name == 'sympy_calculus': return run_sympy_calculus(payload)
        if tool_name == 'sympy_matrix': return run_sympy_matrix(payload)
        if tool_name == 'probability_stats': return run_probability_stats(payload)
        return f'Unknown tool: {tool_name}'
    except Exception as e:
        return f'Tool {tool_name} failed: {type(e).__name__}: {e}'

def normalize_tool_plan(plan):
    if not isinstance(plan, dict): return None
    clean = {str(k).strip().lower().replace(' ', '_'): v for k, v in plan.items()}
    action = str(clean.get('action', '')).strip().lower()
    tool = str(clean.get('tool', clean.get('tool_name', ''))).strip().lower()
    tool_aliases = {
        'sympy.simplify': 'sympy_simplify', 'sympy_simplifier': 'sympy_simplify', 'simplify': 'sympy_simplify',
        'sympy.solve': 'sympy_solve', 'sympy_solver': 'sympy_solve', 'solve': 'sympy_solve',
        'sympy.calculus': 'sympy_calculus', 'calculus': 'sympy_calculus', 'sympy.diff': 'sympy_calculus',
        'sympy.matrix': 'sympy_matrix', 'matrix': 'sympy_matrix',
        'probability.stats': 'probability_stats', 'probability': 'probability_stats', 'stats': 'probability_stats'
    }
    tool = tool_aliases.get(tool, tool)
    payload = clean.get('input') or {}
    return {'action': action, 'tool': tool, 'input': payload}

def try_llm_tool_solver(question, tokenizer, model, options_text: str):
    start = time.time()
    route_tokenizer = globals().get('planner_tokenizer', tokenizer)
    route_model = globals().get('planner_model', model)
    planner_messages = [
        {'role': 'system', 'content': 'You are a tool router. Think briefly, then write FINAL_JSON with one valid JSON object. Do not solve the problem or choose an option.'},
        {'role': 'user', 'content': f'{TOOL_DESCRIPTIONS}\nREAL QUESTION:\n{question.text}\n\nREAL OPTIONS:\n{options_text}\n\nThink briefly, then return FINAL_JSON.'}
    ]
    plan_raw = llm_chat(route_tokenizer, route_model, planner_messages, max_new_tokens=220, max_time=5.0)
    plan = normalize_tool_plan(parse_tool_plan(plan_raw))
    print(f'Tool plan raw {plan_raw.strip()}')
    valid_tools = ['sympy_simplify', 'sympy_solve', 'sympy_calculus', 'sympy_matrix', 'probability_stats']
    if not plan or plan.get('action') != 'tool' or plan.get('tool') not in valid_tools:
        print('Tool plan no valid tool used')
        return None
    tool_name = plan.get('tool')
    tool_input = plan.get('input') or {}
    tool_result = run_tool_call(tool_name, tool_input)
    print(f'Tool used {tool_name} input {tool_input} result {tool_result}')

    remaining = 28.5 - (time.time() - start)
    if remaining <= 1.0:
        return None
    final_messages = [
        {'role': 'system', 'content': 'Use the calculator result to choose the matching multiple-choice option. Be brief. Conclude with exactly one boxed letter.'},
        {'role': 'user', 'content': f'Question:\n{question.text}\n\nOptions:\n{options_text}\n\nCalculator result:\n{tool_result}\n\nReply with one final option: \\boxed{{A}}, \\boxed{{B}}, \\boxed{{C}}, or \\boxed{{D}}.'}
    ]
    final_raw = llm_chat(tokenizer, model, final_messages, max_new_tokens=120, max_time=remaining)
    letter, rule = extract_option_letter(final_raw)
    wrapup_used = False
    if rule == 'fallback':
        remaining = 28.5 - (time.time() - start)
        if remaining > 0.5:
            wrap_raw = llm_chat(tokenizer, model, final_messages + [{'role': 'assistant', 'content': final_raw}, {'role': 'user', 'content': 'Time is almost finished. Stop and submit nearest guess only as one boxed letter.'}], max_new_tokens=20, max_time=remaining)
            w_letter, w_rule = extract_option_letter(wrap_raw)
            final_raw += '\n' + wrap_raw
            letter, rule = w_letter, 'wrapup_' + w_rule
            wrapup_used = True
            print(f'Tool wrapup raw {wrap_raw.strip()}')
    print(f'Tool final raw\n{final_raw.strip()}')
    print(f'Tool extract letter {letter} rule {rule} wrapup {wrapup_used} elapsed {time.time()-start:.2f} seconds')
    idx = ['A', 'B', 'C', 'D'].index(letter) if letter in ['A', 'B', 'C', 'D'] else 2
    return question.options[idx].id, ['A', 'B', 'C', 'D'][idx], f'tool_{tool_name}_{rule}'

### 7.2 Answer Function

In [97]:
def choose_answer_math_fixed(question, tokenizer, model) -> tuple:
    answer_start = time.time()
    options_text = "\n".join(f"{chr(65+i)}) {opt.text}" for i, opt in enumerate(question.options))
  
    if globals().get('USE_TOOL', False):
        tool_answer = try_llm_tool_solver(question, tokenizer, model, options_text)
        if tool_answer:
            return tool_answer
    else:
        print('Tool plan disabled because use tool is false')

    # Direct LLM fallback with token and time limits.
    messages = [
        {"role": "system", "content": "You are a math solver. Reason step-by-step but BE EXTREMELY BRIEF. DO NOT write long paragraphs. Conclude with your final option letter inside a box, like \\boxed{A}, \\boxed{B}, \\boxed{C}, or \\boxed{D}."},
        {"role": "user", "content": f"{question.text}\n\nOptions:\n{options_text}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    THINK_TOKENS = 480
    WRAPUP_TOKENS = 20
    TOKEN_LIMIT = THINK_TOKENS + WRAPUP_TOKENS
    TIME_LIMIT = 29.5
    WRAPUP_TIME = 27.0

    stopping = TimeLimitStoppingCriteria(start_time=answer_start, time_limit=TIME_LIMIT)

    think_ids = model.generate(
        **inputs,
        max_new_tokens=THINK_TOKENS,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        stopping_criteria=StoppingCriteriaList([stopping]),
    )

    think_new_tokens = think_ids.shape[1] - input_len
    think_text = tokenizer.decode(think_ids[0][input_len:], skip_special_tokens=True).strip()
    elapsed = time.time() - answer_start

    need_wrapup = False
    wrap_reason = 'none'
    if elapsed >= WRAPUP_TIME:
        need_wrapup = True
        wrap_reason = 'time'
    elif TOKEN_LIMIT - think_new_tokens <= WRAPUP_TOKENS:
        need_wrapup = True
        wrap_reason = 'tokens'
    elif not extract_answer_letter(think_text):
        need_wrapup = True
        wrap_reason = 'missing_answer'

    final_text = think_text
    if need_wrapup and elapsed < TIME_LIMIT:
        remaining_time = max(0.5, TIME_LIMIT - elapsed)
        wrap_messages = [
            {"role": "system", "content": "You must answer now. Use the previous work, make the nearest guess if unsure, and output only the final option letter in the form \\boxed{A}, \\boxed{B}, \\boxed{C}, or \\boxed{D}."},
            {"role": "user", "content": f"Question:\n{question.text}\n\nOptions:\n{options_text}\n\nPrevious work:\n{think_text[-1200:]}"}
        ]
        wrap_prompt = tokenizer.apply_chat_template(wrap_messages, tokenize=False, add_generation_prompt=True)
        wrap_inputs = tokenizer(wrap_prompt, return_tensors='pt').to(model.device)
        wrap_input_len = wrap_inputs['input_ids'].shape[1]
        wrap_stopping = TimeLimitStoppingCriteria(start_time=time.time(), time_limit=remaining_time)
        wrap_ids = model.generate(
            **wrap_inputs,
            max_new_tokens=WRAPUP_TOKENS,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=StoppingCriteriaList([wrap_stopping]),
        )
        wrap_text = tokenizer.decode(wrap_ids[0][wrap_input_len:], skip_special_tokens=True).strip()
        final_text = think_text + "\n" + wrap_text

    print(f"Monitor think tokens {think_new_tokens} of {TOKEN_LIMIT} direct budget {TIME_LIMIT - elapsed:.2f} seconds think time {elapsed:.2f} seconds total time {time.time() - answer_start:.2f} seconds wrapup {need_wrapup} reason {wrap_reason}")
    if think_text:
        print("Model tail")
        print(think_text[-1200:])
    print("Model log")
    print(final_text)
    print("End log")

    # Extract final option letter.
    boxed = re.findall(r'\\boxed\{\s*([A-D])\s*\}', final_text, flags=re.I)
    if boxed:
        letter = boxed[-1].upper()
        idx = ord(letter) - 65
        print(f"Extract rule boxed letter {letter}")
        return question.options[idx].id, letter, 'boxed_latex_algebra'

    # First fallback: text such as option X or answer X.
    matches = re.findall(r'(?i)\b(?:option|answer|choice)\s*[:\-]?\s*([A-D])\b', final_text)
    if matches:
        letter = matches[-1].upper()
        idx = ord(letter) - 65
        print(f"Extract rule option word letter {letter}")
        return question.options[idx].id, letter, 'option_word'

    # Final fallback: last standalone A to D letter.
    matches = re.findall(r'\b([A-D])\b', final_text)
    if matches:
        letter = matches[-1].upper()
        idx = ord(letter) - 65
        print(f"Extract rule standalone letter {letter}")
        return question.options[idx].id, letter, 'fallback_letter'

    print('Could not extract answer; defaulting to A')
    return question.options[0].id, 'A', 'fallback_default'


def answer_maths(question):
    """Full-notebook adapter: return the tuple expected by play_full_game."""
    start = time.time()
    option_id, letter, method = choose_answer_math_fixed(question, math_tokenizer, math_model)
    elapsed = time.time() - start
    return option_id, letter, elapsed, method

### 7.3 Maths Game Run

In [96]:
# Run the Maths competition and collect logs for the Evaluation section.
maths_log, maths_level, maths_earned = play_full_game(
    competition_id=MATHS_COMPETITION_ID,
    answer_fn=answer_maths,
    label=f"Qwen Math 1.5B + SymPy tools | Planner {MODEL_ID}",
    mode="text",
)


=== Game Started: Qwen Math 1.5B + SymPy tools | Planner Qwen/Qwen2.5-7B-Instruct | Competition 3 | Mode text | Session 341746 ===

--- Level 1 | Time left: 29.9s ---
Q: What is the shortest distance between the y-axis and the point (2, 7)?
   [0] 1
   [1] 2
   [2] 7
   [3] 3
Tool plan disabled because use tool is false
Monitor think tokens 238 of 500 direct budget 16.82 seconds think time 10.18 seconds total time 10.18 seconds wrapup False reason none
Model tail
To determine the shortest distance between the y-axis and the point \((2, 7)\), we need to understand that the shortest distance from a point to a line (in this case, the y-axis) is the perpendicular distance. The y-axis is the vertical line where the x-coordinate is 0. The point \((2, 7)\) has an x-coordinate of 2. Therefore, the shortest distance from the point \((2, 7)\) to the y-axis is the absolute value of the x-coordinate of the point, which is 2.

Here are the steps to solve the problem:

1. Identify the coordinates o

In [106]:
# Run the Maths competition and collect logs for the Evaluation section.
maths_log, maths_level, maths_earned = play_full_game(
    competition_id=MATHS_COMPETITION_ID,
    answer_fn=answer_maths,
    label=f"Qwen Math 1.5B + SymPy tools | Planner {MODEL_ID}",
    mode="speech",
)


=== Game Started: Qwen Math 1.5B + SymPy tools | Planner Qwen/Qwen2.5-7B-Instruct | Competition 3 | Mode speech | Session 342050 ===
Fetching question audio...
Fetching option A audio...
Fetching option B audio...
Fetching option C audio...
Fetching option D audio...
Transcribing question audio...
Cleaned transcript noise: 'If x and y are directly proportional and x equals 3 when y equals 8, what is the value of x when y equals 13?' -> 'If x and y are directly proportional and x equals 3 when y equals 8, what is the value of x when y equals 13'
Question transcript: If x and y are directly proportional and x equals 3 when y equals 8, what is the value of x when y equals 13
Cleaned transcript noise: 'Option A, 4.875.' -> '4.875'
Option A transcript: 4.875
Cleaned transcript noise: 'Option B, 34.667.' -> '34.667'
Option B transcript: 34.667
Cleaned transcript noise: 'Option C, 15.' -> '15'
Option C transcript: 15
Cleaned transcript noise: 'Option D, 0.615.' -> '0.615'
Option D transcript

---
## 8. Evaluation & Analysis
_Run after collecting logs from any or all pipelines above._


In [107]:
# Compile all available results into a DataFrame
def log_to_df(log, model_name, competition_id):
    rows = []
    for i, entry in enumerate(log or []):
        rows.append({
            "model": model_name,
            "competition_id": competition_id,
            "level": entry.get("level", i+1),
            "correct": entry.get("correct", False),
            "timed_out": entry.get("timed_out", False),
            "elapsed_s": entry.get("elapsed", 0),
            "earned": entry.get("earned", 0),
            "question": entry.get("question", ""),
        })
    return pd.DataFrame(rows)

# Combine whichever logs have actually been collected.
log_specs = [
    ("baseline_log", "Zero-Shot", globals().get("COMP_ID", 0)),
    ("fewshot_log", "Few-Shot", globals().get("COMP_ID", 0)),
    ("rag_log", "RAG", 1),
    ("maths_log", "Model + SymPy", 3),
    ("ensemble_log", "Ensemble", globals().get("COMP_ID", 0)),
]

all_dfs = []
missing_logs = []
for log_var, model_name, comp_id in log_specs:
    log = globals().get(log_var)
    if log:
        all_dfs.append(log_to_df(log, model_name, comp_id))
    else:
        missing_logs.append(log_var)

if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)
else:
    df = pd.DataFrame(columns=["model", "competition_id", "level", "correct", "timed_out", "elapsed_s", "earned", "question"])

print(f"Total game entries logged: {len(df)}")
if missing_logs:
    print("Logs not collected yet:", ", ".join(missing_logs))
df.head(10)

Total game entries logged: 23


,model,competition_id,level,correct,timed_out,elapsed_s,earned,question
0,Zero-Shot,0,1,True,False,0.457257,100,What is the fundamental principle behind Radio...
1,Zero-Shot,0,2,True,False,0.526619,200,How does Marlon Brando's portrayal of Stanley ...
2,Zero-Shot,0,3,True,False,0.459409,300,What is the name of the protagonist in Squid Game
3,Zero-Shot,0,4,True,False,0.461618,500,That is the fundamental principle of music the...
4,Zero-Shot,0,5,False,False,0.464817,500,what is the primary connection between David B...
5,Few-Shot,0,1,True,False,0.592396,100,What term describes Drake's achievement of hav...
6,Few-Shot,0,2,True,False,0.544067,200,Which of the following best describes the conn...
7,Few-Shot,0,3,False,False,0.624975,200,"How did Eminem's relationship with his mother,..."
8,RAG,1,1,True,False,0.562883,100,What is the primary reason Joey Tribbiani purs...
9,RAG,1,2,False,False,0.533783,100,How does the film 'Lawrence of Arabia' relate ...


In [108]:
# Summary statistics by model, compute we shall
summary = df.groupby("model").agg(
    questions_answered=("correct", "count"),
    accuracy=("correct", lambda x: x.dropna().mean()),
    avg_response_time=("elapsed_s", "mean"),
    timeouts=("timed_out", "sum"),
    max_earned=("earned", "max"),
).round(3)

print("=== Model Performance Summary ===")
print(summary.to_string())

=== Model Performance Summary ===
               questions_answered  accuracy  avg_response_time  timeouts  max_earned
model                                                                               
Ensemble                       11     0.909              0.547         0       32000
Few-Shot                        3     0.667              0.587         0         200
Model + SymPy                   2     0.500             14.628         0         100
RAG                             2     0.500              0.548         0         100
Zero-Shot                       5     0.800              0.474         0         500


In [ ]:
# Plot: Accuracy by Model
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy bar chart
models = summary.index.tolist()
accs = summary["accuracy"].tolist()
axes[0].bar(models, accs, color=["#4CAF50","#2196F3","#FF9800","#9C27B0","#F44336"][:len(models)])
axes[0].set_title("Accuracy by Model")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=30)

# Response time
times = summary["avg_response_time"].tolist()
axes[1].bar(models, times, color="#607D8B")
axes[1].set_title("Avg Response Time (s)")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=30)

# Earnings
earned = summary["max_earned"].tolist()
axes[2].bar(models, earned, color="#FF5722")
axes[2].set_title("Max Earned ($)")
axes[2].set_ylabel("USD")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()
print("Plot saved, it has been.")

In [ ]:
# Accuracy by difficulty level - harder questions, worse models do?
level_acc = df.groupby("level")["correct"].mean().reset_index()
level_acc.columns = ["Level", "Accuracy"]

plt.figure(figsize=(10, 4))
plt.plot(level_acc["Level"], level_acc["Accuracy"], marker="o", color="#2196F3", linewidth=2)
plt.axhline(0.25, linestyle="--", color="red", label="Random chance (25%)")
plt.fill_between(level_acc["Level"], level_acc["Accuracy"], 0.25,
                 where=level_acc["Accuracy"] > 0.25, alpha=0.2, color="green", label="Above chance")
plt.title("Accuracy by Question Level (All Models)")
plt.xlabel("Level")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig("accuracy_by_level.png", dpi=150)
plt.show()

In [ ]:
# Response time distribution - stay within 30 seconds
plt.figure(figsize=(10, 4))
for model_name, grp in df.groupby("model"):
    plt.hist(grp["elapsed_s"], bins=15, alpha=0.5, label=model_name)
plt.axvline(30, color="red", linestyle="--", label="30s timeout")
plt.title("Response Time Distribution by Model")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig("response_times.png", dpi=150)
plt.show()

too_slow = df[df["elapsed_s"] > 25]
print(f"Responses dangerously close to timeout (>25s): {len(too_slow)}")

## 9. Research Questions Analysis

In [ ]:
# Q1: Are some models better at certain topics than others?
# Explore competition_id as a category proxy.
cat_map = {0: "Entertainment", 1: "Ancient History", 2: "Science", 3: "Maths"}

cat_acc = df.groupby(["model","competition_id"])["correct"].mean().reset_index()
cat_acc["category"] = cat_acc["competition_id"].map(cat_map)

pivot = cat_acc.pivot(index="model", columns="category", values="correct")
print("=== Accuracy by Model and Category ===")
print(pivot.round(2).to_string())

In [ ]:
# Q2: Is the model overconfident? (rough approximation via prompt output entropy)
# Q3: What types of questions do models struggle on?
wrong_qs = df[df["correct"] == False][["model","level","question"]].dropna()
print(f"=== Sample of Wrong Answers ({len(wrong_qs)} total) ===")
print(wrong_qs.head(10).to_string(index=False))

In [ ]:
# Q4: Does RAG help vs. not?
# Compare Flan-T5 Zero-Shot vs Flan-T5 RAG on the same competition (if available)
rag_comp = df[df["model"].isin(["Flan-T5 Zero-Shot", "Flan-T5 RAG"])]
if len(rag_comp) > 0:
    comparison = rag_comp.groupby("model")["correct"].mean()
    print("=== RAG vs. No RAG ===")
    print(comparison)
    improvement = comparison.get("Flan-T5 RAG", 0) - comparison.get("Flan-T5 Zero-Shot", 0)
    print(f"RAG improvement: {improvement:+.1%}")

---
## 10. Best System - Final Run

Update `best_strategy` below based on your evaluation results, then run all
competitions in one go.

In [ ]:
# Best strategy, define we must based on evaluation results above
# Update this after running all experiments!

def best_strategy(question):
    """
    Winning strategy, this is. Combine RAG + few-shot + math tool, we do.
    - Maths questions -> SymPy tool
    - Other questions -> RAG + few-shot prompt
    """
    question_text_lower = question.text.lower()

    # Math detection heuristic
    math_keywords = ["calculate","compute","solve","equation","percentage","%" ,"sum",
                     "product","divided","multiplied","squared","factorial","derivative"]
    is_math = any(kw in question_text_lower for kw in math_keywords)

    if is_math:
        return answer_maths(question)
    else:
        return answer_with_rag(question)

print("Best strategy ready, it is. To the leaderboard, go we shall!")

In [ ]:
# Play all 4 competitions with the best strategy, we shall
comp_names = {0: "Entertainment", 1: "Ancient History", 2: "Science & Nature", 3: "Maths"}
final_results = {}

for comp_id in [0, 1, 2, 3]:
    print(f"\n{'='*60}")
    print(f"Playing: {comp_names[comp_id]}")
    print(f"{'='*60}")
    log, level, earned = play_full_game(
        competition_id=comp_id,
        answer_fn=best_strategy,
        label=f"Best System - {comp_names[comp_id]}",
        mode="text",
    )
    final_results[comp_id] = {"level": level, "earned": earned}
    time.sleep(2)  # Be polite to the server, we must

print("\n=== FINAL RESULTS ===")
for cid, res in final_results.items():
    print(f"  {comp_names[cid]}: Level {res['level']} | ${res['earned']:,.0f}")

In [ ]:
# Check leaderboard positions, we shall
print("=== Leaderboard Positions ===")
for comp_id in [0, 1, 2, 3]:
    lb = client.leaderboard.get(competition_id=comp_id, limit=20)
    print(f"\n--- {lb.competition.name} ---")
    for i, entry in enumerate(lb.entries[:10], 1):
        marker = " <- YOU" if entry.username == USERNAME else ""
        print(f"  {i}. {entry.username}: ${entry.score:,.0f} (Level {entry.reached_level}){marker}")

## 11. Conclusions

### Key Findings

| Question | Finding |
|----------|---------|
| Zero-shot vs few-shot vs CoT? | *(fill in after experiments)* |
| Does RAG improve accuracy? | *(fill in after experiments)* |
| Does SymPy help for Maths? | *(fill in after experiments)* |
| Are bigger models better? | *(fill in after experiments)* |
| Can we answer within 30s? | *(fill in after experiments)* |
| Which category is hardest? | *(fill in after experiments)* |